# MAIN

In [ ]:
# ============================================================================
# FINAL FIXED VERSION WITH PROPER VALIDATION FOR ALL MODELS
# ENHANCED EWS RUL PREDICTION WITH LSTM+ATTENTION
# ============================================================================
import os
os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'
os.environ['PYTHONHASHSEED'] = '42'
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import glob
import matplotlib.pyplot as plt
from scipy import signal
from scipy.stats import kurtosis, skew, pearsonr, spearmanr
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import warnings
import logging
from datetime import datetime
import pickle
import hashlib
import json
from pathlib import Path
import pywt
import random
import time
import shutil
import traceback
from collections import defaultdict
from scipy.stats import skew, kurtosis
from scipy import signal
import pywt  # For wavelet transform
import numpy as np
import logging
warnings.filterwarnings('ignore')

# ============================================================================
# SETUP FUNCTIONS
# ============================================================================
def set_seed(seed=42):
    """Set random seed for full reproducibility"""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
    os.environ['PYTHONHASHSEED'] = str(seed)
    os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'  # For deterministic behavior
    #torch.use_deterministic_algorithms(True)  # If you need absolute reproducibility
    print(f"✓ Random seed set to {seed} for full reproducibility")
    
def setup_logging():
    """Setup logging configuration"""
    log_dir = "/kaggle/working/logs"
    os.makedirs(log_dir, exist_ok=True)
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    log_file = os.path.join(log_dir, f"ews_rul_{timestamp}.log")
    logging.basicConfig(
        level=logging.INFO,
        format='%(asctime)s - %(levelname)s - %(message)s',
        handlers=[
            logging.FileHandler(log_file),
            logging.StreamHandler()
        ]
    )
    return logging.getLogger(__name__)

logger = setup_logging()

# ============================================================================
# STEP 1: ENHANCED EARLY WARNING SIGNALS (EWS) EXTRACTION WITH WAVELET
# ============================================================================
class EnhancedEarlyWarningSignals:
    def __init__(self, window_size=1024, step_size=1024):  # CHANGED: 1024, 512
        self.window_size = window_size
        self.step_size = step_size
        logger.info(f"Initialized EnhancedEarlyWarningSignals with window_size={window_size}, step_size={step_size}")

    def compute_variance(self, signal_data):
        return np.var(signal_data, ddof=1)

    def compute_autocorrelation(self, signal_data, lag=1):
        if len(signal_data) < lag + 10:
            return 0.0
        try:
            signal_detrended = signal_data - np.mean(signal_data)
            std = np.std(signal_detrended)
            if std == 0:
                return 0.0
            n = len(signal_detrended)
            fft_signal = np.fft.fft(signal_detrended, n=2 * n)
            autocorr = np.fft.ifft(fft_signal * np.conj(fft_signal))[:n].real
            autocorr = autocorr / (autocorr[0] + 1e-10)
            return float(autocorr[lag]) if lag < len(autocorr) else 0.0
        except Exception as e:
            logger.warning(f"Autocorrelation failed: {e}, returning 0.0")
            return 0.0

    def compute_skewness_kurtosis(self, signal_data):
        try:
            signal_clean = signal_data.copy()
            q25, q75 = np.percentile(signal_clean, [25, 75])
            iqr = q75 - q25
            lower = q25 - 1.5 * iqr
            upper = q75 + 1.5 * iqr
            signal_clean = signal_clean[(signal_clean >= lower) & (signal_clean <= upper)]
            if len(signal_clean) < 10:
                signal_clean = signal_data
            skewness = skew(signal_clean, nan_policy='omit')
            kurt = kurtosis(signal_clean, nan_policy='omit', fisher=True)
            return float(np.clip(skewness, -10, 10)), float(np.clip(kurt, -10, 10))
        except:
            return 0.0, 0.0

    def compute_spectral_ews(self, signal_data, fs=25600):
        try:
            nperseg = min(1024, len(signal_data) // 4)
            if nperseg < 8:
                return 0.0, 0.0, 0.0, 0.0
            freqs, psd = signal.welch(signal_data, fs=fs, nperseg=nperseg, noverlap=nperseg // 2, window='hann')
            mask = freqs > 0.1
            freqs, psd = freqs[mask], psd[mask]
            if len(psd) == 0 or np.sum(psd) <= 0:
                return 0.0, 0.0, 0.0, 0.0
            psd_norm = psd / np.sum(psd)
            centroid = np.sum(freqs * psd_norm)
            spread = np.sqrt(np.sum((freqs - centroid) ** 2 * psd_norm))
            skew_s = 0.0 if spread <= 1e-10 else np.sum((freqs - centroid) ** 3 * psd_norm) / (spread ** 3)
            kurt_s = 0.0 if spread <= 1e-10 else np.sum((freqs - centroid) ** 4 * psd_norm) / (spread ** 4) - 3
            return float(centroid), float(spread), float(skew_s), float(kurt_s)
        except:
            return 0.0, 0.0, 0.0, 0.0

    def compute_detrended_fluctuation_analysis(self, signal_data):
        try:
            N = len(signal_data)
            if N < 100:
                return 0.5
            y = np.cumsum(signal_data - np.mean(signal_data))
            min_w, max_w = 10, min(N // 4, 200)
            windows = np.unique(np.logspace(np.log10(min_w), np.log10(max_w), num=8, dtype=int))
            F_n, valid = [], []
            for n in windows:
                if n >= N or n < 4:
                    continue
                n_seg = N // n
                if n_seg < 2:
                    continue
                F = []
                for i in range(n_seg):
                    seg = y[i * n:(i + 1) * n]
                    x = np.arange(len(seg))
                    coeff = np.polyfit(x, seg, 1)
                    trend = np.polyval(coeff, x)
                    F.append(np.sqrt(np.mean((seg - trend) ** 2)))
                if F:
                    F_n.append(np.mean(F))
                    valid.append(n)
            if len(F_n) < 2:
                return 0.5
            slope = np.polyfit(np.log10(valid), np.log10(F_n), 1)[0]
            return float(slope)
        except:
            return 0.5

    def compute_hurst_exponent(self, signal_data):
        try:
            N = len(signal_data)
            if N < 100:
                return 0.5
            methods = []
            dev = signal_data - np.mean(signal_data)
            cum = np.cumsum(dev)
            wins = np.unique(np.logspace(1, np.log10(N // 4), num=6, dtype=int))
            RS, valid = [], []
            for n in wins:
                if n >= N or n < 10:
                    continue
                n_seg = N // n
                if n_seg < 2:
                    continue
                rs_vals = []
                for i in range(n_seg):
                    seg = cum[i * n:(i + 1) * n]
                    R = np.max(seg) - np.min(seg)
                    S = np.std(signal_data[i * n:(i + 1) * n]) + 1e-10
                    rs_vals.append(R / S)
                if rs_vals:
                    RS.append(np.mean(rs_vals))
                    valid.append(n)
            if len(RS) >= 2:
                methods.append(np.polyfit(np.log10(valid), np.log10(RS), 1)[0])
            if methods:
                return float(np.clip(np.mean(methods), 0, 1))
            return 0.5
        except:
            return 0.5

    def compute_critical_slowing_down_indicators(self, signal_data, fs=25600):
        try:
            ar1 = self.compute_autocorrelation(signal_data, lag=1)
            variance = self.compute_variance(signal_data)
            nperseg = min(512, len(signal_data) // 4)
            spectral_ratio = 1.0
            if nperseg >= 8:
                freqs, psd = signal.welch(signal_data, fs=fs, nperseg=nperseg)
                lf = np.sum(psd[freqs <= 100])
                hf = np.sum(psd[(freqs >= 1000) & (freqs <= 3000)])
                spectral_ratio = hf / lf if lf > 0 else 1.0 
            return float(ar1), float(variance), float(spectral_ratio)
        except:
            return 0.0, 1.0, 1.0

    def compute_wavelet_features(self, signal_data, wavelet='db4', max_level=4):
        """Compute wavelet transform features"""
        try:
            coeffs = pywt.wavedec(signal_data, wavelet, level=max_level)
            features = []
            for i, coeff in enumerate(coeffs):
                if len(coeff) > 0:
                    features.extend([
                        np.mean(coeff),
                        np.std(coeff),
                        np.max(np.abs(coeff)),
                        np.median(np.abs(coeff)),
                        np.sum(coeff ** 2),
                        skew(coeff, nan_policy='omit'),
                        kurtosis(coeff, nan_policy='omit', fisher=True)
                    ])
            return np.array(features[:20], dtype=np.float32)
        except:
            return np.zeros(20, dtype=np.float32)

    def compute_all_ews(self, signal_data, fs=25600, include_wavelet=True):
        """Compute EWS on sliding windows of signal_data"""
        if signal_data is None or len(signal_data) == 0:
            return np.zeros(44 if include_wavelet else 24, dtype=np.float32)
        
        signal_data = np.nan_to_num(signal_data, nan=0.0, posinf=0.0, neginf=0.0)
        
        # If signal is shorter than window_size, use entire signal
        if len(signal_data) <= self.window_size:
            return self._compute_ews_on_segment(signal_data, fs, include_wavelet)
        
        # Otherwise, create sliding windows and average features
        all_features = []
        for start_idx in range(0, len(signal_data) - self.window_size + 1, self.step_size):
            segment = signal_data[start_idx:start_idx + self.window_size]
            segment_features = self._compute_ews_on_segment(segment, fs, include_wavelet)
            all_features.append(segment_features)
        
        # Average features across all windows
        if all_features:
            return np.mean(all_features, axis=0)
        else:
            return self._compute_ews_on_segment(signal_data, fs, include_wavelet)
    
    def _compute_ews_on_segment(self, signal_data, fs=25600, include_wavelet=True):
        """Compute EWS on a single segment (original code)"""
        if signal_data is None or len(signal_data) == 0:
            return np.zeros(44 if include_wavelet else 24, dtype=np.float32)
        
        signal_data = np.nan_to_num(signal_data, nan=0.0, posinf=0.0, neginf=0.0)
        features = []
        
        ar1, var, spec_ratio = self.compute_critical_slowing_down_indicators(signal_data, fs)
        features.extend([ar1, var, spec_ratio])
        
        sk, ku = self.compute_skewness_kurtosis(signal_data)
        features.extend([sk, ku])
        
        for lag in [1, 2, 5, 10, 20]:
            features.append(self.compute_autocorrelation(signal_data, lag=lag))
        
        c, s, sk_s, ku_s = self.compute_spectral_ews(signal_data, fs)
        features.extend([c, s, sk_s, ku_s])
        
        features.extend([
            self.compute_detrended_fluctuation_analysis(signal_data),
            self.compute_hurst_exponent(signal_data)
        ])
        
        zero_cross = len(np.where(np.diff(np.sign(signal_data)))[0]) / len(signal_data)
        sp = np.mean(signal_data ** 2)
        noise = np.var(signal_data - signal.detrend(signal_data))
        snr = 10 * np.log10(sp / (noise + 1e-10))
        rms = np.sqrt(sp)
        crest = np.max(np.abs(signal_data)) / (rms + 1e-10)
        impulse = np.max(np.abs(signal_data)) / (np.mean(np.abs(signal_data)) + 1e-10)
        clearance = np.max(np.abs(signal_data)) / (np.mean(np.sqrt(np.abs(signal_data))) ** 2 + 1e-10)
        shape = rms / (np.mean(np.abs(signal_data)) + 1e-10)
        pp = np.ptp(signal_data)
        hist, _ = np.histogram(signal_data, bins=20, density=True)
        hist = hist[hist > 0]
        entropy = -np.sum(hist * np.log2(hist)) if len(hist) > 0 else 0.0
        features.extend([zero_cross, snr, crest, impulse, clearance, shape, pp, entropy])
        
        if include_wavelet:
            wavelet_features = self.compute_wavelet_features(signal_data)
            features.extend(wavelet_features)
        
        arr = np.array(features, dtype=np.float32)
        return np.nan_to_num(arr, nan=0.0)

    def compute_ews_for_both_channels(self, signal_2d, fs=25600, fusion_method='concat', include_wavelet=True):
        if signal_2d is None or signal_2d.shape[1] < 2:
            base_features = 44 if include_wavelet else 24
            if fusion_method == 'concat':
                return np.zeros(base_features * 2, dtype=np.float32)
            else:
                return np.zeros(base_features, dtype=np.float32)
        
        h = self.compute_all_ews(signal_2d[:, 0], fs, include_wavelet)
        v = self.compute_all_ews(signal_2d[:, 1], fs, include_wavelet)
        
        if fusion_method == 'concat':
            return np.concatenate([h, v])
        elif fusion_method == 'mean':
            return (h + v) / 2
        elif fusion_method == 'max':
            return np.where(np.abs(h) > np.abs(v), h, v)
        return h

import numpy as np  # ADD THIS FOR TEMPERATURE SCALING

# ============================================================================
# IMPROVED MODELS BASED ON LITERATURE
# ============================================================================
class GradientReversal(torch.autograd.Function):
    
    @staticmethod
    def forward(ctx, x, lambda_param):
        ctx.lambda_param = lambda_param
        return x.view_as(x)  # Return input unchanged
    
    @staticmethod
    def backward(ctx, grad_output):
        lambda_param = ctx.lambda_param
        # CRITICAL FIX: Negative gradient for domain adaptation
        dx = -lambda_param * grad_output
        return dx, None

class DomainAdaptationLayer(nn.Module):
    """Domain Adaptation Layer - UPDATED"""
    def __init__(self, input_size, num_domains=3):
        super().__init__()
        self.domain_classifier = nn.Sequential(
            nn.Linear(input_size, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, num_domains),
            nn.LogSoftmax(dim=1)
        )
    
    def forward(self, features, lambda_param=0.1, reverse=False):
        if reverse:
            features = GradientReversal.apply(features, lambda_param)
        domain_pred = self.domain_classifier(features)
        return domain_pred

class EWS_LSTM_Attention(nn.Module):
    """Bidirectional LSTM with Attention - UPDATED WITH TEMPERATURE"""
    def __init__(self, input_size, hidden_size=128, num_layers=2, dropout=0.3):
        super().__init__()
        self.input_size = input_size
        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0,
            bidirectional=True
        )
        self.attention = nn.Sequential(
            nn.Linear(hidden_size * 2, 64),
            nn.Dropout(dropout),  # ADDED DROPOUT
            nn.Tanh(),
            nn.Linear(64, 1)
        )
        self.fc = nn.Sequential(
            nn.Linear(hidden_size * 2, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(128, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, 32),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.Linear(32, 1),
            nn.Sigmoid()
        )
        self._init_weights()
        logger.info(f"Created EWS_LSTM_Attention: input={input_size}, hidden={hidden_size}, layers={num_layers}")
    
    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight, mode='fan_in', nonlinearity='relu')
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0.01)
            elif isinstance(m, nn.LSTM):
                for name, param in m.named_parameters():
                    if 'weight' in name:
                        nn.init.orthogonal_(param)
                    elif 'bias' in name:
                        nn.init.constant_(param, 0.0)
    
    def forward(self, x):
        lstm_out, _ = self.lstm(x)
        attention_weights = self.attention(lstm_out)
        
        # ADD TEMPERATURE SCALING: sqrt of attention hidden dimension (64)
        #attention_weights = attention_weights / np.sqrt(64)  old one it works 
        attention_weights = attention_weights / torch.sqrt(torch.tensor(64.0, device=x.device))
        
        attention_weights = torch.softmax(attention_weights, dim=1)
        self.last_attention = attention_weights.detach().cpu()
        
        context = torch.sum(attention_weights * lstm_out, dim=1)
        output = self.fc(context)
        return output

class EWS_LSTM_Attention_DA(nn.Module):
    
    def __init__(self, input_size, hidden_size=128, num_layers=2, dropout=0.3, use_da=True):
        super().__init__()
        self.use_da = use_da
        self.input_size = input_size
        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0,
            bidirectional=True
        )
        self.attention = nn.Sequential(
            nn.Linear(hidden_size * 2, 64),
            nn.Dropout(dropout),
            nn.Tanh(),
            nn.Linear(64, 1)
        )
        self.fc = nn.Sequential(
            nn.Linear(hidden_size * 2, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(128, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, 32),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.Linear(32, 1),
            nn.Sigmoid()
        )
        if self.use_da:
            # Domain adaptation components
            self.domain_layer = DomainAdaptationLayer(hidden_size * 2)
            # Additional domain classifier for gradient reversal
            self.domain_classifier = nn.Sequential(
                nn.Linear(hidden_size * 2, 64),
                nn.ReLU(),
                nn.Dropout(dropout),
                nn.Linear(64, 3)  # 3 domains for XJTU bearings
            )
        self._init_weights()
        logger.info(f"Created EWS_LSTM_Attention_DA: input={input_size}, hidden={hidden_size}, DA={use_da}")
    
    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight, mode='fan_in', nonlinearity='relu')
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0.01)
            elif isinstance(m, nn.LSTM):
                for name, param in m.named_parameters():
                    if 'weight' in name:
                        nn.init.orthogonal_(param)
                    elif 'bias' in name:
                        nn.init.constant_(param, 0.0)
    
    def forward(self, x, return_domain=False, lambda_param=0.1):
        lstm_out, _ = self.lstm(x)
        
        # Attention with temperature scaling
        attention_weights = self.attention(lstm_out)
        #attention_weights = attention_weights / np.sqrt(64)  old one it works 
        attention_weights = attention_weights / torch.sqrt(torch.tensor(64.0, device=x.device))
        attention_weights = torch.softmax(attention_weights, dim=1)
        self.last_attention = attention_weights.detach().cpu()
        
        context = torch.sum(attention_weights * lstm_out, dim=1)
        
        # Domain adaptation path
        domain_pred = None
        if self.use_da and return_domain:
            # Apply gradient reversal
            context_rev = GradientReversal.apply(context, lambda_param)
            domain_pred = self.domain_classifier(context_rev)
        
        rul_pred = self.fc(context)
        
        if return_domain:
            return rul_pred, domain_pred
        return rul_pred
class EWS_Bidirectional_LSTM_NoAttention(nn.Module):
    """Bidirectional LSTM WITHOUT Attention"""
    def __init__(self, input_size, hidden_size=128, num_layers=2, dropout=0.3):
        super().__init__()
        self.input_size = input_size
        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0,
            bidirectional=True
        )
        self.fc = nn.Sequential(
            nn.Linear(hidden_size * 2, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(128, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, 32),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.Linear(32, 1),
            nn.Sigmoid()
        )
        self.attention = None
        self.last_attention = None
        self._init_weights()
        print(f"Created EWS_Bidirectional_LSTM_NoAttention: input={input_size}, hidden={hidden_size}, layers={num_layers}")
    
    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight, mode='fan_in', nonlinearity='relu')
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0.01)
            # ADD THIS LSTM INITIALIZATION:
            elif isinstance(m, nn.LSTM):
                for name, param in m.named_parameters():
                    if 'weight' in name:
                        nn.init.orthogonal_(param)
                    elif 'bias' in name:
                        nn.init.constant_(param, 0.0)
    
    def forward(self, x):
        lstm_out, _ = self.lstm(x)
        context = torch.mean(lstm_out, dim=1)
        output = self.fc(context)
        return output

# For EWS_Unidirectional_LSTM_Attention, also add temperature scaling:
class EWS_Unidirectional_LSTM_Attention(nn.Module):
    """Unidirectional LSTM WITH Attention - UPDATED"""
    def __init__(self, input_size, hidden_size=128, num_layers=2, dropout=0.3):
        super().__init__()
        self.input_size = input_size
        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0,
            bidirectional=False
        )
        self.attention = nn.Sequential(
            nn.Linear(hidden_size, 64),
            nn.Dropout(dropout),  # ADDED
            nn.Tanh(),
            nn.Linear(64, 1)
        )
        self.fc = nn.Sequential(
            nn.Linear(hidden_size, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(128, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, 32),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.Linear(32, 1),
            nn.Sigmoid()
        )
        self._init_weights()
        print(f"Created EWS_Unidirectional_LSTM_Attention: input={input_size}, hidden={hidden_size}, layers={num_layers}")
    
    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight, mode='fan_in', nonlinearity='relu')
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0.01)
            elif isinstance(m, nn.LSTM):
                for name, param in m.named_parameters():
                    if 'weight' in name:
                        nn.init.orthogonal_(param)
                    elif 'bias' in name:
                        nn.init.constant_(param, 0.0)
    
    def forward(self, x):
        lstm_out, _ = self.lstm(x)
        attention_weights = self.attention(lstm_out)
        
        # ADD TEMPERATURE SCALING
        #attention_weights = attention_weights / np.sqrt(64)  old one it works 
        attention_weights = attention_weights / torch.sqrt(torch.tensor(64.0, device=x.device))
        
        attention_weights = torch.softmax(attention_weights, dim=1)
        self.last_attention = attention_weights.detach().cpu()
       
        context = torch.sum(attention_weights * lstm_out, dim=1)
        output = self.fc(context)
        return output
class EWS_Unidirectional_LSTM_NoAttention(nn.Module):
    """Unidirectional LSTM WITHOUT Attention"""
    def __init__(self, input_size, hidden_size=128, num_layers=2, dropout=0.3):
        super().__init__()
        self.input_size = input_size
        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0,
            bidirectional=False
        )
        self.fc = nn.Sequential(
            nn.Linear(hidden_size, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(128, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, 32),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.Linear(32, 1),
            nn.Sigmoid()
        )
        self.attention = None
        self.last_attention = None
        self._init_weights()
        print(f"Created EWS_Unidirectional_LSTM_NoAttention: input={input_size}, hidden={hidden_size}, layers={num_layers}")
    
    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight, mode='fan_in', nonlinearity='relu')
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0.01)
            # ADD THIS LSTM INITIALIZATION:
            elif isinstance(m, nn.LSTM):
                for name, param in m.named_parameters():
                    if 'weight' in name:
                        nn.init.orthogonal_(param)
                    elif 'bias' in name:
                        nn.init.constant_(param, 0.0)
    
    def forward(self, x):
        lstm_out, _ = self.lstm(x)
        context = torch.mean(lstm_out, dim=1)
        output = self.fc(context)
        return output
        
class Simple_LSTM(nn.Module):
    """Simple LSTM (Vanilla)"""
    def __init__(self, input_size, hidden_size=128, num_layers=2, dropout=0.3):
        super().__init__()
        self.input_size = input_size
        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0,
            bidirectional=False
        )
        self.fc = nn.Sequential(
            nn.Linear(hidden_size, 64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 1),
            nn.Sigmoid()
        )
        self.attention = None
        self.last_attention = None
        
        # ADD THIS LINE:
        self._init_weights()
        
        logger.info(f"Created Simple_LSTM: input={input_size}, hidden={hidden_size}, layers={num_layers}")

    
    # ADD THIS METHOD:
    def _init_weights(self):
        """Weight initialization for Simple_LSTM"""
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight, mode='fan_in', nonlinearity='relu')
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0.01)
            elif isinstance(m, nn.LSTM):
                for name, param in m.named_parameters():
                    if 'weight' in name:
                        nn.init.orthogonal_(param)
                    elif 'bias' in name:
                        nn.init.constant_(param, 0.0)
    
    def forward(self, x):
        lstm_out, (hidden, cell) = self.lstm(x)
        last_hidden = hidden[-1]
        output = self.fc(last_hidden)
        return output
# ============================================================================
# DATASET FUNCTIONS - FIXED FOR PHD
# ============================================================================
import os
import json
import pickle
import hashlib
import logging
import numpy as np
import pandas as pd
from torch.utils.data import Dataset
import torch

logger = logging.getLogger(__name__)

# Constants for reproducibility
SIGNAL_LENGTH = 32768  # Fixed signal length for XJTU dataset (25600 Hz × 1.28s = 32768)

def get_dataset_cache_key(data_root, bearing_list, window_size=10, step_size=5,
                         max_files_per_bearing=None, use_both_channels=True,
                         fusion_method='concat', manual_settings=None, 
                         include_wavelet=True, ews_window_size=1024, ews_step_size=1024,
                         train_indices=None, val_indices=None):
    """Generate cache key including split info to prevent data leakage"""
    params = {
        'data_root': data_root,
        'bearing_list': sorted(bearing_list),
        'window_size': window_size,
        'step_size': step_size,
        'max_files_per_bearing': max_files_per_bearing,
        'use_both_channels': use_both_channels,
        'fusion_method': fusion_method,
        'manual_settings': manual_settings,
        'include_wavelet': include_wavelet,
        'ews_window_size': ews_window_size,
        'ews_step_size': ews_step_size,
        'signal_length': SIGNAL_LENGTH,
        'train_indices': sorted(train_indices) if train_indices else None,
        'val_indices': sorted(val_indices) if val_indices else None,
        'code_version': '5.0'
    }
    params_str = json.dumps(params, sort_keys=True, default=str)
    hash_obj = hashlib.md5(params_str.encode())
    return hash_obj.hexdigest()

def save_dataset_to_cache(dataset, cache_key, cache_dir="/kaggle/working/cache"):
    """Save dataset to cache for faster loading"""
    os.makedirs(cache_dir, exist_ok=True)
    cache_file = os.path.join(cache_dir, f"dataset_{cache_key}.pkl")
    cache_data = {
        'ews_sequences': [seq.tolist() for seq in dataset.ews_sequences],
        'rul_targets': dataset.rul_targets,
        'bearing_ids': dataset.bearing_ids,
        'max_ruls': dataset.max_ruls,
        'degradation_starts': dataset.degradation_starts,
        'processed_files': dataset.processed_files,
        'skipped_files': dataset.skipped_files,
        'skipped_bearings': dataset.skipped_bearings,
        'feature_dimension': dataset.get_feature_dimension()
    }
    try:
        with open(cache_file, 'wb') as f:
            pickle.dump(cache_data, f, protocol=pickle.HIGHEST_PROTOCOL)
        logger.info(f"Dataset cached successfully: {cache_file}")
        return True
    except Exception as e:
        logger.error(f"Failed to save cache: {e}")
        return False

def load_dataset_from_cache(cache_key, cache_dir="/kaggle/working/cache"):
    """Load dataset from cache if exists"""
    cache_file = os.path.join(cache_dir, f"dataset_{cache_key}.pkl")
    if not os.path.exists(cache_file):
        return None
    try:
        with open(cache_file, 'rb') as f:
            cache_data = pickle.load(f)
        logger.info(f"Loading dataset from cache: {cache_file}")
        return cache_data
    except Exception as e:
        logger.warning(f"Failed to load cache: {e}")
        try:
            os.remove(cache_file)
        except:
            pass
        return None

class EnhancedXJTU_EWS_Dataset(Dataset):
    """Enhanced XJTU Dataset with Early Warning Signals extraction for PhD research"""
    
    def __init__(self, data_root, bearing_list, window_size=10, step_size=5,
                 max_files_per_bearing=None, ews_extractor=None,
                 use_both_channels=True, fusion_method='concat',
                 enable_caching=True, cache_dir="/kaggle/working/cache",
                 include_wavelet=True, ews_window_size=1024, ews_step_size=1024,
                 train_indices=None, val_indices=None):
        
        self.data_root = data_root
        self.bearing_list = bearing_list
        self.window_size = window_size        # LSTM sequence length
        self.step_size = step_size            # LSTM step
        self.max_files = max_files_per_bearing
        self.use_both_channels = use_both_channels
        self.fusion_method = fusion_method
        self.include_wavelet = include_wavelet
        self.ews_window_size = ews_window_size
        self.ews_step_size = ews_step_size
        
        # Create EWS extractor
        self.ews_extractor = ews_extractor or EnhancedEarlyWarningSignals(
            window_size=ews_window_size, 
            step_size=ews_step_size
        )
        
        self.enable_caching = enable_caching
        self.cache_dir = cache_dir
        
        # Complete manual settings for all XJTU bearings
        # Based on XJTU dataset analysis and literature
        self.manual_settings = {
            'Bearing1_1': {'multiplier': 3.5, 'manual_start': 75, 'condition': '35Hz12kN'},#80
            'Bearing1_2': {'multiplier': 3.5, 'manual_start': 54, 'condition': '35Hz12kN'},#100
            'Bearing1_3': {'multiplier': 5.0, 'manual_start': 59, 'condition': '35Hz12kN'},#58
            'Bearing1_4': {'multiplier': 2.5, 'manual_start': 75, 'condition': '35Hz12kN'},#70
            'Bearing1_5': {'multiplier': 2.5, 'manual_start': 32, 'condition': '35Hz12kN'},#25
            'Bearing2_1': {'multiplier': 3.0, 'manual_start': 453, 'condition': '37.5Hz11kN'},#300
            'Bearing2_2': {'multiplier': 3.0, 'manual_start': 47, 'condition': '37.5Hz11kN'},#100
            'Bearing2_3': {'multiplier': 3.0, 'manual_start': 59, 'condition': '37.5Hz11kN'},#150
           
            'Bearing3_2': {'multiplier': 3.0, 'manual_start': 466, 'condition': '40Hz10kN'},#700
            'Bearing3_3': {'multiplier': 3.0, 'manual_start': 342, 'condition': '40Hz10kN'},#250
            'Bearing3_4': {'multiplier': 3.0, 'manual_start': 45, 'condition': '40Hz10kN'},#80
            
        }
        
        self.ews_sequences = []
        self.rul_targets = []
        self.bearing_ids = []
        self.max_ruls = []
        self.degradation_starts = {}
        self.processed_files = 0
        self.skipped_files = 0
        self.skipped_bearings = []
        
        # Try to load from cache
        if self.enable_caching:
            cache_key = get_dataset_cache_key(
                data_root, bearing_list, window_size, step_size,
                max_files_per_bearing, use_both_channels, fusion_method,
                self.manual_settings, include_wavelet,
                self.ews_window_size, self.ews_step_size,
                train_indices, val_indices
            )
            cached_data = load_dataset_from_cache(cache_key, cache_dir)
            if cached_data is not None:
                # Convert lists back to numpy arrays
                self.ews_sequences = [np.array(seq, dtype=np.float32) for seq in cached_data['ews_sequences']]
                self.rul_targets = cached_data['rul_targets']
                self.bearing_ids = cached_data['bearing_ids']
                self.max_ruls = cached_data['max_ruls']
                self.degradation_starts = cached_data['degradation_starts']
                self.processed_files = cached_data['processed_files']
                self.skipped_files = cached_data['skipped_files']
                self.skipped_bearings = cached_data['skipped_bearings']
                
                # Validate cache integrity
                if (len(self.ews_sequences) == len(self.rul_targets) == 
                    len(self.bearing_ids) == len(self.max_ruls)):
                    logger.info(f"✓ Loaded {len(self.ews_sequences)} sequences from cache")
                    self.print_statistics()
                    return
                else:
                    logger.warning("Cache corrupted, reprocessing dataset")
                    self.ews_sequences = []
                    self.rul_targets = []
                    self.bearing_ids = []
                    self.max_ruls = []
        
        # Process dataset if cache not found or invalid
        self._process_all_bearings()
        
        # Save to cache if enabled
        if self.enable_caching and len(self.ews_sequences) > 0:
            cache_key = get_dataset_cache_key(
                data_root, bearing_list, window_size, step_size,
                max_files_per_bearing, use_both_channels, fusion_method,
                self.manual_settings, self.include_wavelet,
                self.ews_window_size, self.ews_step_size,
                train_indices, val_indices
            )
            save_dataset_to_cache(self, cache_key, cache_dir)
        
        logger.info(f"✓ Dataset loaded: {len(self.ews_sequences)} sequences")
        self.print_statistics()
    
    def _get_bearing_path(self, bearing_name):
        """Get full path to bearing data"""
        # Map bearing to condition based on XJTU naming convention
        if 'Bearing1_' in bearing_name:
            cond = '35Hz12kN'
        elif 'Bearing2_' in bearing_name:
            cond = '37.5Hz11kN'
        else:  # Bearing3_
            cond = '40Hz10kN'
        return os.path.join(self.data_root, cond, bearing_name)
    
    def _read_vibration_signal(self, filepath):
        """Read vibration signal from XJTU CSV file (no headers in original data)"""
        try:
            # XJTU files have no headers and 2 columns (horizontal, vertical)
            df = pd.read_csv(filepath)
            
            if df.shape[1] >= 2:
                h = df.iloc[:, 0].values.astype(np.float32)
                v = df.iloc[:, 1].values.astype(np.float32)
            else:
                # Fallback: if only 1 column, duplicate it for both channels
                h = df.iloc[:, 0].values.astype(np.float32)
                v = h.copy()
            
            signal = np.column_stack([h, v])
            
            # Standardize to fixed length (32768 samples = 1.28s at 25600 Hz)
            if signal.shape[0] > SIGNAL_LENGTH:
                signal = signal[:SIGNAL_LENGTH]
            
            # Handle NaN/inf values
            return np.nan_to_num(signal, nan=0.0, posinf=0.0, neginf=0.0)
        except Exception as e:
            logger.error(f"Error reading {filepath}: {e}")
            return None
    
    def _calculate_piecewise_rul_pdf(self, csv_files, bearing_name):
        """
        Calculate RUL using piecewise-linear approach for PhD methodology:
        1. Healthy phase: RUL = max_rul (constant)
        2. Degradation phase: RUL = total_files - current_file_index - 1
        """
        total_files = len(csv_files)
        logger.info(f"PDF-based RUL calculation for {bearing_name}: {total_files} files")
        
        # Calculate amplitude for each file
        amplitudes = []
        for path in csv_files:
            sig = self._read_vibration_signal(path)
            if sig is None:
                amplitudes.append(0.0)
                continue
            # Use maximum amplitude across both channels
            amp = max(np.max(np.abs(sig[:, 0])), np.max(np.abs(sig[:, 1])))
            amplitudes.append(amp)
        
        amplitudes = np.array(amplitudes)
        
        # Calculate healthy baseline (first 50 files or half of dataset)
        baseline_samples = min(50, total_files // 2)
        if baseline_samples > 0:
            healthy_vals = amplitudes[:baseline_samples]
            healthy_baseline = np.median(healthy_vals)
        else:
            healthy_baseline = 1.0
        
        # Get manual settings for this bearing
        settings = self.manual_settings.get(bearing_name, 
                                          {'multiplier': 3.0, 'manual_start': int(total_files * 0.7)})
        multiplier = settings['multiplier']
        manual_start = settings['manual_start']
        degradation_threshold = healthy_baseline * multiplier
        
        logger.info(f"{bearing_name}: Healthy baseline={healthy_baseline:.4f}, "
                   f"Threshold={degradation_threshold:.4f} (×{multiplier})")
        
        # Ensure degradation start is reasonable
        degradation_start = manual_start
        if degradation_start < baseline_samples:
            degradation_start = baseline_samples + 10
        
        # Minimum degradation phase (at least 20 files or 20% of total)
        min_degradation_phase = int(max(20, total_files * 0.2))
        degradation_start = int(degradation_start)
        max_rul = total_files - degradation_start -1
        
        # Adjust if max_rul is too small
        if max_rul < min_degradation_phase:
            logger.warning(f"Max RUL too small ({max_rul}), adjusting...")
            degradation_start = total_files - min_degradation_phase
            degradation_start = int(degradation_start)
            max_rul = total_files - degradation_start -1
        
        # Ensure degradation doesn't start too early
        if degradation_start < baseline_samples + 10:
            logger.warning(f"Degradation too early ({degradation_start}), adjusting...")
            degradation_start = baseline_samples + 20
            degradation_start = int(degradation_start)
            max_rul = total_files - degradation_start-1
        
        # Final validation
        if max_rul <= 0:
            logger.error(f"Invalid max_rul={max_rul} for {bearing_name}, using fallback")
            degradation_start = total_files - 30
            max_rul = 30
        
        logger.info(f"{bearing_name}: Degradation Start={degradation_start}, Max RUL={max_rul}")
        return degradation_start, max_rul
    
    def _process_all_bearings(self):
        """Process all bearings and extract EWS features"""
        for bearing_name in self.bearing_list:
            path = self._get_bearing_path(bearing_name)
            if not os.path.exists(path):
                logger.warning(f"⚠ Skipping {bearing_name} - path not found: {path}")
                self.skipped_bearings.append(bearing_name)
                continue
            
            try:
                # List and sort CSV files
                files = sorted([f for f in os.listdir(path) if f.endswith('.csv')],
                             key=lambda x: int(x.split('.')[0]))
                csv_files = [os.path.join(path, f) for f in files]
                
                if self.max_files:
                    csv_files = csv_files[:self.max_files]
                    
                logger.debug(f"Found {len(csv_files)} CSV files for {bearing_name}")
            except Exception as e:
                logger.error(f"Error listing files for {bearing_name}: {e}")
                self.skipped_bearings.append(bearing_name)
                continue
            
            # Check minimum file requirement
            if len(csv_files) < self.window_size:
                logger.warning(f"⚠ Skipping {bearing_name} - too few files ({len(csv_files)} < {self.window_size})")
                self.skipped_bearings.append(bearing_name)
                continue
            
            logger.info(f"Processing {bearing_name}: {len(csv_files)} files")
            
            # Calculate RUL
            degradation_start, max_rul = self._calculate_piecewise_rul_pdf(csv_files, bearing_name)
            self.degradation_starts[bearing_name] = degradation_start
            
            # Extract EWS sequences with sliding window
            sequences_added = 0
            for start_idx in range(0, len(csv_files) - self.window_size + 1, self.step_size):
                end_idx = start_idx + self.window_size
                ews_seq = []
                valid = True
                
                # Extract EWS for each file in the window
                for i in range(start_idx, end_idx):
                    sig = self._read_vibration_signal(csv_files[i])
                    if sig is None:
                        valid = False
                        break
                    
                    # Standardize signal length
                    if sig.shape[0] < SIGNAL_LENGTH:
                        # Pad with zeros if too short
                        pad_amount = SIGNAL_LENGTH - sig.shape[0]
                        sig = np.pad(sig, ((0, pad_amount), (0, 0)), 'constant')
                    elif sig.shape[0] > SIGNAL_LENGTH:
                        # Truncate if too long
                        sig = sig[:SIGNAL_LENGTH]
                    
                    # Extract EWS features
                    if self.use_both_channels:
                        feats = self.ews_extractor.compute_ews_for_both_channels(
                            sig, fusion_method=self.fusion_method, include_wavelet=self.include_wavelet
                        )
                    else:
                        feats = self.ews_extractor.compute_all_ews(
                            sig[:, 0], include_wavelet=self.include_wavelet
                        )
                    
                    # Check for invalid features
                    if np.any(np.isnan(feats)) or np.any(np.isinf(feats)):
                        valid = False
                        break
                    
                    ews_seq.append(feats)
                
                # Skip invalid sequences
                if not valid or len(ews_seq) != self.window_size:
                    continue
                
                # Stack features for this sequence
                ews_seq = np.stack(ews_seq, axis=0)
                
                # Calculate RUL for the last file in the window
                last_idx = end_idx - 1
                if last_idx < degradation_start:
                    # Healthy phase: constant RUL
                    rul_minutes = max_rul
                else:
                    # Degradation phase: decreasing RUL
                    rul_minutes = len(csv_files) - last_idx - 1
                
                # Normalize RUL
                rul_norm = rul_minutes / max_rul if max_rul > 0 else 0.0
                
                # Store sequence
                self.ews_sequences.append(ews_seq.astype(np.float32))
                self.rul_targets.append(float(rul_norm))
                self.bearing_ids.append(bearing_name)
                self.max_ruls.append(max_rul)
                self.processed_files += 1
                sequences_added += 1
            
            logger.info(f"✓ Added {sequences_added} sequences from {bearing_name}")
    
    def get_feature_dimension(self):
        """Get feature dimension for model initialization"""
        if len(self.ews_sequences) == 0:
            # Calculate expected dimension
            base_features = 44 if self.include_wavelet else 24
            if self.use_both_channels and self.fusion_method == 'concat':
                return base_features * 2
            return base_features
        return self.ews_sequences[0].shape[1]
    
    def print_statistics(self):
        """Print dataset statistics for PhD paper/reproducibility"""
        if len(self.rul_targets) == 0:
            logger.warning("No data to show statistics")
            return
        
        stats = {
            'Total sequences': len(self),
            'Unique bearings': len(set(self.bearing_ids)),
            'Feature dimension': self.get_feature_dimension(),
            'Average RUL': float(np.mean(self.rul_targets)),
            'Std RUL': float(np.std(self.rul_targets)),
            'Min RUL': float(np.min(self.rul_targets)),
            'Max RUL': float(np.max(self.rul_targets)),
            'Skipped bearings': len(self.skipped_bearings),
            'Processed files': self.processed_files,
            'Signal length (samples)': SIGNAL_LENGTH,
            'EWS window size': self.ews_window_size,
            'LSTM window size': self.window_size,
            'LSTM step size': self.step_size,
        }
        
        logger.info("=" * 60)
        logger.info("DATASET STATISTICS (FOR PHD PAPER):")
        for k, v in stats.items():
            logger.info(f"  {k:<25}: {v}")
        
        # Per-bearing statistics
        if set(self.bearing_ids):
            logger.info("\nPer-bearing statistics:")
            for bearing in sorted(set(self.bearing_ids)):
                indices = [i for i, b in enumerate(self.bearing_ids) if b == bearing]
                bearing_stats = {
                    'sequences': len(indices),
                    'avg_rul': float(np.mean([self.rul_targets[i] for i in indices])),
                    'degradation_start': self.degradation_starts.get(bearing, 'N/A'),
                    'condition': self.manual_settings.get(bearing, {}).get('condition', 'Unknown')
                }
                logger.info(f"  {bearing}: {bearing_stats}")
        
        logger.info("=" * 60)
        return stats
    
    def __len__(self):
        return len(self.ews_sequences)
    
    def __getitem__(self, idx):
        """Get dataset item for PyTorch DataLoader"""
        # Ensure sequences are numpy arrays
        if isinstance(self.ews_sequences[idx], list):
            self.ews_sequences[idx] = np.array(self.ews_sequences[idx], dtype=np.float32)
        
        return (
            torch.FloatTensor(self.ews_sequences[idx]),  # [window_size, feature_dim]
            torch.FloatTensor([self.rul_targets[idx]]),  # [1]
            self.bearing_ids[idx],                       # string identifier
            self.max_ruls[idx]                           # scalar
        )
# ============================================================================
# TRAINING FUNCTIONS - FIXED FOR PHD
# ============================================================================
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset, Subset
from collections import defaultdict
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from scipy.stats import pearsonr, spearmanr
import os
import logging
from datetime import datetime


def create_balanced_train_val_split(dataset, val_split=0.2, random_seed=42):
    bearing_to_indices = defaultdict(list)
    for idx in range(len(dataset)):
        bearing_to_indices[dataset.bearing_ids[idx]].append(idx)

    train_indices = []
    val_indices = []

    for bearing_id, indices in bearing_to_indices.items():
        num_samples = len(indices)
        num_val = max(1, int(num_samples * val_split))
        # Keep natural order – indices are already chronological
        val_indices.extend(indices[-num_val:])          # last windows for validation
        train_indices.extend(indices[:-num_val])        # first windows for training

    # Shuffle combined lists for minibatch randomness
    rng = np.random.RandomState(random_seed)
    rng.shuffle(train_indices)
    rng.shuffle(val_indices)

    logger.info(f"Time‑based split: {len(train_indices)} training, {len(val_indices)} validation")
    return train_indices, val_indices

def train_model_with_validation(model, train_loader, val_loader, epochs=100, lr=0.001,
                                device='cuda', early_stopping_patience=20, use_da=False,
                                lambda_da=0.1):
    """Universal training function for ALL models with proper validation - FIXED"""
    model.to(device)
    
    # ==================== FIXED OPTIMIZER SECTION ====================
    if use_da and hasattr(model, 'domain_classifier'):
        # Separate parameters: domain_classifier gets lower LR
        base_params = []
        domain_params = []
        for name, param in model.named_parameters():
            if 'domain_classifier' in name:
                domain_params.append(param)
            else:
                base_params.append(param)
        
        optimizer = optim.AdamW([
            {'params': base_params, 'lr': lr, 'weight_decay': 1e-4},
            {'params': domain_params, 'lr': lr * 0.1, 'weight_decay': 1e-4}  # lower LR for domain
        ])
    else:
        optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    # =================================================================

    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.5, patience=10
    )
    
    criterion_rul = nn.MSELoss()
    if use_da:
        criterion_domain = nn.NLLLoss()  # For LogSoftmax output
    
    best_val_loss = float('inf')
    patience_counter = 0
    train_losses, val_losses = [], []
    domain_losses = [] if use_da else None

    logger.info(f"Starting training on {device}, epochs={epochs}, DA={use_da}, λ_da={lambda_da}")
    
    for epoch in range(epochs):
        model.train()
        train_rul_loss, train_domain_loss = 0.0, 0.0
        train_batches = 0
        
        for batch_idx, (data, targets, bearing_ids, _) in enumerate(train_loader):
            data, targets = data.to(device), targets.to(device)
            
            # Domain labels
            if use_da and hasattr(model, 'domain_classifier'):
                domain_labels = []
                for bid in bearing_ids:
                    if 'Bearing1_' in bid:
                        domain_labels.append(0)
                    elif 'Bearing2_' in bid:
                        domain_labels.append(1)
                    elif 'Bearing3_' in bid:
                        domain_labels.append(2)
                    else:
                        domain_labels.append(0)
                domain_labels = torch.tensor(domain_labels, device=device)
            
            # Zero gradients (single optimizer)
            optimizer.zero_grad()
            
            # Forward pass
            if use_da and hasattr(model, 'domain_classifier'):
                rul_pred, domain_pred = model(data, return_domain=True, lambda_param=lambda_da)
                rul_loss = criterion_rul(rul_pred, targets)
                domain_loss = criterion_domain(domain_pred, domain_labels)
                total_loss = rul_loss + lambda_da * domain_loss
            else:
                rul_pred = model(data)
                rul_loss = criterion_rul(rul_pred, targets)
                total_loss = rul_loss
                domain_loss = torch.tensor(0.0)
            
            # Backward pass
            total_loss.backward()
            
            # Gradient clipping for stability
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            
            # Single optimizer step
            optimizer.step()
            
            train_rul_loss += rul_loss.item()
            if use_da:
                train_domain_loss += domain_loss.item()
            train_batches += 1
        
        # Validation (unchanged)
        model.eval()
        val_loss = 0.0
        val_batches = 0
        with torch.no_grad():
            for data, targets, _, _ in val_loader:
                data, targets = data.to(device), targets.to(device)
                rul_pred = model(data)
                if isinstance(rul_pred, tuple):
                    rul_pred = rul_pred[0]
                val_loss += criterion_rul(rul_pred, targets).item()
                val_batches += 1
        
        avg_train_loss = train_rul_loss / train_batches
        avg_val_loss = val_loss / val_batches
        train_losses.append(avg_train_loss)
        val_losses.append(avg_val_loss)
        
        if use_da:
            avg_domain_loss = train_domain_loss / train_batches
            domain_losses.append(avg_domain_loss)
        
        scheduler.step(avg_val_loss)
        
        # Early stopping (unchanged)
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            best_state = model.state_dict().copy()
            patience_counter = 0
            torch.save({...}, f"/kaggle/working/best_model_epoch_{epoch}.pth")
        else:
            patience_counter += 1
        
        # Logging (unchanged)
        if (epoch + 1) % 5 == 0 or epoch == 0:
            da_info = f", Domain Loss: {avg_domain_loss:.6f}" if use_da else ""
            logger.info(...)
        
        if patience_counter >= early_stopping_patience:
            logger.info(f"Early stopping at epoch {epoch + 1}")
            break
    
    # Load best model (unchanged)
    if 'best_state' in locals():
        model.load_state_dict(best_state)
        logger.info(f"Loaded best model with val loss: {best_val_loss:.6f}")
    
    return model, train_losses, val_losses, domain_losses

def calculate_comprehensive_metrics(preds, targets):
    """Calculate comprehensive evaluation metrics for PhD analysis"""
    from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
    from scipy.stats import pearsonr, spearmanr
    import numpy as np
    
    preds = np.array(preds).flatten()
    targets = np.array(targets).flatten()
    
    metrics = {
        'MAE': mean_absolute_error(targets, preds),
        'MSE': mean_squared_error(targets, preds),
        'RMSE': np.sqrt(mean_squared_error(targets, preds)),
        'R2': r2_score(targets, preds),
    }
    
    # Add correlation metrics if enough samples
    if len(targets) > 2:
        try:
            pearson_corr, pearson_p = pearsonr(targets, preds)
            spearman_corr, spearman_p = spearmanr(targets, preds)
            metrics['Pearson_r'] = pearson_corr
            metrics['Pearson_p'] = pearson_p
            metrics['Spearman_rho'] = spearman_corr
            metrics['Spearman_p'] = spearman_p
        except Exception as e:
            logger.warning(f"Could not compute correlation metrics: {e}")
            metrics['Pearson_r'] = 0
            metrics['Pearson_p'] = 1.0
            metrics['Spearman_rho'] = 0
            metrics['Spearman_p'] = 1.0
    
    # Percentage-based metrics (only for positive targets)
    mask = targets > 0
    if np.sum(mask) > 0:
        metrics['MAPE'] = np.mean(np.abs((targets[mask] - preds[mask]) / targets[mask])) * 100
        metrics['Within_10%'] = np.mean(np.abs(preds[mask] - targets[mask]) / targets[mask] <= 0.1) * 100
        metrics['Within_20%'] = np.mean(np.abs(preds[mask] - targets[mask]) / targets[mask] <= 0.2) * 100
        metrics['Within_30%'] = np.mean(np.abs(preds[mask] - targets[mask]) / targets[mask] <= 0.3) * 100
    else:
        metrics['MAPE'] = np.nan
        metrics['Within_10%'] = 0
        metrics['Within_20%'] = 0
        metrics['Within_30%'] = 0
    
    # Monotonicity score (important for RUL)
    def monotonicity_score(values):
        if len(values) < 2:
            return 0
        diff = np.diff(values)
        positive = np.sum(diff > 0)
        negative = np.sum(diff < 0)
        return abs(positive - negative) / (len(values) - 1)
    
    metrics['Monotonicity_pred'] = monotonicity_score(preds)
    metrics['Monotonicity_true'] = monotonicity_score(targets)
    
    # Additional statistical metrics
    metrics['Std_Error'] = np.std(preds - targets)
    metrics['Bias'] = np.mean(preds - targets)
    metrics['NRMSE'] = metrics['RMSE'] / (np.max(targets) - np.min(targets)) if np.ptp(targets) > 0 else np.nan
    
    return metrics

def create_directories(base_path="/kaggle/working"):
    """Create all necessary directories for PhD experiments"""
    dirs = [
        os.path.join(base_path, "results"),
        os.path.join(base_path, "logs"),
        os.path.join(base_path, "cache"),
        os.path.join(base_path, "models"),
        os.path.join(base_path, "plots"),
        os.path.join(base_path, "tables"),
    ]
    for dir_path in dirs:
        os.makedirs(dir_path, exist_ok=True)
        logger.info(f"Created directory: {dir_path}")
    return dirs

def run_single_split(split_name, train_bearings, test_bearings,
                     skip_if_exists=True, model_type='lstm_da',
                     include_wavelet=True, use_da=True,
                     sequence_length=10, ews_window_size=1024,
                     ews_step_size=1024, val_split=0.2,
                     lambda_da=0.1, seed=42):
    """Run training and evaluation for a single split with proper validation - FIXED"""
    
    print(f"\n{'=' * 80}")
    print(f"STARTING PHD EXPERIMENT: {split_name}")
    print(f"{'=' * 80}")
    print(f"Training bearings ({len(train_bearings)}): {train_bearings}")
    print(f"Testing bearings ({len(test_bearings)}): {test_bearings}")
    print(f"Model: {model_type}, Wavelet: {include_wavelet}, DA: {use_da}")
    print(f"Sequence Length: {sequence_length}, EWS Window: {ews_window_size}")
    print(f"Validation split: {val_split*100}%, λ_da: {lambda_da}, Seed: {seed}")
    print(f"{'=' * 80}")
    
    # Set seeds for reproducibility
    set_seed(seed)
    
    DATA_ROOT = "/kaggle/input/xj-dataset/XJTU-SY_Bearing_Datasets"
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Using device: {device}")
    
    # Create all directories
    create_directories()
    
    # Generate unique experiment ID
    from datetime import datetime
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    experiment_id = f"{split_name.replace(' ', '_')}_{model_type}_wavelet{include_wavelet}_DA{use_da}_seq{sequence_length}_win{ews_window_size}_val{val_split}_seed{seed}_{timestamp}"
    
    # Results files
    results_dir = "/kaggle/working/results"
    results_file = os.path.join(results_dir, f"results_{experiment_id}.csv")
    metrics_file = os.path.join(results_dir, f"metrics_{experiment_id}.json")
    plot_file = os.path.join("/kaggle/working/plots", f"plots_{experiment_id}.png")
    model_file = os.path.join("/kaggle/working/models", f"model_{experiment_id}.pth")
    config_file = os.path.join("/kaggle/working", f"config_{experiment_id}.txt")
    
    # Save configuration
    config = {
        'split_name': split_name,
        'train_bearings': train_bearings,
        'test_bearings': test_bearings,
        'model_type': model_type,
        'include_wavelet': include_wavelet,
        'use_da': use_da,
        'sequence_length': sequence_length,
        'ews_window_size': ews_window_size,
        'ews_step_size': ews_step_size,
        'val_split': val_split,
        'lambda_da': lambda_da,
        'seed': seed,
        'device': str(device),
        'timestamp': timestamp,
    }
    
    with open(config_file, 'w') as f:
        for key, value in config.items():
            f.write(f"{key}: {value}\n")
    
    # Check if results already exist
    if skip_if_exists and os.path.exists(results_file):
        print(f"✓ Results already exist. Skipping experiment: {experiment_id}")
        return None
    
    # Create datasets
    print("\n📊 Creating/loading datasets...")
    full_train_dataset = EnhancedXJTU_EWS_Dataset(
        data_root=DATA_ROOT,
        bearing_list=train_bearings,
        window_size=sequence_length,
        step_size=max(1, sequence_length // 2),
        max_files_per_bearing=None,
        use_both_channels=True,
        fusion_method='concat',
        enable_caching=True,
        cache_dir="/kaggle/working/cache",
        include_wavelet=include_wavelet,
        ews_window_size=ews_window_size,
        ews_step_size=ews_step_size
    )
    
    test_dataset = EnhancedXJTU_EWS_Dataset(
        data_root=DATA_ROOT,
        bearing_list=test_bearings,
        window_size=sequence_length,
        step_size=max(1, sequence_length // 2),
        max_files_per_bearing=None,
        use_both_channels=True,
        fusion_method='concat',
        enable_caching=True,
        cache_dir="/kaggle/working/cache",
        include_wavelet=include_wavelet,
        ews_window_size=ews_window_size,
        ews_step_size=ews_step_size
    )
    
    # Check dataset validity
    if len(full_train_dataset) == 0:
        print("❌ ERROR: Empty training dataset!")
        return None
    
    if len(test_dataset) == 0:
        print("❌ ERROR: Empty test dataset!")
        return None
    
    print(f"✓ Full training dataset: {len(full_train_dataset)} sequences")
    print(f"✓ Test dataset: {len(test_dataset)} sequences")
    
    # Split into train/validation
    print(f"\n🔀 Splitting into train/validation ({val_split*100}% validation)...")
    train_indices, val_indices = create_balanced_train_val_split(
        full_train_dataset, 
        val_split=val_split,
        random_seed=seed
    )
    
    train_dataset = Subset(full_train_dataset, train_indices)
    val_dataset = Subset(full_train_dataset, val_indices)
    
    print(f"✓ Training split: {len(train_dataset)} sequences")
    print(f"✓ Validation split: {len(val_dataset)} sequences")
    
    # Get feature dimension
    feature_dim = full_train_dataset.get_feature_dimension()
    print(f"✓ Feature dimension: {feature_dim}")
    print(f"✓ Input shape: [batch, {sequence_length}, {feature_dim}]")
    
    # Create model
    print(f"\n🏗️ Creating model: {model_type}")
    
    if model_type == 'lstm_da':
        model = EWS_LSTM_Attention_DA(
            input_size=feature_dim,
            hidden_size=128,
            num_layers=2,
            dropout=0.3,
            use_da=use_da
        )
    elif model_type == 'bilstm_attn':
        model = EWS_LSTM_Attention(
            input_size=feature_dim,
            hidden_size=128,
            num_layers=2,
            dropout=0.3
        )
    elif model_type == 'bilstm_noattn':
        model = EWS_Bidirectional_LSTM_NoAttention(
            input_size=feature_dim,
            hidden_size=128,
            num_layers=2,
            dropout=0.3
        )
    elif model_type == 'lstm_attn':
        model = EWS_Unidirectional_LSTM_Attention(
            input_size=feature_dim,
            hidden_size=128,
            num_layers=2,
            dropout=0.3
        )
    elif model_type == 'lstm_noattn':
        model = EWS_Unidirectional_LSTM_NoAttention(
            input_size=feature_dim,
            hidden_size=128,
            num_layers=2,
            dropout=0.3
        )
    elif model_type == 'simple_lstm':
        model = Simple_LSTM(
            input_size=feature_dim,
            hidden_size=128,
            num_layers=2,
            dropout=0.3
        )
    else:
        print(f"⚠ Unknown model type: {model_type}, using lstm_da as default")
        model = EWS_LSTM_Attention_DA(
            input_size=feature_dim,
            hidden_size=128,
            num_layers=2,
            dropout=0.3,
            use_da=use_da
        )
    
    print(f"✓ Model created: {model.__class__.__name__}")
    print(f"✓ Total parameters: {sum(p.numel() for p in model.parameters()):,}")
    
    # Create data loaders
    batch_size = 32
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=0, pin_memory=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=0, pin_memory=True)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=0, pin_memory=True)
    
    # Train model
    print(f"\n🚀 Starting training...")
    trained_model, train_losses, val_losses, domain_losses = train_model_with_validation(
        model=model,
        train_loader=train_loader,
        val_loader=val_loader,
        epochs=100,
        lr=0.001,
        device=device,
        early_stopping_patience=20,
        use_da=use_da,
        lambda_da=lambda_da
    )
    
    # Evaluate on test set
    print(f"\n📈 Evaluating on test set...")
    trained_model.eval()
    preds, targets, bearing_ids_list, max_ruls_list = [], [], [], []
    
    with torch.no_grad():
        for data, targ, b_ids, m_ruls in test_loader:
            data = data.to(device)
            out = trained_model(data)
            if isinstance(out, tuple):
                out = out[0]  # Take RUL prediction
            
            for i in range(len(out)):
                pred_norm = out[i].item()
                targ_norm = targ[i].item()
                max_rul = m_ruls[i].item()
                
                # Convert normalized predictions back to actual RUL
                pred_actual = pred_norm * max_rul
                targ_actual = targ_norm * max_rul
                
                preds.append(pred_actual)
                targets.append(targ_actual)
                bearing_ids_list.append(b_ids[i])
                max_ruls_list.append(max_rul)
    
    preds = np.array(preds)
    targets = np.array(targets)
    
    # Calculate comprehensive metrics
    print(f"\n📊 Calculating metrics...")
    metrics = calculate_comprehensive_metrics(preds, targets)
    
    # Print results
    print(f"\n{'=' * 80}")
    print(f"🎯 RESULTS FOR: {split_name}")
    print(f"{'=' * 80}")
    print(f"MAE:          {metrics['MAE']:8.2f} minutes")
    print(f"RMSE:         {metrics['RMSE']:8.2f} minutes")
    print(f"R² Score:     {metrics['R2']:8.4f}")
    print(f"Pearson r:    {metrics.get('Pearson_r', 0):8.4f}")
    print(f"Spearman ρ:   {metrics.get('Spearman_rho', 0):8.4f}")
    
    if 'MAPE' in metrics and not np.isnan(metrics['MAPE']):
        print(f"MAPE:         {metrics['MAPE']:8.2f}%")
    
    if 'Within_10%' in metrics:
        print(f"Within 10%:   {metrics['Within_10%']:8.2f}%")
        print(f"Within 20%:   {metrics['Within_20%']:8.2f}%")
        print(f"Within 30%:   {metrics['Within_30%']:8.2f}%")
    
    print(f"Monotonicity: {metrics.get('Monotonicity_pred', 0):8.4f}")
    print(f"Bias:         {metrics.get('Bias', 0):8.2f}")
    print(f"{'=' * 80}")
    
    # Save results
    print(f"\n💾 Saving results...")
    
    # Save detailed predictions
    results_df = pd.DataFrame({
        'experiment_id': experiment_id,
        'bearing_id': bearing_ids_list,
        'max_rul': max_ruls_list,
        'actual_rul': targets,
        'predicted_rul': preds,
        'error': preds - targets,
        'abs_error': np.abs(preds - targets),
        'rel_error': np.abs(preds - targets) / (targets + 1e-10),
        'model_type': model_type,
        'include_wavelet': include_wavelet,
        'use_da': use_da,
        'sequence_length': sequence_length,
        'ews_window_size': ews_window_size,
        'val_split': val_split,
        'lambda_da': lambda_da,
        'seed': seed,
    })
    
    results_df.to_csv(results_file, index=False)
    
    # Save metrics
    import json
    with open(metrics_file, 'w') as f:
        json.dump(metrics, f, indent=2)
    
    # Save model
    torch.save({
        'model_state_dict': trained_model.state_dict(),
        'model_type': model_type,
        'feature_dim': feature_dim,
        'config': config,
        'metrics': metrics,
        'train_losses': train_losses,
        'val_losses': val_losses,
        'domain_losses': domain_losses if domain_losses else [],
    }, model_file)
    
    # Create comprehensive plot
    print(f"\n🎨 Creating comprehensive visualization...")
    fig = plt.figure(figsize=(20, 12))
    
    # 1. Training history
    ax1 = plt.subplot(3, 4, 1)
    ax1.plot(train_losses, label='Train', linewidth=2, alpha=0.8)
    ax1.plot(val_losses, label='Validation', linewidth=2, alpha=0.8)
    if domain_losses:
        ax1.plot(domain_losses, label='Domain', linewidth=2, alpha=0.8, linestyle='--')
    ax1.set_title('Training History', fontsize=12, fontweight='bold')
    ax1.set_xlabel('Epoch')
    ax1.set_ylabel('Loss')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    # 2. Predictions vs Actual
    ax2 = plt.subplot(3, 4, 2)
    scatter = ax2.scatter(targets, preds, c=np.abs(preds - targets), alpha=0.6, s=30, cmap='viridis')
    max_val = max(targets.max(), preds.max())
    ax2.plot([0, max_val], [0, max_val], 'r--', label='Perfect', linewidth=2)
    ax2.set_xlabel('Actual RUL (minutes)', fontsize=10)
    ax2.set_ylabel('Predicted RUL (minutes)', fontsize=10)
    ax2.set_title(f'Predictions vs Actual\nR² = {metrics["R2"]:.3f}', fontsize=12, fontweight='bold')
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    plt.colorbar(scatter, ax=ax2, label='Absolute Error')
    
    # 3. Error distribution
    ax3 = plt.subplot(3, 4, 3)
    errors = preds - targets
    ax3.hist(errors, bins=50, alpha=0.7, edgecolor='black', density=True)
    ax3.axvline(0, color='r', linestyle='--', linewidth=2, label='Zero error')
    ax3.axvline(errors.mean(), color='g', linestyle='-', linewidth=2, label=f'Mean: {errors.mean():.1f}')
    ax3.set_xlabel('Error (Predicted - Actual)', fontsize=10)
    ax3.set_ylabel('Density', fontsize=10)
    ax3.set_title(f'Error Distribution\nMAE = {metrics["MAE"]:.1f}', fontsize=12, fontweight='bold')
    ax3.legend()
    ax3.grid(True, alpha=0.3)
    
    # 4. MAE by bearing
    ax4 = plt.subplot(3, 4, 4)
    unique_bearings = sorted(set(bearing_ids_list))
    bearing_maes = []
    for b in unique_bearings:
        mask = np.array(bearing_ids_list) == b
        if np.sum(mask) > 0:
            mae = np.mean(np.abs(preds[mask] - targets[mask]))
            bearing_maes.append(mae)
    
    bars = ax4.bar(range(len(unique_bearings)), bearing_maes)
    ax4.set_xlabel('Bearing', fontsize=10)
    ax4.set_ylabel('MAE (minutes)', fontsize=10)
    ax4.set_title('MAE by Bearing', fontsize=12, fontweight='bold')
    ax4.set_xticks(range(len(unique_bearings)))
    ax4.set_xticklabels(unique_bearings, rotation=45, ha='right', fontsize=8)
    ax4.grid(True, alpha=0.3, axis='y')
    
    # Color bars by bearing group
    for idx, b in enumerate(unique_bearings):
        if 'Bearing1' in b:
            bars[idx].set_color('blue')
        elif 'Bearing2' in b:
            bars[idx].set_color('green')
        elif 'Bearing3' in b:
            bars[idx].set_color('orange')
    
    # 5. Residual plot
    ax5 = plt.subplot(3, 4, 5)
    ax5.scatter(targets, errors, alpha=0.6, s=20)
    ax5.axhline(0, color='r', linestyle='--', linewidth=2)
    ax5.set_xlabel('Actual RUL', fontsize=10)
    ax5.set_ylabel('Residual', fontsize=10)
    ax5.set_title('Residual Plot', fontsize=12, fontweight='bold')
    ax5.grid(True, alpha=0.3)
    
    # 6. Learning curve (log scale)
    ax6 = plt.subplot(3, 4, 6)
    ax6.semilogy(train_losses, label='Train', linewidth=2)
    ax6.semilogy(val_losses, label='Validation', linewidth=2)
    ax6.set_xlabel('Epoch', fontsize=10)
    ax6.set_ylabel('Loss (log scale)', fontsize=10)
    ax6.set_title('Learning Curve (Log Scale)', fontsize=12, fontweight='bold')
    ax6.legend()
    ax6.grid(True, alpha=0.3)
    
    # 7. Monotonicity comparison
    ax7 = plt.subplot(3, 4, 7)
    sample_indices = np.arange(len(preds))
    ax7.plot(sample_indices, sorted(preds), 'b-', label='Predicted', linewidth=2, alpha=0.7)
    ax7.plot(sample_indices, sorted(targets), 'r-', label='Actual', linewidth=2, alpha=0.7)
    ax7.set_xlabel('Sample Index (sorted)', fontsize=10)
    ax7.set_ylabel('RUL (minutes)', fontsize=10)
    ax7.set_title(f'Monotonicity\nPred: {metrics.get("Monotonicity_pred", 0):.3f}, True: {metrics.get("Monotonicity_true", 0):.3f}', 
                  fontsize=12, fontweight='bold')
    ax7.legend()
    ax7.grid(True, alpha=0.3)
    
    # 8. Metrics comparison
    ax8 = plt.subplot(3, 4, 8)
    metric_names = ['MAE', 'RMSE', 'R²', 'Within_10%']
    metric_values = [metrics['MAE'], metrics['RMSE'], metrics['R2'], metrics.get('Within_10%', 0)]
    colors = ['blue', 'green', 'red', 'orange']
    bars = ax8.bar(metric_names, metric_values, color=colors, alpha=0.7)
    ax8.set_ylabel('Value', fontsize=10)
    ax8.set_title('Key Metrics', fontsize=12, fontweight='bold')
    ax8.grid(True, alpha=0.3, axis='y')
    
    # Add value labels
    for i, (bar, value) in enumerate(zip(bars, metric_values)):
        height = bar.get_height()
        ax8.text(bar.get_x() + bar.get_width()/2., height + 0.01,
                f'{value:.2f}' if metric_names[i] != 'R²' else f'{value:.3f}',
                ha='center', va='bottom', fontsize=9)
        
    # 9. Error vs RUL magnitude
    ax9 = plt.subplot(3, 4, 9)
    abs_errors = np.abs(preds - targets)
    ax9.scatter(targets, abs_errors, alpha=0.6, s=20)
    ax9.set_xlabel('Actual RUL', fontsize=10)
    ax9.set_ylabel('Absolute Error', fontsize=10)
    ax9.set_title('Error vs RUL Magnitude', fontsize=12, fontweight='bold')
    
    # Add trend line
    if len(targets) > 1:
        z = np.polyfit(targets, abs_errors, 1)
        p = np.poly1d(z)
        ax9.plot(sorted(targets), p(sorted(targets)), "r--", linewidth=2, 
                label=f'Trend: y={z[0]:.3f}x+{z[1]:.3f}')
        ax9.legend()
    
    ax9.grid(True, alpha=0.3)
    
    # 10. Cumulative error distribution
    ax10 = plt.subplot(3, 4, 10)
    sorted_errors = np.sort(np.abs(errors))
    cum_dist = np.arange(1, len(sorted_errors) + 1) / len(sorted_errors)
    ax10.plot(sorted_errors, cum_dist, 'b-', linewidth=2)
    ax10.axvline(metrics['MAE'], color='r', linestyle='--', label=f'MAE={metrics["MAE"]:.1f}')
    ax10.axvline(metrics['RMSE'], color='g', linestyle='--', label=f'RMSE={metrics["RMSE"]:.1f}')
    ax10.set_xlabel('Absolute Error', fontsize=10)
    ax10.set_ylabel('Cumulative Probability', fontsize=10)
    ax10.set_title('Cumulative Error Distribution', fontsize=12, fontweight='bold')
    ax10.legend()
    ax10.grid(True, alpha=0.3)
    
    # 11. Text summary
    ax11 = plt.subplot(3, 4, (11, 12))
    ax11.axis('off')
    
    summary_text = (
        f"EXPERIMENT SUMMARY\n"
        f"{'='*40}\n"
        f"Split: {split_name}\n"
        f"Model: {model_type}\n"
        f"Wavelet: {include_wavelet}\n"
        f"DA: {use_da} (λ={lambda_da})\n"
        f"Seq Length: {sequence_length}\n"
        f"Val Split: {val_split*100}%\n"
        f"Seed: {seed}\n"
        f"{'='*40}\n"
        f"Train Samples: {len(train_dataset)}\n"
        f"Val Samples: {len(val_dataset)}\n"
        f"Test Samples: {len(test_dataset)}\n"
        f"{'='*40}\n"
        f"MAE: {metrics['MAE']:.2f} min\n"
        f"RMSE: {metrics['RMSE']:.2f} min\n"
        f"R²: {metrics['R2']:.4f}\n"
        f"Pearson r: {metrics.get('Pearson_r', 0):.4f}\n"
        f"Within 10%: {metrics.get('Within_10%', 0):.1f}%\n"
        f"Within 20%: {metrics.get('Within_20%', 0):.1f}%\n"
        f"Experiment ID: {experiment_id}"
    )
    
    ax11.text(0, 1, summary_text, fontsize=10, fontfamily='monospace',
              verticalalignment='top', horizontalalignment='left',
              transform=ax11.transAxes)
    
    plt.suptitle(f'PhD Experiment: {split_name} - {model_type}', 
                 fontsize=16, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.savefig(plot_file, dpi=150, bbox_inches='tight')
    plt.close()
    
    # Print file sizes
    print(f"\n✅ Experiment completed successfully!")
    print(f"📁 Results saved to:")
    print(f"   📄 Results: {results_file}")
    print(f"   📊 Metrics: {metrics_file}")
    print(f"   🖼️  Plots: {plot_file}")
    print(f"   🤖 Model: {model_file}")
    print(f"   ⚙️  Config: {config_file}")
    
    return {
        'experiment_id': experiment_id,
        'split_name': split_name,
        'model_type': model_type,
        'include_wavelet': include_wavelet,
        'use_da': use_da,
        'lambda_da': lambda_da,
        'metrics': metrics,
        'train_samples': len(train_dataset),
        'val_samples': len(val_dataset),
        'test_samples': len(test_dataset),
        'results_file': results_file,
        'model_file': model_file,
    }

def run_selected_split(split_index, skip_if_exists=True, model_type='lstm_da', use_da=True,
                      include_wavelet=True, sequence_length=10, ews_window_size=1024,
                      ews_step_size=1024, val_split=0.2, model_seed=42):
    """Run a specific split by index with proper validation"""
    set_seed(model_seed)
    DATA_ROOT = "/kaggle/input/xj-dataset/XJTU-SY_Bearing_Datasets"
    
    ALL_BEARINGS = [
        'Bearing1_1', 'Bearing1_2', 'Bearing1_3', 'Bearing1_4', 'Bearing1_5',
        'Bearing2_1', 'Bearing2_2', 'Bearing2_3',
        'Bearing3_2', 'Bearing3_3', 'Bearing3_4'
    ]
    
    available_bearings = []
    for b in ALL_BEARINGS:
        cond = ('35Hz12kN' if 'Bearing1_' in b else
                '37.5Hz11kN' if 'Bearing2_' in b else '40Hz10kN')
        if os.path.exists(os.path.join(DATA_ROOT, cond, b)):
            available_bearings.append(b)
    
    bearing1_set = [b for b in available_bearings if 'Bearing1_' in b]
    bearing2_set = [b for b in available_bearings if 'Bearing2_' in b]
    bearing3_set = [b for b in available_bearings if 'Bearing3_' in b]
    
    split_options = [
        ("Cross-condition 1", bearing1_set + bearing2_set, bearing3_set),
        ("Cross-condition 2", bearing1_set + bearing3_set, bearing2_set),
        ("Cross-condition 3", bearing2_set + bearing3_set, bearing1_set),
    ]
    
    if split_index < 0 or split_index >= len(split_options):
        print(f"Invalid split index. Must be between 0 and {len(split_options) - 1}")
        return
    
    split_name, train_bearings, test_bearings = split_options[split_index]
    return run_single_split(split_name, train_bearings, test_bearings, skip_if_exists, 
                          model_type, include_wavelet, use_da, sequence_length, 
                          ews_window_size, ews_step_size, val_split)
# ============================================================================
# RAW VIBRATION DATASET FOR ABLATION STUDY (NO EWS FEATURES) - FIXED
# ============================================================================
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import pandas as pd
import os
import logging
import pickle
import hashlib
import json
from torch.utils.data import Dataset, DataLoader, Subset
from collections import defaultdict
import matplotlib.pyplot as plt

logger = logging.getLogger(__name__)

def get_raw_dataset_cache_key(data_root, bearing_list, window_size=10, step_size=5,
                             max_files_per_bearing=None, use_both_channels=True,
                             max_samples=4096, downsample_factor=8):
    """Generate cache key for raw vibration dataset"""
    params = {
        'data_root': data_root,
        'bearing_list': sorted(bearing_list),
        'window_size': window_size,
        'step_size': step_size,
        'max_files_per_bearing': max_files_per_bearing,
        'use_both_channels': use_both_channels,
        'max_samples': max_samples,
        'downsample_factor': downsample_factor,
        'code_version': '1.0'
    }
    params_str = json.dumps(params, sort_keys=True, default=str)
    return hashlib.md5(params_str.encode()).hexdigest()

class RawVibrationDataset(Dataset):
    """Dataset using raw vibration signals instead of EWS features - FIXED"""
    def __init__(self, data_root, bearing_list, window_size=10, step_size=5,
                 max_files_per_bearing=None, use_both_channels=True,
                 max_samples=4096, downsample_factor=8, enable_caching=True,
                 cache_dir="/kaggle/working/cache"):
        """
        Args:
            max_samples: 4096 samples = 0.16s at 25600Hz (matches EWS window of 1024 samples)
            downsample_factor: 8 = 25600Hz → 3200Hz (for computational efficiency)
        """
        self.data_root = data_root
        self.bearing_list = bearing_list
        self.window_size = window_size
        self.step_size = step_size
        self.max_files = max_files_per_bearing
        self.use_both_channels = use_both_channels
        self.max_samples = max_samples
        self.downsample_factor = downsample_factor
        self.enable_caching = enable_caching
        self.cache_dir = cache_dir
        
        # Use EXACT SAME RUL calculation as EWS dataset
        self.manual_settings = {
            'Bearing1_1': {'multiplier': 3.5, 'manual_start': 75},
            'Bearing1_2': {'multiplier': 3.5, 'manual_start': 54},
            'Bearing1_3': {'multiplier': 5.0, 'manual_start': 59},
            'Bearing1_4': {'multiplier': 2.5, 'manual_start': 75},
            'Bearing1_5': {'multiplier': 2.5, 'manual_start': 32},
            'Bearing2_1': {'multiplier': 3.0, 'manual_start': 453},
            'Bearing2_2': {'multiplier': 3.0, 'manual_start': 47},
            'Bearing2_3': {'multiplier': 3.0, 'manual_start': 59},
            
            'Bearing3_2': {'multiplier': 3.0, 'manual_start': 466},
            'Bearing3_3': {'multiplier': 3.0, 'manual_start': 342},
            'Bearing3_4': {'multiplier': 3.0, 'manual_start': 45},
            
        }
        
        self.sequences = []
        self.rul_targets = []
        self.bearing_ids = []
        self.max_ruls = []
        self.degradation_starts = {}
        
        # Try to load from cache
        if self.enable_caching:
            cache_key = get_raw_dataset_cache_key(
                data_root, bearing_list, window_size, step_size,
                max_files_per_bearing, use_both_channels,
                max_samples, downsample_factor
            )
            cache_file = os.path.join(cache_dir, f"raw_dataset_{cache_key}.pkl")
            
            if os.path.exists(cache_file):
                try:
                    with open(cache_file, 'rb') as f:
                        cache_data = pickle.load(f)
                    
                    self.sequences = [np.array(seq, dtype=np.float32) for seq in cache_data['sequences']]
                    self.rul_targets = cache_data['rul_targets']
                    self.bearing_ids = cache_data['bearing_ids']
                    self.max_ruls = cache_data['max_ruls']
                    self.degradation_starts = cache_data['degradation_starts']
                    
                    logger.info(f"✓ Loaded raw dataset from cache: {len(self.sequences)} sequences")
                    return
                except Exception as e:
                    logger.warning(f"Failed to load cache: {e}")
        
        # Process if no cache
        self._process_all_bearings()
        
        # Save to cache
        if self.enable_caching and len(self.sequences) > 0:
            cache_key = get_raw_dataset_cache_key(
                data_root, bearing_list, window_size, step_size,
                max_files_per_bearing, use_both_channels,
                max_samples, downsample_factor
            )
            cache_file = os.path.join(cache_dir, f"raw_dataset_{cache_key}.pkl")
            
            cache_data = {
                'sequences': [seq.tolist() for seq in self.sequences],
                'rul_targets': self.rul_targets,
                'bearing_ids': self.bearing_ids,
                'max_ruls': self.max_ruls,
                'degradation_starts': self.degradation_starts,
            }
            
            with open(cache_file, 'wb') as f:
                pickle.dump(cache_data, f, protocol=pickle.HIGHEST_PROTOCOL)
            
            logger.info(f"✓ Saved raw dataset to cache: {cache_file}")
        
        logger.info(f"✓ Raw vibration dataset loaded: {len(self.sequences)} sequences")
        if len(self.sequences) > 0:
            logger.info(f"  Sequence shape: {self.sequences[0].shape}")
    
    def _get_bearing_path(self, bearing_name):
        """Get full path - EXACT SAME as EWS dataset"""
        if 'Bearing1_' in bearing_name:
            cond = '35Hz12kN'
        elif 'Bearing2_' in bearing_name:
            cond = '37.5Hz11kN'
        else:
            cond = '40Hz10kN'
        return os.path.join(self.data_root, cond, bearing_name)
    
    def _read_raw_signal(self, filepath):
        """Read raw signal - EXACT SAME pattern as EWS dataset (no headers!)"""
        try:
            # CRITICAL FIX: XJTU files have NO headers
            df = pd.read_csv(filepath)
            
            if df.shape[1] >= 2:
                h = df.iloc[:, 0].values.astype(np.float32)
                v = df.iloc[:, 1].values.astype(np.float32)
            else:
                h = df.iloc[:, 0].values.astype(np.float32)
                v = h.copy()
            
            # For raw ablation: keep raw values (no normalization here!)
            # We'll normalize later in the model if needed
            
            # Take first max_samples
            h = h[:self.max_samples]
            v = v[:self.max_samples]
            
            # Downsample
            if self.downsample_factor > 1:
                h = h[::self.downsample_factor]
                v = v[::self.downsample_factor]
            
            if self.use_both_channels:
                # Stack as (2, time_steps)
                return np.stack([h, v], axis=0)
            else:
                # Use only horizontal channel
                return h.reshape(1, -1)
                
        except Exception as e:
            logger.warning(f"Error reading {filepath}: {e}")
            return None
    
    def _calculate_piecewise_rul_pdf(self, csv_files, bearing_name):
        """Calculate RUL - EXACT SAME method as EWS dataset"""
        total_files = len(csv_files)
        
        # Calculate amplitude from RAW SIGNAL (not normalized)
        amplitudes = []
        for path in csv_files:
            # Read without processing to get raw amplitude
            try:
                df = pd.read_csv(path)
                if df.shape[1] >= 2:
                    sig = df.iloc[:, 0].values.astype(np.float32)
                else:
                    sig = df.iloc[:, 0].values.astype(np.float32)
                
                # Use maximum absolute value (same as EWS dataset)
                amp = np.max(np.abs(sig[:1000]))  # Use first 1000 samples for speed
                amplitudes.append(amp)
            except:
                amplitudes.append(0.0)
        
        amplitudes = np.array(amplitudes)
        
        # Same baseline calculation
        baseline_samples = min(50, total_files // 2)
        healthy_vals = amplitudes[:baseline_samples]
        
        if len(healthy_vals) > 0:
            healthy_baseline = np.median(healthy_vals)
        else:
            healthy_baseline = 1.0
        
        settings = self.manual_settings.get(bearing_name, 
                                          {'multiplier': 3.0, 'manual_start': int(total_files * 0.7)})
        multiplier = settings['multiplier']
        manual_start = settings['manual_start']
        
        degradation_start = manual_start
        if degradation_start < baseline_samples:
            degradation_start = baseline_samples + 10
        
        min_degradation_phase = int(max(20, total_files * 0.2))
        degradation_start = int(degradation_start)
        max_rul = total_files - degradation_start-1
        
        if max_rul < min_degradation_phase:
            degradation_start = total_files - min_degradation_phase
            max_rul = total_files - degradation_start-1
        
        if degradation_start < baseline_samples + 10:
            degradation_start = baseline_samples + 20
            max_rul = total_files - degradation_start-1
        
        # Store degradation start
        self.degradation_starts[bearing_name] = degradation_start
        
        return degradation_start, max_rul
    
    def _process_all_bearings(self):
        """Process all bearings - similar to EWS dataset"""
        for bearing_name in self.bearing_list:
            path = self._get_bearing_path(bearing_name)
            if not os.path.exists(path):
                logger.warning(f"Skipping {bearing_name} - path not found")
                continue
            
            try:
                files = sorted([f for f in os.listdir(path) if f.endswith('.csv')],
                             key=lambda x: int(x.split('.')[0]))
                csv_files = [os.path.join(path, f) for f in files]
                
                if self.max_files:
                    csv_files = csv_files[:self.max_files]
            except Exception as e:
                logger.error(f"Error listing files for {bearing_name}: {e}")
                continue
            
            if len(csv_files) < self.window_size:
                logger.warning(f"Skipping {bearing_name} - too few files")
                continue
            
            logger.info(f"Processing raw data for {bearing_name}: {len(csv_files)} files")
            
            # Calculate RUL
            degradation_start, max_rul = self._calculate_piecewise_rul_pdf(csv_files, bearing_name)
            
            sequences_added = 0
            for start_idx in range(0, len(csv_files) - self.window_size + 1, self.step_size):
                end_idx = start_idx + self.window_size
                raw_seq = []
                valid = True
                
                for i in range(start_idx, end_idx):
                    sig = self._read_raw_signal(csv_files[i])
                    if sig is None:
                        valid = False
                        break
                    
                    # CRITICAL: Normalize each signal to zero mean, unit variance
                    # This helps CNN training but preserves raw characteristics
                    sig = (sig - np.mean(sig, axis=1, keepdims=True)) / (np.std(sig, axis=1, keepdims=True) + 1e-8)
                    
                    raw_seq.append(sig)
                
                if not valid or len(raw_seq) != self.window_size:
                    continue
                
                # Stack to shape: (window_size, channels, time_steps)
                raw_seq = np.stack(raw_seq, axis=0).astype(np.float32)
                
                # Calculate RUL (same as EWS dataset)
                last_idx = end_idx - 1
                if last_idx < degradation_start:
                    rul_minutes = max_rul
                else:
                    rul_minutes = len(csv_files) - last_idx - 1
                
                rul_norm = rul_minutes / max_rul if max_rul > 0 else 0.0
                
                self.sequences.append(raw_seq)
                self.rul_targets.append(float(rul_norm))
                self.bearing_ids.append(bearing_name)
                self.max_ruls.append(max_rul)
                sequences_added += 1
            
            logger.info(f"  Added {sequences_added} sequences")
    
    def get_sequence_shape(self):
        """Get shape of sequences"""
        if len(self.sequences) == 0:
            channels = 2 if self.use_both_channels else 1
            time_steps = self.max_samples // self.downsample_factor
            return (self.window_size, channels, time_steps)
        return self.sequences[0].shape
    
    def __len__(self):
        return len(self.sequences)
    
    def __getitem__(self, idx):
        return (torch.FloatTensor(self.sequences[idx]),
                torch.FloatTensor([self.rul_targets[idx]]),
                self.bearing_ids[idx],
                self.max_ruls[idx])

# ============================================================================
# IMPROVED 1D CNN FOR RAW VIBRATION
# ============================================================================
class RawVibration_1DCNN(nn.Module):
    """Improved 1D CNN for raw vibration data with better architecture"""
    def __init__(self, input_channels=2, sequence_length=10, time_steps=512):
        super().__init__()
        
        self.sequence_length = sequence_length
        
        # Improved CNN architecture with residual connections
        self.conv1 = nn.Sequential(
            nn.Conv1d(input_channels, 64, kernel_size=7, padding=3),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.MaxPool1d(2),  # 512 → 256
        )
        
        self.conv2 = nn.Sequential(
            nn.Conv1d(64, 128, kernel_size=5, padding=2),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.MaxPool1d(2),  # 256 → 128
        )
        
        self.conv3 = nn.Sequential(
            nn.Conv1d(128, 256, kernel_size=3, padding=1),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.MaxPool1d(2),  # 128 → 64
        )
        
        self.conv4 = nn.Sequential(
            nn.Conv1d(256, 512, kernel_size=3, padding=1),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.AdaptiveAvgPool1d(1)  # Global average pooling
        )
        
        # Calculate output dimension
        with torch.no_grad():
            test_input = torch.randn(1, input_channels, time_steps)
            out1 = self.conv1(test_input)
            out2 = self.conv2(out1)
            out3 = self.conv3(out2)
            out4 = self.conv4(out3)
            cnn_output_dim = out4.shape[1]  # Should be 512
        
        # Temporal aggregation with attention
        self.temporal_attention = nn.Sequential(
            nn.Linear(cnn_output_dim, 128),
            nn.Tanh(),
            nn.Linear(128, 1)
        )
        
        # Final prediction
        self.fc = nn.Sequential(
            nn.Linear(cnn_output_dim, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.3),
            
            nn.Linear(256, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.3),
            
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.2),
            
            nn.Linear(64, 1),
            nn.Sigmoid()
        )
        
        logger.info(f"Created RawVibration_1DCNN: channels={input_channels}, "
                   f"seq_len={sequence_length}, time_steps={time_steps}")
    
    def forward(self, x):
        # x shape: (batch, seq_len, channels, time_steps)
        batch_size, seq_len, channels, time_steps = x.shape
        
        # Process each timestep through shared CNN
        cnn_features = []
        for i in range(seq_len):
            # Get signal for this timestep: (batch, channels, time_steps)
            x_i = x[:, i, :, :]
            
            # CNN processing
            out1 = self.conv1(x_i)
            out2 = self.conv2(out1)
            out3 = self.conv3(out2)
            out4 = self.conv4(out3)  # (batch, 512, 1)
            
            cnn_features.append(out4.squeeze(-1))  # (batch, 512)
        
        # Stack features: (batch, seq_len, 512)
        cnn_features = torch.stack(cnn_features, dim=1)
        
        # Temporal attention
        attention_weights = self.temporal_attention(cnn_features)  # (batch, seq_len, 1)
        attention_weights = torch.softmax(attention_weights, dim=1)
        
        # Weighted sum
        context = torch.sum(attention_weights * cnn_features, dim=1)  # (batch, 512)
        
        # Final prediction
        output = self.fc(context)
        return output

# ============================================================================
# IMPROVED RAW VIBRATION TRAINING
# ============================================================================
def train_raw_vibration_model(model, train_loader, val_loader, epochs=100, lr=0.001,
                             device='cuda', early_stopping_patience=20):
    """Training function for raw vibration model with proper validation"""
    model.to(device)
    
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    criterion = nn.MSELoss()
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.5, patience=10
    )
    
    best_val_loss = float('inf')
    patience_counter = 0
    train_losses, val_losses = [], []
    
    logger.info(f"Starting raw vibration training on {device}")
    
    for epoch in range(epochs):
        model.train()
        train_loss = 0.0
        
        for batch_idx, (data, targets, _, _) in enumerate(train_loader):
            data, targets = data.to(device), targets.to(device)
            
            optimizer.zero_grad()
            predictions = model(data)
            loss = criterion(predictions, targets)
            loss.backward()
            
            # Gradient clipping
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            
            optimizer.step()
            train_loss += loss.item()
        
        # Validation
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for data, targets, _, _ in val_loader:
                data, targets = data.to(device), targets.to(device)
                predictions = model(data)
                val_loss += criterion(predictions, targets).item()
        
        avg_train_loss = train_loss / len(train_loader)
        avg_val_loss = val_loss / len(val_loader)
        train_losses.append(avg_train_loss)
        val_losses.append(avg_val_loss)
        
        scheduler.step(avg_val_loss)
        
        # Early stopping
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            best_state = model.state_dict().copy()
            patience_counter = 0
            
            # Save checkpoint
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'val_loss': avg_val_loss,
                'train_loss': avg_train_loss,
            }, "/kaggle/working/best_raw_model.pth")
        else:
            patience_counter += 1
        
        # Logging
        if (epoch + 1) % 5 == 0 or epoch == 0:
            logger.info(f"Epoch {epoch + 1:3d}/{epochs} | "
                       f"Train: {avg_train_loss:.6f} | "
                       f"Val: {avg_val_loss:.6f} | "
                       f"Patience: {patience_counter:2d}")
        
        if patience_counter >= early_stopping_patience:
            logger.info(f"Early stopping at epoch {epoch + 1}")
            break
    
    # Load best model
    model.load_state_dict(best_state)
    return model, train_losses, val_losses

# ============================================================================
# FIXED RAW VIBRATION ABLATION RUNNER
# ============================================================================
def run_raw_vibration_ablation(split_index=1, skip_if_exists=True, val_split=0.2, seed=42):
    """Run ablation study using raw vibration data - FIXED for PhD"""
    print(f"\n{'=' * 80}")
    print("RUNNING RAW VIBRATION ABLATION STUDY FOR PHD")
    print("(Comparing raw vibration data vs. EWS features)")
    print(f"{'=' * 80}")
    
    # Set seed for reproducibility
    set_seed(seed)
    
    DATA_ROOT = "/kaggle/input/xj-dataset/XJTU-SY_Bearing_Datasets"
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    
    # Define splits - EXACT SAME as EWS experiments
    ALL_BEARINGS = [
        'Bearing1_1', 'Bearing1_2', 'Bearing1_3', 'Bearing1_4', 'Bearing1_5',
        'Bearing2_1', 'Bearing2_2', 'Bearing2_3', 'Bearing2_4', 'Bearing2_5',
        'Bearing3_1', 'Bearing3_2', 'Bearing3_3', 'Bearing3_4', 'Bearing3_5'
    ]
    
    # Check which bearings actually exist
    available_bearings = []
    for b in ALL_BEARINGS:
        if 'Bearing1_' in b:
            cond = '35Hz12kN'
        elif 'Bearing2_' in b:
            cond = '37.5Hz11kN'
        else:
            cond = '40Hz10kN'
        
        path = os.path.join(DATA_ROOT, cond, b)
        if os.path.exists(path):
            available_bearings.append(b)
    
    # Create splits
    bearing1_set = [b for b in available_bearings if 'Bearing1_' in b]
    bearing2_set = [b for b in available_bearings if 'Bearing2_' in b]
    bearing3_set = [b for b in available_bearings if 'Bearing3_' in b]
    
    split_options = [
        ("Cross-condition 1", bearing1_set + bearing2_set, bearing3_set),
        ("Cross-condition 2", bearing1_set + bearing3_set, bearing2_set),
        ("Cross-condition 3", bearing2_set + bearing3_set, bearing1_set),
    ]
    
    if split_index < 0 or split_index >= len(split_options):
        print(f"Invalid split index. Must be 0, 1, or 2")
        return None
    
    split_name, train_bearings, test_bearings = split_options[split_index]
    
    print(f"Split: {split_name}")
    print(f"Training bearings ({len(train_bearings)}): {train_bearings}")
    print(f"Testing bearings ({len(test_bearings)}): {test_bearings}")
    print(f"Device: {device}")
    print(f"Seed: {seed}")
    
    # Create results directory
    results_dir = "/kaggle/working/results"
    os.makedirs(results_dir, exist_ok=True)
    
    # Create unique experiment ID
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    experiment_id = f"raw_vibration_{split_name.replace(' ', '_')}_val{val_split}_seed{seed}_{timestamp}"
    
    results_file = os.path.join(results_dir, f"results_{experiment_id}.csv")
    metrics_file = os.path.join(results_dir, f"metrics_{experiment_id}.json")
    model_file = os.path.join(results_dir, f"model_{experiment_id}.pth")
    config_file = os.path.join(results_dir, f"config_{experiment_id}.txt")
    
    # Save configuration
    config = {
        'experiment_type': 'raw_vibration_ablation',
        'split_name': split_name,
        'train_bearings': train_bearings,
        'test_bearings': test_bearings,
        'window_size': 10,
        'step_size': 5,
        'max_samples': 4096,
        'downsample_factor': 8,
        'val_split': val_split,
        'seed': seed,
        'device': str(device),
        'timestamp': timestamp,
    }
    
    with open(config_file, 'w') as f:
        for key, value in config.items():
            f.write(f"{key}: {value}\n")
    
    # Check if results already exist
    if skip_if_exists and os.path.exists(results_file):
        print(f"✓ Raw vibration results already exist. Loading...")
        try:
            results_df = pd.read_csv(results_file)
            print(f"  Loaded {len(results_df)} results from previous run")
            return results_df
        except Exception as e:
            print(f"  Error loading existing results: {e}")
    
    # Create datasets
    print("\n📊 Creating raw vibration datasets...")
    
    raw_train_dataset = RawVibrationDataset(
        data_root=DATA_ROOT,
        bearing_list=train_bearings,
        window_size=10,
        step_size=5,
        max_files_per_bearing=None,
        use_both_channels=True,
        max_samples=4096,
        downsample_factor=8,
        enable_caching=True,
        cache_dir="/kaggle/working/cache"
    )
    
    raw_test_dataset = RawVibrationDataset(
        data_root=DATA_ROOT,
        bearing_list=test_bearings,
        window_size=10,
        step_size=5,
        max_files_per_bearing=None,
        use_both_channels=True,
        max_samples=4096,
        downsample_factor=8,
        enable_caching=True,
        cache_dir="/kaggle/working/cache"
    )
    
    if len(raw_train_dataset) == 0:
        print("❌ ERROR: Empty training dataset!")
        return None
    
    if len(raw_test_dataset) == 0:
        print("❌ ERROR: Empty test dataset!")
        return None
    
    print(f"✓ Training dataset: {len(raw_train_dataset)} sequences")
    print(f"✓ Test dataset: {len(raw_test_dataset)} sequences")
    
    # Create balanced train/val split
    print(f"\n🔀 Creating train/validation split ({val_split*100}% validation)...")
    
    bearing_to_indices = defaultdict(list)
    for idx in range(len(raw_train_dataset)):
        bearing_id = raw_train_dataset.bearing_ids[idx]
        bearing_to_indices[bearing_id].append(idx)
    
    train_indices = []
    val_indices = []
    
    # Use consistent random seed
    rng = np.random.RandomState(seed)
    
    for bearing_id, indices in bearing_to_indices.items():
        num_samples = len(indices)
        num_val = max(1, int(num_samples * val_split))
        
        shuffled_indices = indices.copy()
        rng.shuffle(shuffled_indices)
        
        val_indices.extend(shuffled_indices[:num_val])
        train_indices.extend(shuffled_indices[num_val:])
    
    # Shuffle final lists
    rng.shuffle(train_indices)
    rng.shuffle(val_indices)
    
    train_subset = Subset(raw_train_dataset, train_indices)
    val_subset = Subset(raw_train_dataset, val_indices)
    
    print(f"✓ Training split: {len(train_subset)} sequences")
    print(f"✓ Validation split: {len(val_subset)} sequences")
    
    # Get sequence shape
    seq_shape = raw_train_dataset.get_sequence_shape()
    print(f"✓ Input shape: {seq_shape}")
    print(f"  - Sequence length: {seq_shape[0]}")
    print(f"  - Channels: {seq_shape[1]}")
    print(f"  - Time steps: {seq_shape[2]}")
    
    # Create model
    print(f"\n🏗️ Creating 1D CNN model...")
    model = RawVibration_1DCNN(
        input_channels=seq_shape[1],
        sequence_length=seq_shape[0],
        time_steps=seq_shape[2]
    )
    
    print(f"✓ Model created: {model.__class__.__name__}")
    print(f"✓ Total parameters: {sum(p.numel() for p in model.parameters()):,}")
    
    # Create data loaders
    batch_size = 32
    train_loader = DataLoader(train_subset, batch_size=batch_size, shuffle=True, 
                             num_workers=0, pin_memory=True)
    val_loader = DataLoader(val_subset, batch_size=batch_size, shuffle=False,
                           num_workers=0, pin_memory=True)
    test_loader = DataLoader(raw_test_dataset, batch_size=batch_size, shuffle=False,
                            num_workers=0, pin_memory=True)
    
    # Train model
    print(f"\n🚀 Training raw vibration model...")
    trained_model, train_losses, val_losses = train_raw_vibration_model(
        model=model,
        train_loader=train_loader,
        val_loader=val_loader,
        epochs=100,
        lr=0.001,
        device=device,
        early_stopping_patience=20
    )
    
    # Test evaluation
    print(f"\n📈 Evaluating on test set...")
    trained_model.eval()
    preds, targets, bearing_ids_list, max_ruls_list = [], [], [], []
    
    with torch.no_grad():
        for data, targ, b_ids, m_ruls in test_loader:
            data = data.to(device)
            out = trained_model(data)
            
            for i in range(len(out)):
                pred_norm = out[i].item()
                targ_norm = targ[i].item()
                max_rul = m_ruls[i].item()
                
                # Convert to actual RUL
                pred_actual = pred_norm * max_rul
                targ_actual = targ_norm * max_rul
                
                preds.append(pred_actual)
                targets.append(targ_actual)
                bearing_ids_list.append(b_ids[i])
                max_ruls_list.append(max_rul)
    
    preds = np.array(preds)
    targets = np.array(targets)
    
    # Calculate metrics
    print(f"\n📊 Calculating metrics...")
    metrics = calculate_comprehensive_metrics(preds, targets)
    
    # Print results
    print(f"\n{'=' * 80}")
    print(f"🎯 RAW VIBRATION RESULTS FOR: {split_name}")
    print(f"{'=' * 80}")
    print(f"MAE:          {metrics['MAE']:8.2f} minutes")
    print(f"RMSE:         {metrics['RMSE']:8.2f} minutes")
    print(f"R² Score:     {metrics['R2']:8.4f}")
    print(f"Pearson r:    {metrics.get('Pearson_r', 0):8.4f}")
    
    if 'MAPE' in metrics and not np.isnan(metrics['MAPE']):
        print(f"MAPE:         {metrics['MAPE']:8.2f}%")
    
    if 'Within_10%' in metrics:
        print(f"Within 10%:   {metrics['Within_10%']:8.2f}%")
        print(f"Within 20%:   {metrics['Within_20%']:8.2f}%")
    
    print(f"Monotonicity: {metrics.get('Monotonicity_pred', 0):8.4f}")
    print(f"{'=' * 80}")
    
    # Save results
    print(f"\n💾 Saving results...")
    
    results_df = pd.DataFrame({
        'experiment_id': experiment_id,
        'bearing_id': bearing_ids_list,
        'max_rul': max_ruls_list,
        'actual_rul': targets,
        'predicted_rul': preds,
        'error': preds - targets,
        'abs_error': np.abs(preds - targets),
        'model_type': 'raw_vibration_1dcnn',
        'window_size': 10,
        'max_samples': 4096,
        'downsample_factor': 8,
        'val_split': val_split,
        'seed': seed,
    })
    
    results_df.to_csv(results_file, index=False)
    
    # Save metrics
    import json
    with open(metrics_file, 'w') as f:
        json.dump(metrics, f, indent=2)
    
    # Save model
    torch.save({
        'model_state_dict': trained_model.state_dict(),
        'model_type': 'raw_vibration_1dcnn',
        'sequence_shape': seq_shape,
        'config': config,
        'metrics': metrics,
        'train_losses': train_losses,
        'val_losses': val_losses,
    }, model_file)
    
    # Create comparison plot
    print(f"\n🎨 Creating results visualization...")
    fig, axes = plt.subplots(2, 3, figsize=(15, 10))
    
    # Training history
    axes[0, 0].plot(train_losses, label='Train', linewidth=2)
    axes[0, 0].plot(val_losses, label='Validation', linewidth=2)
    axes[0, 0].set_title('Training History')
    axes[0, 0].legend()
    axes[0, 0].grid(True, alpha=0.3)
    
    # Predictions vs Actual
    axes[0, 1].scatter(targets, preds, alpha=0.6, s=10)
    max_val = max(targets.max(), preds.max())
    axes[0, 1].plot([0, max_val], [0, max_val], 'r--', label='Perfect')
    axes[0, 1].set_xlabel('Actual RUL')
    axes[0, 1].set_ylabel('Predicted RUL')
    axes[0, 1].set_title(f'Predictions vs Actual (R²={metrics["R2"]:.3f})')
    axes[0, 1].legend()
    axes[0, 1].grid(True, alpha=0.3)
    
    # Error distribution
    errors = preds - targets
    axes[0, 2].hist(errors, bins=50, alpha=0.7, edgecolor='black', density=True)
    axes[0, 2].axvline(0, color='r', linestyle='--', label='Zero error')
    axes[0, 2].axvline(errors.mean(), color='g', linestyle='-', label=f'Mean: {errors.mean():.1f}')
    axes[0, 2].set_title(f'Error Distribution (MAE={metrics["MAE"]:.1f})')
    axes[0, 2].legend()
    axes[0, 2].grid(True, alpha=0.3)
    
    # MAE by bearing
    unique_bearings = sorted(set(bearing_ids_list))
    bearing_maes = []
    for b in unique_bearings:
        mask = np.array(bearing_ids_list) == b
        if np.sum(mask) > 0:
            mae = np.mean(np.abs(preds[mask] - targets[mask]))
            bearing_maes.append(mae)
    
    bars = axes[1, 0].bar(range(len(unique_bearings)), bearing_maes)
    axes[1, 0].set_xlabel('Bearing')
    axes[1, 0].set_ylabel('MAE (minutes)')
    axes[1, 0].set_title('MAE by Bearing')
    axes[1, 0].set_xticks(range(len(unique_bearings)))
    axes[1, 0].set_xticklabels(unique_bearings, rotation=45, ha='right')
    axes[1, 0].grid(True, alpha=0.3, axis='y')
    
    # Color by bearing group
    for idx, b in enumerate(unique_bearings):
        if 'Bearing1' in b:
            bars[idx].set_color('blue')
        elif 'Bearing2' in b:
            bars[idx].set_color('green')
        else:
            bars[idx].set_color('orange')
    
    # Residual plot
    axes[1, 1].scatter(targets, errors, alpha=0.6, s=10)
    axes[1, 1].axhline(0, color='r', linestyle='--')
    axes[1, 1].set_xlabel('Actual RUL')
    axes[1, 1].set_ylabel('Residual (Predicted - Actual)')
    axes[1, 1].set_title('Residual Plot')
    axes[1, 1].grid(True, alpha=0.3)
    
    # Text summary
    axes[1, 2].axis('off')
    summary_text = (
        f"RAW VIBRATION ABLATION\n"
        f"{'='*40}\n"
        f"Split: {split_name}\n"
        f"Model: 1D CNN\n"
        f"Input shape: {seq_shape}\n"
        f"Train samples: {len(train_subset)}\n"
        f"Test samples: {len(raw_test_dataset)}\n"
        f"{'='*40}\n"
        f"MAE: {metrics['MAE']:.1f} min\n"
        f"RMSE: {metrics['RMSE']:.1f} min\n"
        f"R²: {metrics['R2']:.4f}\n"
        f"Within 10%: {metrics.get('Within_10%', 0):.1f}%\n"
        f"Experiment ID: {experiment_id}"
    )
    axes[1, 2].text(0.1, 0.5, summary_text, fontsize=10, fontfamily='monospace',
                   verticalalignment='center')
    
    plt.suptitle(f'Raw Vibration Ablation Study - {split_name}', fontsize=14, fontweight='bold')
    plt.tight_layout()
    
    plot_file = os.path.join(results_dir, f"plot_{experiment_id}.png")
    plt.savefig(plot_file, dpi=150, bbox_inches='tight')
    plt.close()
    
    print(f"\n✅ Raw vibration ablation completed!")
    print(f"📁 Results saved:")
    print(f"   📄 Results: {os.path.basename(results_file)}")
    print(f"   📊 Metrics: {os.path.basename(metrics_file)}")
    print(f"   🖼️  Plot: {os.path.basename(plot_file)}")
    print(f"   🤖 Model: {os.path.basename(model_file)}")
    
    return results_df

# ============================================================================
# IMPROVED COMPARISON FUNCTION
# ============================================================================
def compare_raw_vs_ews(split_name="Cross-condition 2"):
    """Compare raw vibration results with EWS features results - FIXED"""
    print(f"\n{'=' * 80}")
    print(f"PHD COMPARISON: RAW VIBRATION vs EWS FEATURES")
    print(f"{'=' * 80}")
    
    results_dir = "/kaggle/working/results"
    
    # Find the most recent raw vibration results
    raw_files = [f for f in os.listdir(results_dir) 
                if f.startswith('results_raw_vibration') and split_name.replace(' ', '_') in f]
    
    if not raw_files:
        print(f"❌ No raw vibration results found for {split_name}")
        print("Run the raw vibration ablation first.")
        return None
    
    # Use the most recent raw file
    raw_files.sort(reverse=True)
    raw_path = os.path.join(results_dir, raw_files[0])
    print(f"Using raw vibration results: {raw_files[0]}")
    
    raw_df = pd.read_csv(raw_path)
    
    # Find corresponding EWS results (simple_lstm is usually best)
    ews_files = [f for f in os.listdir(results_dir) 
                if f.startswith('results_') and split_name.replace(' ', '_') in f 
                and 'simple_lstm' in f and 'raw_vibration' not in f]
    
    if not ews_files:
        # Try to find any EWS results
        ews_files = [f for f in os.listdir(results_dir) 
                    if f.startswith('results_') and split_name.replace(' ', '_') in f 
                    and 'raw_vibration' not in f]
    
    if not ews_files:
        print(f"❌ No EWS results found for {split_name}")
        print("Run an EWS experiment first (option 14).")
        return None
    
    # Use the most recent EWS file
    ews_files.sort(reverse=True)
    ews_path = os.path.join(results_dir, ews_files[0])
    print(f"Using EWS results: {ews_files[0]}")
    
    ews_df = pd.read_csv(ews_path)
    
    # Calculate metrics
    raw_metrics = calculate_comprehensive_metrics(raw_df['predicted_rul'].values,
                                                 raw_df['actual_rul'].values)
    ews_metrics = calculate_comprehensive_metrics(ews_df['predicted_rul'].values,
                                                 ews_df['actual_rul'].values)
    
    # Print comparison
    print(f"\n{'Metric':<15} {'Raw Vibration':<15} {'EWS Features':<15} {'Improvement':<15} {'% Change':<15}")
    print("-" * 75)
    
    metrics_to_compare = ['MAE', 'RMSE', 'R2', 'Within_10%', 'Pearson_r']
    for metric in metrics_to_compare:
        if metric in raw_metrics and metric in ews_metrics:
            raw_val = raw_metrics[metric]
            ews_val = ews_metrics[metric]
            
            if metric in ['MAE', 'RMSE']:
                # Lower is better
                diff = raw_val - ews_val
                pct_change = (diff / raw_val * 100) if raw_val != 0 else 0
                improvement = f"{pct_change:+.1f}%" if diff > 0 else f"{pct_change:.1f}%"
                print(f"{metric:<15} {raw_val:<15.3f} {ews_val:<15.3f} {diff:<15.3f} {improvement:<15}")
            else:
                # Higher is better
                diff = ews_val - raw_val
                pct_change = (diff / abs(raw_val) * 100) if raw_val != 0 else 0
                improvement = f"{pct_change:+.1f}%" if diff > 0 else f"{pct_change:.1f}%"
                print(f"{metric:<15} {raw_val:<15.4f} {ews_val:<15.4f} {diff:<15.4f} {improvement:<15}")
    
    # Statistical significance test
    print(f"\n📊 Statistical Significance Test:")
    
    from scipy import stats
    # Remove NaNs
    raw_errors = np.abs(raw_df['error'].values)
    ews_errors = np.abs(ews_df['error'].values)
    
    # Ensure same length (take minimum)
    min_len = min(len(raw_errors), len(ews_errors))
    raw_errors = raw_errors[:min_len]
    ews_errors = ews_errors[:min_len]
    
    # Paired t-test
    t_stat, p_value = stats.ttest_rel(raw_errors, ews_errors)
    
    print(f"   Paired t-test:")
    print(f"   t-statistic = {t_stat:.3f}")
    print(f"   p-value = {p_value:.4f}")
    
    if p_value < 0.05:
        print(f"   ✅ Statistically significant (p < 0.05)")
        if np.mean(ews_errors) < np.mean(raw_errors):
            print(f"   → EWS features perform SIGNIFICANTLY better")
        else:
            print(f"   → Raw vibration performs SIGNIFICANTLY better")
    else:
        print(f"   ⚠️ Not statistically significant (p ≥ 0.05)")
        print(f"   → No significant difference between methods")
    
    # Create comparison plot
    fig, axes = plt.subplots(2, 3, figsize=(15, 10))
    
    # 1. MAE comparison
    models = ['Raw Vibration', 'EWS Features']
    mae_values = [raw_metrics['MAE'], ews_metrics['MAE']]
    colors = ['red', 'green']
    axes[0, 0].bar(models, mae_values, color=colors, alpha=0.7)
    axes[0, 0].set_title('MAE Comparison\n(lower is better)', fontweight='bold')
    axes[0, 0].set_ylabel('MAE (minutes)')
    axes[0, 0].grid(True, alpha=0.3, axis='y')
    
    # Add value labels
    for i, v in enumerate(mae_values):
        axes[0, 0].text(i, v + max(mae_values)*0.02, f'{v:.1f}', 
                       ha='center', va='bottom', fontweight='bold')
    
    # 2. R² comparison
    r2_values = [raw_metrics['R2'], ews_metrics['R2']]
    axes[0, 1].bar(models, r2_values, color=colors, alpha=0.7)
    axes[0, 1].set_title('R² Comparison\n(higher is better)', fontweight='bold')
    axes[0, 1].set_ylabel('R² Score')
    axes[0, 1].grid(True, alpha=0.3, axis='y')
    
    for i, v in enumerate(r2_values):
        axes[0, 1].text(i, v + max(r2_values)*0.02, f'{v:.3f}', 
                       ha='center', va='bottom', fontweight='bold')
    
    # 3. Within 10% comparison
    if 'Within_10%' in raw_metrics and 'Within_10%' in ews_metrics:
        within_10 = [raw_metrics['Within_10%'], ews_metrics['Within_10%']]
        axes[0, 2].bar(models, within_10, color=colors, alpha=0.7)
        axes[0, 2].set_title('Within 10% Accuracy\n(higher is better)', fontweight='bold')
        axes[0, 2].set_ylabel('Percentage (%)')
        axes[0, 2].grid(True, alpha=0.3, axis='y')
        
        for i, v in enumerate(within_10):
            axes[0, 2].text(i, v + max(within_10)*0.02, f'{v:.1f}%', 
                           ha='center', va='bottom', fontweight='bold')
    
    # 4. Error distribution
    axes[1, 0].hist(raw_errors, bins=30, alpha=0.5, label='Raw Vibration', 
                   color='red', density=True)
    axes[1, 0].hist(ews_errors, bins=30, alpha=0.5, label='EWS Features', 
                   color='green', density=True)
    axes[1, 0].axvline(np.mean(raw_errors), color='red', linestyle='--', 
                      label=f'Raw mean: {np.mean(raw_errors):.1f}')
    axes[1, 0].axvline(np.mean(ews_errors), color='green', linestyle='--', 
                      label=f'EWS mean: {np.mean(ews_errors):.1f}')
    axes[1, 0].set_xlabel('Absolute Error (minutes)')
    axes[1, 0].set_ylabel('Density')
    axes[1, 0].set_title('Error Distribution Comparison')
    axes[1, 0].legend()
    axes[1, 0].grid(True, alpha=0.3)
    
    # 5. Scatter plot comparison
    axes[1, 1].scatter(raw_df['actual_rul'], raw_df['predicted_rul'], 
                      alpha=0.5, s=10, label='Raw Vibration', color='red')
    axes[1, 1].scatter(ews_df['actual_rul'], ews_df['predicted_rul'], 
                      alpha=0.5, s=10, label='EWS Features', color='green')
    max_val = max(raw_df['actual_rul'].max(), ews_df['actual_rul'].max(),
                 raw_df['predicted_rul'].max(), ews_df['predicted_rul'].max())
    axes[1, 1].plot([0, max_val], [0, max_val], 'k--', label='Perfect', linewidth=2)
    axes[1, 1].set_xlabel('Actual RUL')
    axes[1, 1].set_ylabel('Predicted RUL')
    axes[1, 1].set_title('Predictions Comparison')
    axes[1, 1].legend()
    axes[1, 1].grid(True, alpha=0.3)
    
    # 6. Summary statistics
    axes[1, 2].axis('off')
    
    summary_text = (
        f"PHD ABLATION STUDY SUMMARY\n"
        f"{'='*40}\n"
        f"Split: {split_name}\n"
        f"Test samples: {len(raw_df)}\n"
        f"Statistical test: Paired t-test\n"
        f"t-statistic: {t_stat:.3f}\n"
        f"p-value: {p_value:.4f}\n"
        f"{'='*40}\n"
        f"RAW VIBRATION:\n"
        f"  MAE: {raw_metrics['MAE']:.1f} min\n"
        f"  R²: {raw_metrics['R2']:.4f}\n"
        f"  Within 10%: {raw_metrics.get('Within_10%', 0):.1f}%\n"
        f"{'='*40}\n"
        f"EWS FEATURES:\n"
        f"  MAE: {ews_metrics['MAE']:.1f} min\n"
        f"  R²: {ews_metrics['R2']:.4f}\n"
        f"  Within 10%: {ews_metrics.get('Within_10%', 0):.1f}%\n"
        f"{'='*40}\n"
        f"Conclusion: {'EWS SIGNIFICANTLY BETTER' if p_value < 0.05 and ews_metrics['MAE'] < raw_metrics['MAE'] else 'No significant difference'}"
    )
    
    axes[1, 2].text(0.1, 0.5, summary_text, fontsize=9, fontfamily='monospace',
                   verticalalignment='center')
    
    plt.suptitle(f'PhD Ablation Study: Raw Vibration vs EWS Features\n{split_name}', 
                fontsize=16, fontweight='bold')
    plt.tight_layout()
    
    comparison_file = os.path.join(results_dir, f"phd_comparison_{split_name.replace(' ', '_')}.png")
    plt.savefig(comparison_file, dpi=150, bbox_inches='tight')
    plt.show()
    
    print(f"\n✅ Comparison saved to: {comparison_file}")
    
    return raw_metrics, ews_metrics, p_value
# ============================================================================
# HYBRID APPROACH: WARM-START FINE-TUNING - FIXED FOR PHD
# ============================================================================

def create_hybrid_split(bearing_list, test_condition, early_data_ratio=0.2):
    """
    Create hybrid split for warm-start fine-tuning - FIXED
    
    Args:
        test_condition: 1 (35Hz12kN), 2 (37.5Hz11kN), or 3 (40Hz10kN)
        early_data_ratio: Ratio of early healthy data (e.g., 0.2 = 20%)
    """
    # Group bearings by condition
    condition1 = [b for b in bearing_list if 'Bearing1_' in b]
    condition2 = [b for b in bearing_list if 'Bearing2_' in b]
    condition3 = [b for b in bearing_list if 'Bearing3_' in b]
    
    if test_condition == 1:
        test_bearings = condition1
        train_bearings = condition2 + condition3
    elif test_condition == 2:
        test_bearings = condition2
        train_bearings = condition1 + condition3
    else:  # test_condition == 3
        test_bearings = condition3
        train_bearings = condition1 + condition2
    
    # Warm-start uses same test bearings but only early healthy files
    warm_start_bearings = test_bearings.copy()
    
    logger.info(f"Hybrid Split Configuration:")
    logger.info(f"  Test Condition: {test_condition}")
    logger.info(f"  Training Bearings: {len(train_bearings)} bearings")
    logger.info(f"  Warm-start Bearings: {len(warm_start_bearings)} bearings")
    logger.info(f"  Test Bearings: {len(test_bearings)} bearings")
    logger.info(f"  Early Data Ratio: {early_data_ratio*100}%")
    
    return train_bearings, warm_start_bearings, test_bearings

class EarlyDataDataset(Dataset):
    """
    Dataset that uses only early HEALTHY files from bearings
    with CONSISTENT RUL calculation - FIXED
    """
    def __init__(self, data_root, bearing_list, early_ratio=0.2, 
                 window_size=10, step_size=5, ews_extractor=None,
                 use_both_channels=True, fusion_method='concat'):
        
        self.data_root = data_root
        self.bearing_list = bearing_list
        self.early_ratio = early_ratio  # Ratio of early files to use
        self.window_size = window_size
        self.step_size = step_size
        self.use_both_channels = use_both_channels
        self.fusion_method = fusion_method
        
        self.ews_extractor = ews_extractor or EnhancedEarlyWarningSignals(
            window_size=1024, step_size=1024
        )
        
        # Use EXACT SAME manual settings as main dataset
        self.manual_settings = {
            'Bearing1_1': {'multiplier': 3.5, 'manual_start': 75, 'condition': '35Hz12kN'},#80
            'Bearing1_2': {'multiplier': 3.5, 'manual_start': 54, 'condition': '35Hz12kN'},#100
            'Bearing1_3': {'multiplier': 5.0, 'manual_start': 59, 'condition': '35Hz12kN'},#58
            'Bearing1_4': {'multiplier': 2.5, 'manual_start': 75, 'condition': '35Hz12kN'},#70
            'Bearing1_5': {'multiplier': 2.5, 'manual_start': 32, 'condition': '35Hz12kN'},#25
            'Bearing2_1': {'multiplier': 3.0, 'manual_start': 453, 'condition': '37.5Hz11kN'},#300
            'Bearing2_2': {'multiplier': 3.0, 'manual_start': 47, 'condition': '37.5Hz11kN'},#100
            'Bearing2_3': {'multiplier': 3.0, 'manual_start': 59, 'condition': '37.5Hz11kN'},#150
           
            'Bearing3_2': {'multiplier': 3.0, 'manual_start': 466, 'condition': '40Hz10kN'},#700
            'Bearing3_3': {'multiplier': 3.0, 'manual_start': 342, 'condition': '40Hz10kN'},#250
            'Bearing3_4': {'multiplier': 3.0, 'manual_start': 45, 'condition': '40Hz10kN'},#80
            
        }
        
        self.ews_sequences = []
        self.rul_targets = []
        self.bearing_ids = []
        self.max_ruls = []
        self.degradation_starts = {}
        
        self._process_bearings()
        
        logger.info(f"✓ EarlyDataDataset: {len(self.ews_sequences)} sequences from early {early_ratio*100}% of data")
    
    def _get_bearing_path(self, bearing_name):
        """Get full path - CONSISTENT with main dataset"""
        if 'Bearing1_' in bearing_name:
            cond = '35Hz12kN'
        elif 'Bearing2_' in bearing_name:
            cond = '37.5Hz11kN'
        else:
            cond = '40Hz10kN'
        return os.path.join(self.data_root, cond, bearing_name)
    
    def _read_vibration_signal(self, filepath):
        """Read signal - CONSISTENT with main dataset"""
        try:
            # CRITICAL FIX: XJTU files have NO headers
            df = pd.read_csv(filepath)
            
            if df.shape[1] >= 2:
                h = df.iloc[:, 0].values.astype(np.float32)
                v = df.iloc[:, 1].values.astype(np.float32)
            else:
                h = df.iloc[:, 0].values.astype(np.float32)
                v = h.copy()
            
            signal = np.column_stack([h, v])
            
            # Use same length as main dataset
            if signal.shape[0] > 32768:
                signal = signal[:32768]
            
            return np.nan_to_num(signal, nan=0.0, posinf=0.0, neginf=0.0)
        except Exception as e:
            logger.error(f"Error reading {filepath}: {e}")
            return None
    
    def _calculate_early_rul(self, csv_files, bearing_name, early_files_count):
        """
        Calculate RUL for early data - CONSISTENT with main dataset
        
        For early healthy data, RUL should be close to 1.0 (high)
        but decreasing slightly as we approach degradation
        """
        total_files = len(csv_files)
        early_files = csv_files[:early_files_count]
        
        # Calculate amplitude for each early file
        amplitudes = []
        for path in early_files:
            try:
                # Read raw signal for amplitude calculation
                df = pd.read_csv(path)
                if df.shape[1] >= 2:
                    sig = df.iloc[:, 0].values.astype(np.float32)
                else:
                    sig = df.iloc[:, 0].values.astype(np.float32)
                amp = np.max(np.abs(sig[:1000]))  # First 1000 samples
                amplitudes.append(amp)
            except:
                amplitudes.append(0.0)
        
        amplitudes = np.array(amplitudes)
        
        # Get settings for this bearing
        settings = self.manual_settings.get(bearing_name, 
                                          {'multiplier': 3.0, 'manual_start': int(total_files * 0.7)})
        multiplier = settings['multiplier']
        degradation_start = settings['manual_start']
        
        # Calculate healthy baseline from first 20% of early files
        baseline_samples = min(20, early_files_count // 2)
        if baseline_samples > 0:
            healthy_vals = amplitudes[:baseline_samples]
            healthy_baseline = np.median(healthy_vals)
        else:
            healthy_baseline = 1.0
        
        degradation_threshold = healthy_baseline * multiplier
        
        # For early data, we're in healthy phase
        # RUL = max_rul (constant) during healthy phase
        max_rul = total_files - degradation_start-1
        
        # Ensure max_rul is reasonable
        if max_rul < 20:
            logger.warning(f"Adjusting max_rul for {bearing_name} from {max_rul} to 30")
            max_rul = 30
        
        return degradation_start, max_rul
    
    def _process_bearings(self):
        """Process early files with CONSISTENT RUL calculation"""
        for bearing_name in self.bearing_list:
            path = self._get_bearing_path(bearing_name)
            if not os.path.exists(path):
                logger.warning(f"Skipping {bearing_name} - path not found")
                continue
            
            try:
                files = sorted([f for f in os.listdir(path) if f.endswith('.csv')],
                             key=lambda x: int(x.split('.')[0]))
                csv_files = [os.path.join(path, f) for f in files]
            except Exception as e:
                logger.error(f"Error listing files for {bearing_name}: {e}")
                continue
            
            # Only take early files (first X%)
            early_count = max(self.window_size, int(len(csv_files) * self.early_ratio))
            early_files = csv_files[:early_count]
            
            if len(early_files) < self.window_size:
                logger.warning(f"Skipping {bearing_name} - too few early files")
                continue
            
            logger.info(f"Processing early data for {bearing_name}: {len(early_files)}/{len(csv_files)} files")
            
            # Calculate RUL parameters CONSISTENTLY
            degradation_start, max_rul = self._calculate_early_rul(
                csv_files, bearing_name, len(early_files))
            
            self.degradation_starts[bearing_name] = degradation_start
            
            # Create sequences from early files
            sequences_added = 0
            for start_idx in range(0, len(early_files) - self.window_size + 1, self.step_size):
                end_idx = start_idx + self.window_size
                ews_seq = []
                valid = True
                
                for i in range(start_idx, end_idx):
                    sig = self._read_vibration_signal(early_files[i])
                    if sig is None:
                        valid = False
                        break
                    
                    feats = self.ews_extractor.compute_ews_for_both_channels(
                        sig, fusion_method=self.fusion_method, include_wavelet=True)
                    
                    if np.any(np.isnan(feats)):
                        valid = False
                        break
                    
                    ews_seq.append(feats)
                
                if not valid or len(ews_seq) != self.window_size:
                    continue
                
                ews_seq = np.stack(ews_seq)
                
                # CRITICAL: Calculate RUL CONSISTENTLY with main dataset
                last_idx_in_full = end_idx - 1  # Index in early_files
                
                # Convert to index in full dataset for RUL calculation
                # Since we're using early files, last_idx should be < degradation_start
                if last_idx_in_full < degradation_start:
                    # Healthy phase: constant RUL
                    rul_minutes = max_rul
                else:
                    # Shouldn't happen if early_ratio is small enough
                    rul_minutes = len(csv_files) - last_idx_in_full - 1
                
                rul_norm = rul_minutes / max_rul if max_rul > 0 else 0.0
                
                # For early data, RUL should be high (> 0.8)
                # This ensures model learns healthy patterns
                if rul_norm < 0.8:
                    logger.debug(f"Low RUL for early data: {rul_norm:.3f}, adjusting to 0.9")
                    rul_norm = 0.9
                
                self.ews_sequences.append(ews_seq)
                self.rul_targets.append(rul_norm)
                self.bearing_ids.append(bearing_name)
                self.max_ruls.append(max_rul)
                sequences_added += 1
            
            logger.info(f"  Added {sequences_added} early sequences")
    
    def get_feature_dimension(self):
        if len(self.ews_sequences) == 0:
            base_features = 44  # With wavelet
            if self.use_both_channels and self.fusion_method == 'concat':
                return base_features * 2
            return base_features
        return self.ews_sequences[0].shape[1]
    
    def __len__(self):
        return len(self.ews_sequences)
    
    def __getitem__(self, idx):
        return (torch.FloatTensor(self.ews_sequences[idx]),
                torch.FloatTensor([self.rul_targets[idx]]),
                self.bearing_ids[idx],
                self.max_ruls[idx])

def run_hybrid_approach(test_condition=2, early_ratio=0.2, 
                        model_type='simple_lstm', seed=42):
    """
    Run hybrid approach with CONSISTENT RUL calculation - FIXED
    
    Phase 1: Pre-train on conditions A+B
    Phase 2: Fine-tune on early healthy data from condition C
    Phase 3: Test on full condition C
    """
    print(f"\n{'=' * 80}")
    print(f"HYBRID APPROACH: Warm-Start Fine-Tuning - PHD VERSION")
    print(f"Testing on Condition {test_condition}")
    print(f"Using {early_ratio*100}% early healthy data for fine-tuning")
    print(f"Model: {model_type}, Seed: {seed}")
    print(f"{'=' * 80}")
    
    # Set seed for reproducibility
    set_seed(seed)
    
    DATA_ROOT = "/kaggle/input/xj-dataset/XJTU-SY_Bearing_Datasets"
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    
    # All bearings
    ALL_BEARINGS = [
        'Bearing1_1', 'Bearing1_2', 'Bearing1_3', 'Bearing1_4', 'Bearing1_5',
        'Bearing2_1', 'Bearing2_2', 'Bearing2_3', 'Bearing2_4', 'Bearing2_5',
        'Bearing3_1', 'Bearing3_2', 'Bearing3_3', 'Bearing3_4', 'Bearing3_5'
    ]
    
    # Check which bearings actually exist
    available_bearings = []
    for b in ALL_BEARINGS:
        if 'Bearing1_' in b:
            cond = '35Hz12kN'
        elif 'Bearing2_' in b:
            cond = '37.5Hz11kN'
        else:
            cond = '40Hz10kN'
        
        path = os.path.join(DATA_ROOT, cond, b)
        if os.path.exists(path):
            available_bearings.append(b)
    
    # Create hybrid split
    train_bearings, warm_start_bearings, test_bearings = create_hybrid_split(
        available_bearings, test_condition, early_ratio
    )
    
    print(f"✓ Training bearings: {len(train_bearings)}")
    print(f"✓ Warm-start bearings: {len(warm_start_bearings)}")
    print(f"✓ Test bearings: {len(test_bearings)}")
    
    # Create unique experiment ID
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    experiment_id = f"hybrid_cond{test_condition}_early{early_ratio}_{model_type}_seed{seed}_{timestamp}"
    
    # Phase 1: Pre-training on Conditions A+B
    print(f"\n{'=' * 40}")
    print("PHASE 1: Pre-training on Conditions A+B")
    print(f"{'=' * 40}")
    
    train_dataset = EnhancedXJTU_EWS_Dataset(
        data_root=DATA_ROOT,
        bearing_list=train_bearings,
        window_size=10,
        step_size=5,
        max_files_per_bearing=None,
        use_both_channels=True,
        fusion_method='concat',
        enable_caching=True,
        cache_dir="/kaggle/working/cache",
        include_wavelet=True
    )
    
    if len(train_dataset) == 0:
        logger.error("❌ Empty training dataset")
        return None
    
    print(f"✓ Pre-training dataset: {len(train_dataset)} sequences")
    
    # Create balanced train/val split
    train_indices, val_indices = create_balanced_train_val_split(
        train_dataset, val_split=0.2, random_seed=seed
    )
    
    train_subset = Subset(train_dataset, train_indices)
    val_subset = Subset(train_dataset, val_indices)
    
    print(f"✓ Training split: {len(train_subset)} sequences")
    print(f"✓ Validation split: {len(val_subset)} sequences")
    
    feature_dim = train_dataset.get_feature_dimension()
    print(f"✓ Feature dimension: {feature_dim}")
    
    # Create model
    if model_type == 'simple_lstm':
        model = Simple_LSTM(
            input_size=feature_dim,
            hidden_size=128,
            num_layers=2,
            dropout=0.3
        )
    elif model_type == 'lstm_da':
        model = EWS_LSTM_Attention_DA(
            input_size=feature_dim,
            hidden_size=128,
            num_layers=2,
            dropout=0.3,
            use_da=True
        )
    elif model_type == 'bilstm_attn':
        model = EWS_LSTM_Attention(
            input_size=feature_dim,
            hidden_size=128,
            num_layers=2,
            dropout=0.3
        )
    else:
        logger.warning(f"Unknown model type {model_type}, using simple_lstm")
        model = Simple_LSTM(input_size=feature_dim, hidden_size=128, num_layers=2)
    
    print(f"✓ Model: {model.__class__.__name__}")
    print(f"✓ Parameters: {sum(p.numel() for p in model.parameters()):,}")
    
    # Data loaders for pre-training
    batch_size = 32
    train_loader = DataLoader(train_subset, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_subset, batch_size=batch_size, shuffle=False)
    
    # Pre-train with early stopping
    model.to(device)
    optimizer = optim.AdamW(model.parameters(), lr=0.001, weight_decay=1e-4)
    criterion = nn.MSELoss()
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.5, patience=10
    )
    
    best_val_loss = float('inf')
    patience_counter = 0
    pretrain_losses = []
    
    print(f"\nPre-training for up to 50 epochs...")
    
    for epoch in range(50):
        model.train()
        train_loss = 0.0
        
        for data, targets, _, _ in train_loader:
            data, targets = data.to(device), targets.to(device)
            
            optimizer.zero_grad()
            predictions = model(data)
            loss = criterion(predictions, targets)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            
            train_loss += loss.item()
        
        # Validation
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for data, targets, _, _ in val_loader:
                data, targets = data.to(device), targets.to(device)
                predictions = model(data)
                val_loss += criterion(predictions, targets).item()
        
        avg_val_loss = val_loss / len(val_loader)
        pretrain_losses.append(avg_val_loss)
        
        scheduler.step(avg_val_loss)
        
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            best_pretrained_state = model.state_dict().copy()
            patience_counter = 0
        else:
            patience_counter += 1
        
        if (epoch + 1) % 10 == 0:
            print(f"  Epoch {epoch+1}/50 | Val loss: {avg_val_loss:.6f}")
        
        if patience_counter >= 15:
            print(f"Early stopping at epoch {epoch+1}")
            break
    
    model.load_state_dict(best_pretrained_state)
    print(f"✓ Pre-training completed. Best val loss: {best_val_loss:.6f}")
    
    # Phase 2: Fine-tuning on early healthy data from Condition C
    print(f"\n{'=' * 40}")
    print(f"PHASE 2: Fine-tuning on early Condition {test_condition} data")
    print(f"{'=' * 40}")
    
    warm_start_dataset = EarlyDataDataset(
        data_root=DATA_ROOT,
        bearing_list=warm_start_bearings,
        early_ratio=early_ratio,
        window_size=10,
        step_size=5,
        use_both_channels=True,
        fusion_method='concat'
    )
    
    if len(warm_start_dataset) == 0:
        logger.error("❌ No warm-start data available")
        return None
    
    print(f"✓ Warm-start dataset: {len(warm_start_dataset)} sequences")
    print(f"✓ Average RUL in warm-start: {np.mean(warm_start_dataset.rul_targets):.3f}")
    
    # Create fine-tuning loader
    fine_tune_loader = DataLoader(warm_start_dataset, batch_size=16, shuffle=True)
    
    # Reduce learning rate for fine-tuning
    for param_group in optimizer.param_groups:
        param_group['lr'] = 0.0001  # 10x smaller
    
    print(f"Fine-tuning for 10 epochs with LR={optimizer.param_groups[0]['lr']:.6f}...")
    
    fine_tune_losses = []
    for epoch in range(10):
        model.train()
        ft_loss = 0.0
        
        for data, targets, _, _ in fine_tune_loader:
            data, targets = data.to(device), targets.to(device)
            
            optimizer.zero_grad()
            predictions = model(data)
            loss = criterion(predictions, targets)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=0.5)
            optimizer.step()
            
            ft_loss += loss.item()
        
        avg_ft_loss = ft_loss / len(fine_tune_loader)
        fine_tune_losses.append(avg_ft_loss)
        
        if (epoch + 1) % 2 == 0:
            print(f"  Fine-tuning Epoch {epoch+1}/10 | Loss: {avg_ft_loss:.6f}")
    
    print("✓ Fine-tuning completed")
    
    # Phase 3: Evaluation on full Condition C test data
    print(f"\n{'=' * 40}")
    print(f"PHASE 3: Testing on full Condition {test_condition} data")
    print(f"{'=' * 40}")
    
    test_dataset = EnhancedXJTU_EWS_Dataset(
        data_root=DATA_ROOT,
        bearing_list=test_bearings,
        window_size=10,
        step_size=5,
        max_files_per_bearing=None,
        use_both_channels=True,
        fusion_method='concat',
        enable_caching=True,
        cache_dir="/kaggle/working/cache",
        include_wavelet=True
    )
    
    if len(test_dataset) == 0:
        logger.error("❌ Empty test dataset")
        return None
    
    print(f"✓ Test dataset: {len(test_dataset)} sequences")
    
    test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)
    
    # Evaluate
    model.eval()
    preds, targets, bearing_ids_list, max_ruls_list = [], [], [], []
    
    with torch.no_grad():
        for data, targ, b_ids, m_ruls in test_loader:
            data = data.to(device)
            out = model(data)
            
            for i in range(len(out)):
                pred_norm = out[i].item()
                targ_norm = targ[i].item()
                max_rul = m_ruls[i].item()
                
                pred_actual = pred_norm * max_rul
                targ_actual = targ_norm * max_rul
                
                preds.append(pred_actual)
                targets.append(targ_actual)
                bearing_ids_list.append(b_ids[i])
                max_ruls_list.append(max_rul)
    
    preds = np.array(preds)
    targets = np.array(targets)
    
    # Calculate metrics
    metrics = calculate_comprehensive_metrics(preds, targets)
    
    # Print results
    print(f"\n{'=' * 80}")
    print(f"🎯 HYBRID APPROACH RESULTS (Condition {test_condition}):")
    print(f"{'=' * 80}")
    print(f"MAE:          {metrics['MAE']:8.2f} minutes")
    print(f"RMSE:         {metrics['RMSE']:8.2f} minutes")
    print(f"R² Score:     {metrics['R2']:8.4f}")
    print(f"Within 10%:   {metrics.get('Within_10%', 0):8.2f}%")
    print(f"Within 20%:   {metrics.get('Within_20%', 0):8.2f}%")
    print(f"{'=' * 80}")
    
    # Save results
    results_dir = "/kaggle/working/results"
    os.makedirs(results_dir, exist_ok=True)
    
    results_file = os.path.join(results_dir, f"results_{experiment_id}.csv")
    metrics_file = os.path.join(results_dir, f"metrics_{experiment_id}.json")
    model_file = os.path.join(results_dir, f"model_{experiment_id}.pth")
    
    results_df = pd.DataFrame({
        'experiment_id': experiment_id,
        'bearing_id': bearing_ids_list,
        'max_rul': max_ruls_list,
        'actual_rul': targets,
        'predicted_rul': preds,
        'error': preds - targets,
        'abs_error': np.abs(preds - targets),
        'approach': 'hybrid_warm_start',
        'test_condition': test_condition,
        'early_ratio': early_ratio,
        'model_type': model_type,
        'seed': seed
    })
    
    results_df.to_csv(results_file, index=False)
    
    # Save metrics
    import json
    with open(metrics_file, 'w') as f:
        json.dump(metrics, f, indent=2)
    
    # Save model
    torch.save({
        'model_state_dict': model.state_dict(),
        'model_type': model_type,
        'feature_dim': feature_dim,
        'pretrain_losses': pretrain_losses,
        'fine_tune_losses': fine_tune_losses,
        'metrics': metrics,
        'config': {
            'test_condition': test_condition,
            'early_ratio': early_ratio,
            'seed': seed,
            'train_bearings': train_bearings,
            'test_bearings': test_bearings,
        }
    }, model_file)
    
    print(f"\n✅ Hybrid approach completed!")
    print(f"📁 Results saved:")
    print(f"   📄 Results: {os.path.basename(results_file)}")
    print(f"   📊 Metrics: {os.path.basename(metrics_file)}")
    print(f"   🤖 Model: {os.path.basename(model_file)}")
    
    return {
        'experiment_id': experiment_id,
        'approach': 'hybrid',
        'test_condition': test_condition,
        'early_ratio': early_ratio,
        'model_type': model_type,
        'metrics': metrics,
        'results_file': results_file,
        'model_file': model_file,
    }

def compare_hybrid_vs_standard(test_condition=2):
    """
    Compare hybrid vs standard approach with STATISTICAL SIGNIFICANCE
    """
    print(f"\n{'=' * 80}")
    print(f"PHD COMPARISON: Hybrid vs Standard Approach")
    print(f"Test Condition: {test_condition}")
    print(f"{'=' * 80}")
    
    results_dir = "/kaggle/working/results"
    
    # Find latest hybrid results
    hybrid_files = [f for f in os.listdir(results_dir) 
                   if f.startswith('results_hybrid_cond') and str(test_condition) in f]
    
    if not hybrid_files:
        print(f"❌ No hybrid results found for condition {test_condition}")
        print("Run hybrid approach first (option 17)")
        return None
    
    hybrid_files.sort(reverse=True)
    hybrid_path = os.path.join(results_dir, hybrid_files[0])
    print(f"Using hybrid results: {hybrid_files[0]}")
    
    hybrid_df = pd.read_csv(hybrid_path)
    
    # Find corresponding standard results (same test condition)
    standard_files = [f for f in os.listdir(results_dir) 
                     if f.startswith('results_Cross-condition') and str(test_condition) in f
                     and 'simple_lstm' in f and 'hybrid' not in f]
    
    if not standard_files:
        # Try to find any standard results
        standard_files = [f for f in os.listdir(results_dir) 
                         if f.startswith('results_') and 'Cross-condition' in f 
                         and str(test_condition) in f and 'hybrid' not in f]
    
    if not standard_files:
        print(f"❌ No standard results found for condition {test_condition}")
        print("Run standard approach first (option 14)")
        return None
    
    standard_files.sort(reverse=True)
    standard_path = os.path.join(results_dir, standard_files[0])
    print(f"Using standard results: {standard_files[0]}")
    
    standard_df = pd.read_csv(standard_path)
    
    # Calculate metrics
    hybrid_metrics = calculate_comprehensive_metrics(
        hybrid_df['predicted_rul'].values,
        hybrid_df['actual_rul'].values
    )
    
    standard_metrics = calculate_comprehensive_metrics(
        standard_df['predicted_rul'].values,
        standard_df['actual_rul'].values
    )
    
    # Statistical significance test
    from scipy import stats
    
    hybrid_errors = np.abs(hybrid_df['error'].values)
    standard_errors = np.abs(standard_df['error'].values)
    
    # Ensure same length (take min)
    min_len = min(len(hybrid_errors), len(standard_errors))
    hybrid_errors = hybrid_errors[:min_len]
    standard_errors = standard_errors[:min_len]
    
    # Paired t-test
    t_stat, p_value = stats.ttest_rel(standard_errors, hybrid_errors)
    
    # Print comparison
    print(f"\n{'Metric':<15} {'Standard':<15} {'Hybrid':<15} {'Improvement':<15} {'% Change':<15}")
    print("-" * 75)
    
    for metric in ['MAE', 'RMSE', 'R2', 'Within_10%']:
        if metric in standard_metrics and metric in hybrid_metrics:
            std_val = standard_metrics[metric]
            hyb_val = hybrid_metrics[metric]
            
            if metric in ['MAE', 'RMSE']:
                diff = std_val - hyb_val  # Positive means hybrid is better
                pct_change = (diff / std_val * 100) if std_val != 0 else 0
                print(f"{metric:<15} {std_val:<15.3f} {hyb_val:<15.3f} {diff:<15.3f} {pct_change:+.1f}%")
            else:
                diff = hyb_val - std_val  # Positive means hybrid is better
                pct_change = (diff / abs(std_val) * 100) if std_val != 0 else 0
                print(f"{metric:<15} {std_val:<15.4f} {hyb_val:<15.4f} {diff:<15.4f} {pct_change:+.1f}%")
    
    print(f"\n📊 Statistical Significance Test:")
    print(f"   Paired t-test: t = {t_stat:.3f}, p = {p_value:.4f}")
    
    if p_value < 0.05:
        if np.mean(hybrid_errors) < np.mean(standard_errors):
            print(f"   ✅ SIGNIFICANT: Hybrid approach is BETTER (p < 0.05)")
        else:
            print(f"   ⚠️  SIGNIFICANT: Standard approach is BETTER (p < 0.05)")
    else:
        print(f"   ℹ️  NOT SIGNIFICANT: No difference (p ≥ 0.05)")
    
    # Create comparison plot
    fig, axes = plt.subplots(2, 3, figsize=(15, 10))
    
    # 1. MAE comparison
    models = ['Standard', 'Hybrid']
    mae_values = [standard_metrics['MAE'], hybrid_metrics['MAE']]
    colors = ['red', 'green']
    axes[0, 0].bar(models, mae_values, color=colors, alpha=0.7)
    axes[0, 0].set_title('MAE Comparison\n(lower is better)', fontweight='bold')
    axes[0, 0].set_ylabel('MAE (minutes)')
    axes[0, 0].grid(True, alpha=0.3, axis='y')
    
    # Add value labels
    for i, v in enumerate(mae_values):
        axes[0, 0].text(i, v + max(mae_values)*0.02, f'{v:.1f}', 
                       ha='center', va='bottom', fontweight='bold')
    
    # 2. R² comparison
    r2_values = [standard_metrics['R2'], hybrid_metrics['R2']]
    axes[0, 1].bar(models, r2_values, color=colors, alpha=0.7)
    axes[0, 1].set_title('R² Comparison\n(higher is better)', fontweight='bold')
    axes[0, 1].set_ylabel('R² Score')
    axes[0, 1].grid(True, alpha=0.3, axis='y')
    
    for i, v in enumerate(r2_values):
        axes[0, 1].text(i, v + max(r2_values)*0.02, f'{v:.3f}', 
                       ha='center', va='bottom', fontweight='bold')
    
    # 3. Within 10% comparison
    if 'Within_10%' in standard_metrics and 'Within_10%' in hybrid_metrics:
        within_10 = [standard_metrics['Within_10%'], hybrid_metrics['Within_10%']]
        axes[0, 2].bar(models, within_10, color=colors, alpha=0.7)
        axes[0, 2].set_title('Within 10% Accuracy\n(higher is better)', fontweight='bold')
        axes[0, 2].set_ylabel('Percentage (%)')
        axes[0, 2].grid(True, alpha=0.3, axis='y')
        
        for i, v in enumerate(within_10):
            axes[0, 2].text(i, v + max(within_10)*0.02, f'{v:.1f}%', 
                           ha='center', va='bottom', fontweight='bold')
    
    # 4. Error distribution
    axes[1, 0].hist(standard_errors, bins=30, alpha=0.5, label='Standard', 
                   color='red', density=True)
    axes[1, 0].hist(hybrid_errors, bins=30, alpha=0.5, label='Hybrid', 
                   color='green', density=True)
    axes[1, 0].axvline(np.mean(standard_errors), color='red', linestyle='--', 
                      label=f'Std mean: {np.mean(standard_errors):.1f}')
    axes[1, 0].axvline(np.mean(hybrid_errors), color='green', linestyle='--', 
                      label=f'Hyb mean: {np.mean(hybrid_errors):.1f}')
    axes[1, 0].set_xlabel('Absolute Error (minutes)')
    axes[1, 0].set_ylabel('Density')
    axes[1, 0].set_title('Error Distribution Comparison')
    axes[1, 0].legend()
    axes[1, 0].grid(True, alpha=0.3)
    
    # 5. Scatter plot comparison
    sample_size = min(100, len(standard_df), len(hybrid_df))
    axes[1, 1].scatter(standard_df['actual_rul'].iloc[:sample_size], 
                      standard_df['predicted_rul'].iloc[:sample_size],
                      alpha=0.5, s=10, label='Standard', color='red')
    axes[1, 1].scatter(hybrid_df['actual_rul'].iloc[:sample_size], 
                      hybrid_df['predicted_rul'].iloc[:sample_size],
                      alpha=0.5, s=10, label='Hybrid', color='green')
    max_val = max(standard_df['actual_rul'].max(), hybrid_df['actual_rul'].max(),
                 standard_df['predicted_rul'].max(), hybrid_df['predicted_rul'].max())
    axes[1, 1].plot([0, max_val], [0, max_val], 'k--', label='Perfect', linewidth=2)
    axes[1, 1].set_xlabel('Actual RUL')
    axes[1, 1].set_ylabel('Predicted RUL')
    axes[1, 1].set_title('Predictions Comparison (100 samples)')
    axes[1, 1].legend()
    axes[1, 1].grid(True, alpha=0.3)
    
    # 6. Statistical summary
    axes[1, 2].axis('off')
    
    summary_text = (
        f"PHD COMPARISON SUMMARY\n"
        f"{'='*40}\n"
        f"Test Condition: {test_condition}\n"
        f"Hybrid Early Ratio: {hybrid_df['early_ratio'].iloc[0]*100}%\n"
        f"Sample size: {min_len}\n"
        f"{'='*40}\n"
        f"Statistical Test: Paired t-test\n"
        f"t-statistic: {t_stat:.3f}\n"
        f"p-value: {p_value:.4f}\n"
        f"{'='*40}\n"
        f"STANDARD APPROACH:\n"
        f"  MAE: {standard_metrics['MAE']:.1f} min\n"
        f"  R²: {standard_metrics['R2']:.4f}\n"
        f"  Within 10%: {standard_metrics.get('Within_10%', 0):.1f}%\n"
        f"{'='*40}\n"
        f"HYBRID APPROACH:\n"
        f"  MAE: {hybrid_metrics['MAE']:.1f} min\n"
        f"  R²: {hybrid_metrics['R2']:.4f}\n"
        f"  Within 10%: {hybrid_metrics.get('Within_10%', 0):.1f}%\n"
        f"{'='*40}\n"
        f"Conclusion: {'Hybrid SIGNIFICANTLY BETTER' if p_value < 0.05 and hybrid_metrics['MAE'] < standard_metrics['MAE'] else 'No significant advantage'}"
    )
    
    axes[1, 2].text(0.1, 0.5, summary_text, fontsize=9, fontfamily='monospace',
                   verticalalignment='center')
    
    plt.suptitle(f'PhD Comparison: Standard vs Hybrid Approach\nCondition {test_condition}', 
                fontsize=16, fontweight='bold')
    plt.tight_layout()
    
    comparison_file = os.path.join(results_dir, f"phd_hybrid_vs_standard_cond{test_condition}.png")
    plt.savefig(comparison_file, dpi=150, bbox_inches='tight')
    plt.show()
    
    print(f"\n✅ Comparison saved to: {comparison_file}")
    
    return standard_metrics, hybrid_metrics, p_value

# ============================================================================
# IMPROVED VALIDATION FUNCTIONS FOR PHD
# ============================================================================

from scipy import stats
from statsmodels.stats.power import tt_solve_power  # optional; fallback provided
import warnings

def calculate_power(effect_size, n, alpha=0.05, tails=2):
    """
    Calculate achieved statistical power for a paired t-test.
    
    Parameters:
        effect_size (float): Cohen's d
        n (int): sample size
        alpha (float): significance level
        tails (int): 1 or 2
    
    Returns:
        power (float): achieved power (0-1)
    """
    try:
        # Use statsmodels if available
        from statsmodels.stats.power import tt_solve_power
        return tt_solve_power(effect_size=effect_size, nobs=n, alpha=alpha, power=None, alternative='two-sided')
    except ImportError:
        # Fallback: manual calculation using non-central t-distribution
        from scipy.stats import nct
        nc = effect_size * np.sqrt(n)  # non-centrality parameter
        if tails == 2:
            t_crit = stats.t.ppf(1 - alpha/2, df=n-1)
            power = 1 - nct.cdf(t_crit, df=n-1, nc=nc) + nct.cdf(-t_crit, df=n-1, nc=nc)
        else:
            t_crit = stats.t.ppf(1 - alpha, df=n-1)
            power = 1 - nct.cdf(t_crit, df=n-1, nc=nc)
        return power

def run_quick_validation():
    """
    Quick validation checks for PhD thesis:
    1. Verify that each bearing's windows are in the same split (no leakage)
    2. Check RUL monotonicity and range
    3. Ensure no file reading errors
    """
    print("\n" + "="*80)
    print("QUICK VALIDATION CHECKS")
    print("="*80)
    
    # Load one dataset as a test (Split 2)
    DATA_ROOT = "/kaggle/input/xj-dataset/XJTU-SY_Bearing_Datasets"
    from collections import defaultdict
    
    # Identify bearings for Split 2 (Cross-condition 2)
    # We'll just load a small dataset for validation
    test_bearings = ['Bearing2_1', 'Bearing2_2', 'Bearing2_3']  # any available
    try:
        dataset = EnhancedXJTU_EWS_Dataset(
            data_root=DATA_ROOT,
            bearing_list=test_bearings,
            window_size=10,
            step_size=5,
            enable_caching=False,  # force fresh processing
            include_wavelet=True,
            ews_window_size=1024,
            ews_step_size=1024
        )
    except Exception as e:
        print(f"❌ Could not load dataset: {e}")
        return
    
    # 1. Check that each bearing's windows are contiguous in the split
    bearing_to_indices = defaultdict(list)
    for idx, bid in enumerate(dataset.bearing_ids):
        bearing_to_indices[bid].append(idx)
    
    leakage_ok = True
    for bid, indices in bearing_to_indices.items():
        # Check if indices are consecutive (no mixing with other bearings)
        # Since dataset is created by iterating over bearings in order,
        # indices for each bearing should be a contiguous block.
        if len(indices) > 1:
            # Verify that the indices are consecutive
            diffs = np.diff(indices)
            if not np.all(diffs == 1):
                print(f"❌ Bearing {bid} indices are not consecutive: {indices[:5]}...")
                leakage_ok = False
            else:
                print(f"✅ Bearing {bid}: {len(indices)} consecutive windows")
        else:
            print(f"⚠️ Bearing {bid} has only {len(indices)} window")
    
    if leakage_ok:
        print("✅ All bearings have contiguous windows – no data leakage across bearings.")
    else:
        print("❌ Data leakage detected! Windows from same bearing are interleaved.")
    
    # 2. Check RUL monotonicity and range
    rul_vals = np.array(dataset.rul_targets)
    print(f"\nRUL statistics:")
    print(f"  Min: {rul_vals.min():.4f}, Max: {rul_vals.max():.4f}")
    print(f"  Mean: {rul_vals.mean():.4f}, Std: {rul_vals.std():.4f}")
    
    if np.all((rul_vals >= 0) & (rul_vals <= 1)):
        print("✅ All RUL values in [0,1]")
    else:
        print("❌ Some RUL values outside [0,1]")
    
    # Check monotonicity per bearing
    print("\nMonotonicity per bearing:")
    for bid in set(dataset.bearing_ids):
        mask = [i for i, b in enumerate(dataset.bearing_ids) if b == bid]
        if len(mask) > 1:
            rul_b = rul_vals[mask]
            # RUL should decrease over time (within bearing)
            # We'll check if sorted order is decreasing
            if np.all(np.diff(rul_b) <= 0):
                print(f"  ✅ {bid}: RUL is monotonic decreasing")
            else:
                # Maybe small fluctuations due to noise – count violations
                violations = np.sum(np.diff(rul_b) > 0)
                print(f"  ⚠️ {bid}: {violations} increases in RUL (may be acceptable)")
        else:
            print(f"  ⚠️ {bid}: insufficient windows to check monotonicity")
    
    # 3. File reading errors: already handled by dataset creation
    print(f"\nTotal sequences created: {len(dataset)}")
    print(f"Skipped files: {dataset.skipped_files}")
    if dataset.skipped_files == 0:
        print("✅ No file reading errors")
    else:
        print(f"⚠️ {dataset.skipped_files} files were skipped – check logs")
    
    print("="*80)

def run_statistical_tests(split_index=1, baseline_model='simple_lstm'):
    """
    Perform statistical significance testing and power analysis
    on existing results for a given split.
    
    Args:
        split_index: 0,1,2 for the three cross-condition splits
        baseline_model: name of the baseline model for comparisons
    """
    print("\n" + "="*80)
    print("STATISTICAL SIGNIFICANCE TESTING")
    print("="*80)
    
    results_dir = "/kaggle/working/results"
    if not os.path.exists(results_dir):
        print("❌ Results directory not found. Run experiments first.")
        return
    
    # Map split index to split name pattern
    split_names = ["Cross-condition 1", "Cross-condition 2", "Cross-condition 3"]
    split_name = split_names[split_index]
    split_tag = split_name.replace(' ', '_')
    
    # Find all result files for this split
    result_files = [f for f in os.listdir(results_dir) 
                    if f.startswith('results_') and split_tag in f and f.endswith('.csv')]
    
    if not result_files:
        print(f"❌ No result files found for {split_name}")
        return
    
    # Group by model type (extract model type from filename)
    model_groups = {}
    for f in result_files:
        # Filename format: results_{split_tag}_{model_type}_wavelet..._seed...csv
        parts = f.split('_')
        # Find model type – usually after split_tag
        # e.g., results_Cross-condition_2_lstm_da_waveletTrue_...
        # So model type is at index 4 (depending on split_tag length)
        # Simpler: extract everything between split_tag and 'wavelet'
        import re
        match = re.search(rf'{split_tag}_(.*?)_wavelet', f)
        if match:
            model_type = match.group(1)
        else:
            model_type = 'unknown'
        
        df = pd.read_csv(os.path.join(results_dir, f))
        model_groups[model_type] = df
    
    if baseline_model not in model_groups:
        print(f"⚠️ Baseline model '{baseline_model}' not found. Available: {list(model_groups.keys())}")
        # use first model as baseline
        baseline_model = list(model_groups.keys())[0]
        print(f"Using '{baseline_model}' as baseline.")
    
    baseline_df = model_groups[baseline_model]
    baseline_mae = baseline_df['abs_error'].values  # absolute error
    
    # Compare each other model against baseline
    print(f"\nComparing against baseline: {baseline_model}")
    print("-" * 80)
    print(f"{'Model':<20} {'MAE':<10} {'t-stat':<8} {'p-value':<10} {'Sig.':<6} {'Cohen\'s d':<10} {'Power':<10}")
    print("-" * 80)
    
    comparisons = []
    for model, df in model_groups.items():
        if model == baseline_model:
            continue
        # Ensure same length – take intersection of indices? Actually they are different samples.
        # We need to align by bearing_id and time? Since the test set is the same across experiments,
        # the order of rows should be the same if the test set is fixed. We'll assume they are aligned.
        # But to be safe, we can merge on bearing_id and actual_rul? Better: we expect same number of rows.
        if len(df) != len(baseline_df):
            print(f"⚠️ {model}: length mismatch ({len(df)} vs {len(baseline_df)}), skipping")
            continue
        
        # Paired t-test on absolute errors
        t_stat, p_val = stats.ttest_rel(baseline_mae, df['abs_error'].values)
        cohen_d = (baseline_mae.mean() - df['abs_error'].mean()) / np.sqrt((baseline_mae.var() + df['abs_error'].var()) / 2)
        # Power calculation
        power = calculate_power(abs(cohen_d), n=len(df), alpha=0.05)
        sig = "Yes" if p_val < 0.05 else "No"
        
        print(f"{model:<20} {df['abs_error'].mean():<10.2f} {t_stat:<8.3f} {p_val:<10.4f} {sig:<6} {cohen_d:<10.3f} {power:<10.3f}")
        comparisons.append((model, p_val, cohen_d, power))
    
    # Also compare the two best models (lowest MAE) against each other
    if len(model_groups) >= 2:
        # Find two best by MAE
        maes = {m: df['abs_error'].mean() for m, df in model_groups.items()}
        sorted_models = sorted(maes, key=maes.get)
        best = sorted_models[0]
        second = sorted_models[1]
        print("\n" + "-"*80)
        print(f"Comparing best two models: {best} vs {second}")
        df_best = model_groups[best]
        df_second = model_groups[second]
        if len(df_best) == len(df_second):
            t_stat, p_val = stats.ttest_rel(df_best['abs_error'].values, df_second['abs_error'].values)
            cohen_d = (df_best['abs_error'].mean() - df_second['abs_error'].mean()) / np.sqrt((df_best['abs_error'].var() + df_second['abs_error'].var()) / 2)
            power = calculate_power(abs(cohen_d), n=len(df_best), alpha=0.05)
            sig = "Yes" if p_val < 0.05 else "No"
            print(f"t-stat: {t_stat:.3f}, p-value: {p_val:.4f}, Significant: {sig}")
            print(f"Cohen's d: {cohen_d:.3f}, Power: {power:.3f}")
        else:
            print("Length mismatch – cannot compare.")
    
    print("="*80)
    
    # Save comparison to file
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    out_file = os.path.join(results_dir, f"statistical_tests_split{split_index}_{timestamp}.txt")
    with open(out_file, 'w') as f:
        f.write("Statistical Significance Testing Results\n")
        f.write(f"Split: {split_name}\n\n")
        for line in comparisons:
            f.write(f"{line}\n")
    print(f"Results saved to {out_file}")

def run_kfold_cv(split_index=1, n_folds=5, model_type='simple_lstm', seed=42):
    """
    Perform k-fold cross-validation on the training set of a given split.
    This demonstrates model stability and avoids overfitting to a single validation split.
    
    Args:
        split_index: 0,1,2
        n_folds: number of folds
        model_type: which model to evaluate
        seed: random seed
    """
    print("\n" + "="*80)
    print(f"K-FOLD CROSS-VALIDATION ({n_folds} folds) on Split {split_index+1}")
    print("="*80)
    
    DATA_ROOT = "/kaggle/input/xj-dataset/XJTU-SY_Bearing_Datasets"
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    
    # Determine bearings for this split
    ALL_BEARINGS = [
        'Bearing1_1', 'Bearing1_2', 'Bearing1_3', 'Bearing1_4', 'Bearing1_5',
        'Bearing2_1', 'Bearing2_2', 'Bearing2_3', 'Bearing2_4', 'Bearing2_5',
        'Bearing3_1', 'Bearing3_2', 'Bearing3_3', 'Bearing3_4', 'Bearing3_5'
    ]
    available_bearings = []
    for b in ALL_BEARINGS:
        if 'Bearing1_' in b:
            cond = '35Hz12kN'
        elif 'Bearing2_' in b:
            cond = '37.5Hz11kN'
        else:
            cond = '40Hz10kN'
        if os.path.exists(os.path.join(DATA_ROOT, cond, b)):
            available_bearings.append(b)
    
    bearing1_set = [b for b in available_bearings if 'Bearing1_' in b]
    bearing2_set = [b for b in available_bearings if 'Bearing2_' in b]
    bearing3_set = [b for b in available_bearings if 'Bearing3_' in b]
    
    split_options = [
        (bearing1_set + bearing2_set, bearing3_set),  # Split 1: train A+B, test C
        (bearing1_set + bearing3_set, bearing2_set),  # Split 2: train A+C, test B
        (bearing2_set + bearing3_set, bearing1_set),  # Split 3: train B+C, test A
    ]
    train_bearings, _ = split_options[split_index]
    
    # Create full training dataset
    dataset = EnhancedXJTU_EWS_Dataset(
        data_root=DATA_ROOT,
        bearing_list=train_bearings,
        window_size=10,
        step_size=5,
        use_both_channels=True,
        fusion_method='concat',
        enable_caching=True,
        include_wavelet=True,
        ews_window_size=1024,
        ews_step_size=1024
    )
    
    if len(dataset) == 0:
        print("❌ Empty dataset")
        return
    
    feature_dim = dataset.get_feature_dimension()
    print(f"Total sequences: {len(dataset)}, Feature dim: {feature_dim}")
    
    # Get bearing IDs for stratification
    bearing_ids = dataset.bearing_ids
    unique_bearings = list(set(bearing_ids))
    
    # Group indices by bearing
    bearing_indices = {b: [] for b in unique_bearings}
    for idx, bid in enumerate(bearing_ids):
        bearing_indices[bid].append(idx)
    
    # Create folds: each fold uses whole bearings (to avoid leakage)
    # Shuffle bearings
    rng = np.random.RandomState(seed)
    rng.shuffle(unique_bearings)
    
    # Split bearings into n_folds
    folds = []
    fold_size = len(unique_bearings) // n_folds
    for i in range(n_folds):
        if i == n_folds - 1:
            fold_bearings = unique_bearings[i*fold_size:]
        else:
            fold_bearings = unique_bearings[i*fold_size:(i+1)*fold_size]
        fold_indices = []
        for b in fold_bearings:
            fold_indices.extend(bearing_indices[b])
        folds.append(fold_indices)
    
    # Now perform cross-validation
    fold_results = []
    
    for fold in range(n_folds):
        print(f"\n--- Fold {fold+1}/{n_folds} ---")
        
        # Validation indices = this fold
        val_idx = folds[fold]
        # Train indices = all others
        train_idx = []
        for f in range(n_folds):
            if f != fold:
                train_idx.extend(folds[f])
        
        # Create subsets
        train_set = Subset(dataset, train_idx)
        val_set = Subset(dataset, val_idx)
        
        train_loader = DataLoader(train_set, batch_size=32, shuffle=True, num_workers=0)
        val_loader = DataLoader(val_set, batch_size=32, shuffle=False, num_workers=0)
        
        # Create model
        if model_type == 'simple_lstm':
            model = Simple_LSTM(input_size=feature_dim, hidden_size=128, num_layers=2, dropout=0.3)
        elif model_type == 'lstm_da':
            model = EWS_LSTM_Attention_DA(input_size=feature_dim, hidden_size=128, num_layers=2, dropout=0.3, use_da=True)
        elif model_type == 'bilstm_attn':
            model = EWS_LSTM_Attention(input_size=feature_dim, hidden_size=128, num_layers=2, dropout=0.3)
        else:
            model = Simple_LSTM(input_size=feature_dim, hidden_size=128, num_layers=2)
        
        model.to(device)
        optimizer = optim.AdamW(model.parameters(), lr=0.001, weight_decay=1e-4)
        criterion = nn.MSELoss()
        
        # Train for a few epochs (or use early stopping)
        epochs = 30  # reduced for speed
        for epoch in range(epochs):
            model.train()
            for data, targets, _, _ in train_loader:
                data, targets = data.to(device), targets.to(device)
                optimizer.zero_grad()
                outputs = model(data)
                loss = criterion(outputs, targets)
                loss.backward()
                optimizer.step()
        
        # Evaluate on validation set
        model.eval()
        preds, targs = [], []
        with torch.no_grad():
            for data, targets, _, max_ruls in val_loader:
                data = data.to(device)
                outputs = model(data)
                for i in range(len(outputs)):
                    pred_norm = outputs[i].item()
                    targ_norm = targets[i].item()
                    max_rul = max_ruls[i].item()
                    pred_actual = pred_norm * max_rul
                    targ_actual = targ_norm * max_rul
                    preds.append(pred_actual)
                    targs.append(targ_actual)
        
        mae = mean_absolute_error(targs, preds)
        rmse = np.sqrt(mean_squared_error(targs, preds))
        r2 = r2_score(targs, preds)
        fold_results.append({'fold': fold+1, 'mae': mae, 'rmse': rmse, 'r2': r2})
        print(f"  MAE: {mae:.2f}, RMSE: {rmse:.2f}, R²: {r2:.4f}")
    
    # Summarize
    print("\n" + "="*80)
    print("CROSS-VALIDATION SUMMARY")
    print("="*80)
    mae_vals = [r['mae'] for r in fold_results]
    rmse_vals = [r['rmse'] for r in fold_results]
    r2_vals = [r['r2'] for r in fold_results]
    print(f"MAE:  mean={np.mean(mae_vals):.2f} ± {np.std(mae_vals):.2f}")
    print(f"RMSE: mean={np.mean(rmse_vals):.2f} ± {np.std(rmse_vals):.2f}")
    print(f"R²:   mean={np.mean(r2_vals):.4f} ± {np.std(r2_vals):.4f}")
    print("="*80)
    
    # Save results
    results_dir = "/kaggle/working/results"
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    out_file = os.path.join(results_dir, f"kfold_split{split_index}_{model_type}_{timestamp}.csv")
    pd.DataFrame(fold_results).to_csv(out_file, index=False)
    print(f"Results saved to {out_file}")

def run_comprehensive_validation_suite():
    """Run all validation checks sequentially."""
    print("\n" + "="*80)
    print("COMPREHENSIVE PHD VALIDATION SUITE")
    print("="*80)
    
    # 1. Quick validation
    run_quick_validation()
    
    # 2. Statistical tests on Split 2 (or user can specify)
    run_statistical_tests(split_index=1)
    
    # 3. K-fold CV on Split 2 with simple LSTM
    run_kfold_cv(split_index=1, n_folds=5, model_type='simple_lstm')
    
    print("\n" + "="*80)
    print("PHD VALIDATION SUITE COMPLETED")
    print("="*80)
# ============================================================================
# MAIN EXECUTION
# ============================================================================
if __name__ == "__main__":
    print("=" * 80)
    print("ENHANCED EWS RUL PREDICTION WITH LSTM+ATTENTION (FIXED VERSION)")
    print("Based on Literature Findings: Deep sequence models with attention")
    print("=" * 80)
    
    create_directories()
    set_seed(42)
    
    print("\nSelect option:")
    print("1. Run Split 1 (Cross-condition 1) with LSTM+Attention+DA")
    print("2. Run Split 2 (Cross-condition 2) with LSTM+Attention+DA")
    print("3. Run Split 3 (Cross-condition 3) with LSTM+Attention+DA")
    print("4. Run Split 1 with Bidirectional LSTM+Attention (no DA)")
    print("5. Run Split 2 with Bidirectional LSTM+Attention (no DA)")
    print("6. Run Split 3 with Bidirectional LSTM+Attention (no DA)")
    print("7. Run all splits with LSTM+DA (skip existing)")
    print("8. Check existing results")
    print("9. Clear cache")
    print("10. Compare models on Split 2")
    print("11. Run Split 2 with Bidirectional LSTM (No Attention)")
    print("12. Run Split 2 with Unidirectional LSTM (With Attention)")
    print("13. Run Split 2 with Unidirectional LSTM (No Attention)")
    print("14. Run Split 2 with Simple LSTM (Vanilla)")
    print("15. Run Complete Model Comparison Study on Split 2")
    print("\n--- ABLATION STUDIES (Phase 2) ---")
    print("16. Run ablation: No wavelet features (Split 2)")
    print("17. Run ablation: Sequence length variations (Split 2)")
    print("18. Run ablation: Window size variations (Split 2)")
    print("19. Run Raw vibration ablation (Split 2)")
    print("20. Run complete ablation study (Split 2)")
    # Add to your main menu options
    print("\n--- HYBRID APPROACH (Phase 3) ---")
    print("21. Run Hybrid Approach (Condition 2)")
    print("22. Run Hybrid Approach (Condition 1)")
    print("23. Run Hybrid Approach (Condition 3)")
    print("24. Compare Standard vs Hybrid (Condition 2)")
    print("25. Run ablation: Early data ratio variations")
    # In the main execution section, update the menu:
    print("\n--- VALIDATION SUITE ---")
    print("26. Run quick validation (5 min)")
    print("27. Run statistical tests (10 min)")
    print("28. Run k-fold CV (30 min)")
    print("29. Run comprehensive validation suite (60 min)")


    
    # Default validation split
    DEFAULT_VAL_SPLIT = 0.2
    choice = input("\nChoice (1-25): ").strip()

    if choice == "1":
        print("\nRunning Split 1 (Cross-condition 1) with LSTM+Attention+DA...")
        run_selected_split(0, skip_if_exists=True, model_type='lstm_da', use_da=True, 
                          val_split=DEFAULT_VAL_SPLIT, ews_step_size=1024)
    elif choice == "2":
        print("\nRunning Split 2 (Cross-condition 2) with LSTM+Attention+DA...")
        run_selected_split(1, skip_if_exists=True, model_type='lstm_da', use_da=True, 
                          val_split=DEFAULT_VAL_SPLIT, ews_step_size=1024)
    elif choice == "3":
        print("\nRunning Split 3 (Cross-condition 3) with LSTM+Attention+DA...")
        run_selected_split(2, skip_if_exists=True, model_type='lstm_da', use_da=True, 
                          val_split=DEFAULT_VAL_SPLIT, ews_step_size=1024)
    elif choice == "4":
        print("\nRunning Split 1 with Bidirectional LSTM+Attention (no DA)...")
        run_selected_split(0, skip_if_exists=True, model_type='bilstm_attn', use_da=False, 
                          val_split=DEFAULT_VAL_SPLIT, ews_step_size=1024)
    elif choice == "5":
        print("\nRunning Split 2 with Bidirectional LSTM+Attention (no DA)...")
        run_selected_split(1, skip_if_exists=True, model_type='bilstm_attn', use_da=False, 
                          val_split=DEFAULT_VAL_SPLIT, ews_step_size=1024)
    elif choice == "6":
        print("\nRunning Split 3 with Bidirectional LSTM+Attention (no DA)...")
        run_selected_split(2, skip_if_exists=True, model_type='bilstm_attn', use_da=False, 
                          val_split=DEFAULT_VAL_SPLIT, ews_step_size=1024)
    elif choice == "7":
        print("\nRunning all splits with LSTM+DA (will skip existing)...")
        for i in range(3):
            run_selected_split(i, skip_if_exists=True, model_type='lstm_da', use_da=True, 
                              val_split=DEFAULT_VAL_SPLIT, ews_step_size=1024)
    elif choice == "8":
        print("\n📁 Checking existing results...")
        results_dir = "/kaggle/working/results"
        if os.path.exists(results_dir):
            print(f"Files in {results_dir}:")
            for file in sorted(os.listdir(results_dir)):
                file_path = os.path.join(results_dir, file)
                size_kb = os.path.getsize(file_path) / 1024
                print(f"  {file} ({size_kb:.1f} KB)")
        else:
            print("❌ Results directory does not exist.")
    elif choice == "9":
        cache_dir = "/kaggle/working/cache"
        if os.path.exists(cache_dir):
            shutil.rmtree(cache_dir)
            print(f"Cache cleared: {cache_dir}")
        else:
            print("Cache directory does not exist.")
    elif choice == "10":
        print("\nComparing models on Split 2...")
        models_to_try = ['lstm_da', 'bilstm_attn', 'bilstm_noattn', 'lstm_attn', 'lstm_noattn', 'simple_lstm']
        for model_type in models_to_try:
            print(f"\n{'=' * 40}")
            print(f"Testing {model_type} on Split 2")
            print('=' * 40)
            run_selected_split(1, skip_if_exists=True, model_type=model_type, 
                             use_da=(model_type == 'lstm_da'), val_split=DEFAULT_VAL_SPLIT,
                             ews_step_size=1024)
    elif choice == "11":
        print("\nRunning Split 2 with Bidirectional LSTM (No Attention)...")
        run_selected_split(1, skip_if_exists=True, model_type='bilstm_noattn', use_da=False, 
                          val_split=DEFAULT_VAL_SPLIT, ews_step_size=1024)
    elif choice == "12":
        print("\nRunning Split 2 with Unidirectional LSTM (With Attention)...")
        run_selected_split(1, skip_if_exists=True, model_type='lstm_attn', use_da=False, 
                          val_split=DEFAULT_VAL_SPLIT, ews_step_size=1024)
    elif choice == "13":
        print("\nRunning Split 2 with Unidirectional LSTM (No Attention)...")
        run_selected_split(1, skip_if_exists=True, model_type='lstm_noattn', use_da=False, 
                          val_split=DEFAULT_VAL_SPLIT, ews_step_size=1024)
    elif choice == "14":
        print("\nRunning Split 2 with Simple LSTM (Vanilla)...")
        run_selected_split(1, skip_if_exists=True, model_type='simple_lstm', use_da=False, 
                          val_split=DEFAULT_VAL_SPLIT, ews_step_size=1024)
    elif choice == "15":
        print("\nRunning Complete Model Comparison Study on Split 2...")
        models_to_try = ['lstm_da', 'bilstm_attn', 'bilstm_noattn', 'lstm_attn', 'lstm_noattn', 'simple_lstm']
        for model_type in models_to_try:
            print(f"\n{'=' * 60}")
            print(f"Testing {model_type} on Split 2")
            print('=' * 60)
            run_selected_split(1, skip_if_exists=True, model_type=model_type, 
                             use_da=(model_type == 'lstm_da'), val_split=DEFAULT_VAL_SPLIT,
                             ews_step_size=1024)
    elif choice == "16":
        print("\nRunning ablation: No wavelet features (Split 2)...")
        run_selected_split(1, skip_if_exists=True, model_type='simple_lstm', 
                          use_da=False, include_wavelet=False, val_split=DEFAULT_VAL_SPLIT,
                          ews_step_size=1024)
    elif choice == "17":
        print("\nRunning ablation: Sequence length variations (Split 2)...")
        for seq_len in [20, 30]:
            print(f"\nTesting Sequence Length: {seq_len}")
            run_selected_split(1, skip_if_exists=True, model_type='lstm_noattn', 
                              use_da=False, sequence_length=seq_len, val_split=DEFAULT_VAL_SPLIT,
                              ews_step_size=1024)
    elif choice == "18":
        print("\nRunning ablation: Window size variations (Split 2)...")
        # Research-based window sizes
        for window_size in [2048]:
            print(f"\nTesting Window Size: {window_size} samples ({window_size/25600*1000:.1f} ms)")
            run_selected_split(1, skip_if_exists=True, model_type='lstm_noattn', 
                              use_da=False, ews_window_size=window_size, val_split=DEFAULT_VAL_SPLIT,
                              ews_step_size=1024)
            
    elif choice == "19":
        print("\nRunning Raw vibration ablation (Split 2)...")
        run_raw_vibration_ablation(split_index=1, skip_if_exists=True, val_split=0.2)
        
    elif choice == "20":
        print("\nRunning complete ablation study (Split 2)...")
        # Run baseline
        run_selected_split(1, skip_if_exists=True, model_type='simple_lstm', 
                          use_da=False, val_split=DEFAULT_VAL_SPLIT, ews_step_size=1024)
        # Run no wavelet
        run_selected_split(1, skip_if_exists=True, model_type='simple_lstm', 
                          use_da=False, include_wavelet=False, val_split=DEFAULT_VAL_SPLIT,
                          ews_step_size=1024)
        # Run sequence lengths
        for seq_len in [5, 20, 30]:
            run_selected_split(1, skip_if_exists=True, model_type='simple_lstm', 
                              use_da=False, sequence_length=seq_len, val_split=DEFAULT_VAL_SPLIT,
                              ews_step_size=1024)
        # Run window sizes
        for window_size in [512, 2048]:
            run_selected_split(1, skip_if_exists=True, model_type='simple_lstm', 
                              use_da=False, ews_window_size=window_size, val_split=DEFAULT_VAL_SPLIT,
                              ews_step_size=1024)
        # Add to your main execution
    elif choice == "21":
        print("\nRunning Hybrid Approach on Condition 2...")
        run_hybrid_approach(test_condition=2, early_ratio=0.2, model_type='lstm_attn')
    elif choice == "24":
        print("\nComparing Standard vs Hybrid Approach on Condition 2...")
        compare_approaches(test_condition=2)
      
    elif choice == "26":
        run_quick_validation()
    elif choice == "27":
        split_choice = input("Enter split index (0/1/2) [default=1]: ").strip()
        if split_choice in ['0','1','2']:
            split_idx = int(split_choice)
        else:
            split_idx = 1
        run_statistical_tests(split_index=split_idx)
    elif choice == "28":
        split_choice = input("Enter split index (0/1/2) [default=1]: ").strip()
        if split_choice in ['0','1','2']:
            split_idx = int(split_choice)
        else:
            split_idx = 1
        model_choice = input("Model type (simple_lstm/lstm_da/bilstm_attn) [default=simple_lstm]: ").strip()
        if model_choice not in ['simple_lstm','lstm_da','bilstm_attn']:
            model_choice = 'simple_lstm'
        run_kfold_cv(split_index=split_idx, n_folds=5, model_type=model_choice)
    elif choice == "29":
        run_comprehensive_validation_suite()

    else:
        print("Invalid choice")

# STATISTICAL VALIDATION

In [ ]:
# ============================================================================
# CORRECTED STATISTICAL VALIDATION – USING MAIN CODE'S MODELS & TRAINING
# ============================================================================
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, Subset
import os
import pickle
import json
from collections import defaultdict
import matplotlib.pyplot as plt
from scipy import stats
import warnings
import logging
from datetime import datetime

warnings.filterwarnings('ignore')

# ============================================================================
# LOGGING SETUP
# ============================================================================
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

# ============================================================================
# MODEL DEFINITIONS – EXACT COPIES FROM MAIN CODE
# ============================================================================
class GradientReversal(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x, lambda_param):
        ctx.lambda_param = lambda_param
        return x.view_as(x)
    @staticmethod
    def backward(ctx, grad_output):
        return -ctx.lambda_param * grad_output, None

class DomainAdaptationLayer(nn.Module):
    def __init__(self, input_size, num_domains=3):
        super().__init__()
        self.domain_classifier = nn.Sequential(
            nn.Linear(input_size, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, num_domains),
            nn.LogSoftmax(dim=1)
        )
    def forward(self, features, lambda_param=0.1, reverse=False):
        if reverse:
            features = GradientReversal.apply(features, lambda_param)
        return self.domain_classifier(features)

class EWS_LSTM_Attention(nn.Module):
    """Bidirectional LSTM with Attention - from main code"""
    def __init__(self, input_size, hidden_size=128, num_layers=2, dropout=0.3):
        super().__init__()
        self.input_size = input_size
        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0,
            bidirectional=True
        )
        self.attention = nn.Sequential(
            nn.Linear(hidden_size * 2, 64),
            nn.Dropout(dropout),
            nn.Tanh(),
            nn.Linear(64, 1)
        )
        self.fc = nn.Sequential(
            nn.Linear(hidden_size * 2, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(128, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, 32),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.Linear(32, 1),
            nn.Sigmoid()
        )
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight, mode='fan_in', nonlinearity='relu')
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0.01)
            elif isinstance(m, nn.LSTM):
                for name, param in m.named_parameters():
                    if 'weight' in name:
                        nn.init.orthogonal_(param)
                    elif 'bias' in name:
                        nn.init.constant_(param, 0.0)

    def forward(self, x):
        lstm_out, _ = self.lstm(x)
        attention_weights = self.attention(lstm_out)
        attention_weights = attention_weights / torch.sqrt(torch.tensor(64.0, device=x.device))
        attention_weights = torch.softmax(attention_weights, dim=1)
        self.last_attention = attention_weights.detach().cpu()
        context = torch.sum(attention_weights * lstm_out, dim=1)
        return self.fc(context)

class EWS_LSTM_Attention_DA(nn.Module):
    """Bidirectional LSTM with Attention + Domain Adaptation"""
    def __init__(self, input_size, hidden_size=128, num_layers=2, dropout=0.3, use_da=True):
        super().__init__()
        self.use_da = use_da
        self.input_size = input_size
        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0,
            bidirectional=True
        )
        self.attention = nn.Sequential(
            nn.Linear(hidden_size * 2, 64),
            nn.Dropout(dropout),
            nn.Tanh(),
            nn.Linear(64, 1)
        )
        self.fc = nn.Sequential(
            nn.Linear(hidden_size * 2, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(128, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, 32),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.Linear(32, 1),
            nn.Sigmoid()
        )
        if self.use_da:
            self.domain_classifier = nn.Sequential(
                nn.Linear(hidden_size * 2, 64),
                nn.ReLU(),
                nn.Dropout(dropout),
                nn.Linear(64, 3)  # 3 domains for XJTU
            )
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight, mode='fan_in', nonlinearity='relu')
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0.01)
            elif isinstance(m, nn.LSTM):
                for name, param in m.named_parameters():
                    if 'weight' in name:
                        nn.init.orthogonal_(param)
                    elif 'bias' in name:
                        nn.init.constant_(param, 0.0)

    def forward(self, x, return_domain=False, lambda_param=0.1):
        lstm_out, _ = self.lstm(x)
        attention_weights = self.attention(lstm_out)
        attention_weights = attention_weights / torch.sqrt(torch.tensor(64.0, device=x.device))
        attention_weights = torch.softmax(attention_weights, dim=1)
        self.last_attention = attention_weights.detach().cpu()
        context = torch.sum(attention_weights * lstm_out, dim=1)

        domain_pred = None
        if self.use_da and return_domain:
            context_rev = GradientReversal.apply(context, lambda_param)
            domain_pred = self.domain_classifier(context_rev)

        rul_pred = self.fc(context)

        if return_domain:
            return rul_pred, domain_pred
        return rul_pred

class EWS_Bidirectional_LSTM_NoAttention(nn.Module):
    """Bidirectional LSTM without Attention"""
    def __init__(self, input_size, hidden_size=128, num_layers=2, dropout=0.3):
        super().__init__()
        self.input_size = input_size
        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0,
            bidirectional=True
        )
        self.fc = nn.Sequential(
            nn.Linear(hidden_size * 2, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(128, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, 32),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.Linear(32, 1),
            nn.Sigmoid()
        )
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight, mode='fan_in', nonlinearity='relu')
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0.01)
            elif isinstance(m, nn.LSTM):
                for name, param in m.named_parameters():
                    if 'weight' in name:
                        nn.init.orthogonal_(param)
                    elif 'bias' in name:
                        nn.init.constant_(param, 0.0)

    def forward(self, x):
        lstm_out, _ = self.lstm(x)
        context = torch.mean(lstm_out, dim=1)
        return self.fc(context)

class EWS_Unidirectional_LSTM_Attention(nn.Module):
    """Unidirectional LSTM with Attention"""
    def __init__(self, input_size, hidden_size=128, num_layers=2, dropout=0.3):
        super().__init__()
        self.input_size = input_size
        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0,
            bidirectional=False
        )
        self.attention = nn.Sequential(
            nn.Linear(hidden_size, 64),
            nn.Dropout(dropout),
            nn.Tanh(),
            nn.Linear(64, 1)
        )
        self.fc = nn.Sequential(
            nn.Linear(hidden_size, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(128, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, 32),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.Linear(32, 1),
            nn.Sigmoid()
        )
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight, mode='fan_in', nonlinearity='relu')
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0.01)
            elif isinstance(m, nn.LSTM):
                for name, param in m.named_parameters():
                    if 'weight' in name:
                        nn.init.orthogonal_(param)
                    elif 'bias' in name:
                        nn.init.constant_(param, 0.0)

    def forward(self, x):
        lstm_out, _ = self.lstm(x)
        attention_weights = self.attention(lstm_out)
        attention_weights = attention_weights / torch.sqrt(torch.tensor(64.0, device=x.device))
        attention_weights = torch.softmax(attention_weights, dim=1)
        self.last_attention = attention_weights.detach().cpu()
        context = torch.sum(attention_weights * lstm_out, dim=1)
        return self.fc(context)

class EWS_Unidirectional_LSTM_NoAttention(nn.Module):
    """Unidirectional LSTM without Attention"""
    def __init__(self, input_size, hidden_size=128, num_layers=2, dropout=0.3):
        super().__init__()
        self.input_size = input_size
        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0,
            bidirectional=False
        )
        self.fc = nn.Sequential(
            nn.Linear(hidden_size, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(128, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, 32),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.Linear(32, 1),
            nn.Sigmoid()
        )
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight, mode='fan_in', nonlinearity='relu')
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0.01)
            elif isinstance(m, nn.LSTM):
                for name, param in m.named_parameters():
                    if 'weight' in name:
                        nn.init.orthogonal_(param)
                    elif 'bias' in name:
                        nn.init.constant_(param, 0.0)

    def forward(self, x):
        lstm_out, _ = self.lstm(x)
        context = torch.mean(lstm_out, dim=1)
        return self.fc(context)

class Simple_LSTM(nn.Module):
    """Simple LSTM (Vanilla)"""
    def __init__(self, input_size, hidden_size=128, num_layers=2, dropout=0.3):
        super().__init__()
        self.input_size = input_size
        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0,
            bidirectional=False
        )
        self.fc = nn.Sequential(
            nn.Linear(hidden_size, 64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 1),
            nn.Sigmoid()
        )
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight, mode='fan_in', nonlinearity='relu')
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0.01)
            elif isinstance(m, nn.LSTM):
                for name, param in m.named_parameters():
                    if 'weight' in name:
                        nn.init.orthogonal_(param)
                    elif 'bias' in name:
                        nn.init.constant_(param, 0.0)

    def forward(self, x):
        lstm_out, (hidden, _) = self.lstm(x)
        last_hidden = hidden[-1]
        return self.fc(last_hidden)

# ============================================================================
# MODEL FACTORY
# ============================================================================
def get_model(model_type, input_size, hidden_size=128, num_layers=2, dropout=0.3):
    models = {
        'lstm_da': EWS_LSTM_Attention_DA,
        'bilstm_attn': EWS_LSTM_Attention,
        'bilstm_noattn': EWS_Bidirectional_LSTM_NoAttention,
        'lstm_attn': EWS_Unidirectional_LSTM_Attention,
        'lstm_noattn': EWS_Unidirectional_LSTM_NoAttention,
        'simple_lstm': Simple_LSTM,
    }
    if model_type not in models:
        raise ValueError(f"Unknown model type: {model_type}")
    model_class = models[model_type]
    # For lstm_da we need to pass use_da=True
    if model_type == 'lstm_da':
        return model_class(input_size=input_size, hidden_size=hidden_size,
                           num_layers=num_layers, dropout=dropout, use_da=True)
    else:
        return model_class(input_size=input_size, hidden_size=hidden_size,
                           num_layers=num_layers, dropout=dropout)

# ============================================================================
# DATASET CLASS FOR CACHED DATA
# ============================================================================
class SimpleDataset(Dataset):
    def __init__(self, features, labels, bearing_ids, max_ruls):
        self.features = torch.FloatTensor(features)
        self.labels = torch.FloatTensor(labels)
        self.bearing_ids = bearing_ids
        self.max_ruls = torch.FloatTensor(max_ruls)

    def __len__(self):
        return len(self.features)

    def __getitem__(self, idx):
        return (self.features[idx],
                self.labels[idx].unsqueeze(0),
                self.bearing_ids[idx],
                self.max_ruls[idx])

# ============================================================================
# TRAIN/VAL SPLIT FUNCTION (from main code)
# ============================================================================
def create_balanced_train_val_split(dataset, val_split=0.2, random_seed=42):
    bearing_to_indices = defaultdict(list)
    for idx in range(len(dataset)):
        bearing_to_indices[dataset.bearing_ids[idx]].append(idx)

    train_indices = []
    val_indices = []

    for bearing_id, indices in bearing_to_indices.items():
        num_samples = len(indices)
        num_val = max(1, int(num_samples * val_split))
        # Keep natural order – indices are already chronological
        val_indices.extend(indices[-num_val:])          # last windows for validation
        train_indices.extend(indices[:-num_val])        # first windows for training

    # Shuffle combined lists for minibatch randomness
    rng = np.random.RandomState(random_seed)
    rng.shuffle(train_indices)
    rng.shuffle(val_indices)

    logger.info(f"Time‑based split: {len(train_indices)} training, {len(val_indices)} validation")
    return train_indices, val_indices

# ============================================================================
# TRAINING FUNCTION – EXACT COPY FROM MAIN CODE
# ============================================================================
def train_model_with_validation(model, train_loader, val_loader, epochs=100, lr=0.001,
                                device='cuda', early_stopping_patience=20, use_da=False,
                                lambda_da=0.1):
    """Universal training function for ALL models with proper validation - FIXED"""
    model.to(device)
    
    # ==================== FIXED OPTIMIZER SECTION ====================
    if use_da and hasattr(model, 'domain_classifier'):
        # Separate parameters: domain_classifier gets lower LR
        base_params = []
        domain_params = []
        for name, param in model.named_parameters():
            if 'domain_classifier' in name:
                domain_params.append(param)
            else:
                base_params.append(param)
        
        optimizer = optim.AdamW([
            {'params': base_params, 'lr': lr, 'weight_decay': 1e-4},
            {'params': domain_params, 'lr': lr * 0.1, 'weight_decay': 1e-4}  # lower LR for domain
        ])
    else:
        optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    # =================================================================

    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.5, patience=10
    )
    
    criterion_rul = nn.MSELoss()
    if use_da:
        criterion_domain = nn.NLLLoss()  # For LogSoftmax output
    
    best_val_loss = float('inf')
    patience_counter = 0
    train_losses, val_losses = [], []
    domain_losses = [] if use_da else None

    logger.info(f"Starting training on {device}, epochs={epochs}, DA={use_da}, λ_da={lambda_da}")
    
    for epoch in range(epochs):
        model.train()
        train_rul_loss, train_domain_loss = 0.0, 0.0
        train_batches = 0
        
        for batch_idx, (data, targets, bearing_ids, _) in enumerate(train_loader):
            data, targets = data.to(device), targets.to(device)
            
            # Domain labels
            if use_da and hasattr(model, 'domain_classifier'):
                domain_labels = []
                for bid in bearing_ids:
                    if 'Bearing1_' in bid:
                        domain_labels.append(0)
                    elif 'Bearing2_' in bid:
                        domain_labels.append(1)
                    elif 'Bearing3_' in bid:
                        domain_labels.append(2)
                    else:
                        domain_labels.append(0)
                domain_labels = torch.tensor(domain_labels, device=device)
            
            # Zero gradients (single optimizer)
            optimizer.zero_grad()
            
            # Forward pass
            if use_da and hasattr(model, 'domain_classifier'):
                rul_pred, domain_pred = model(data, return_domain=True, lambda_param=lambda_da)
                rul_loss = criterion_rul(rul_pred, targets)
                domain_loss = criterion_domain(domain_pred, domain_labels)
                total_loss = rul_loss + lambda_da * domain_loss
            else:
                rul_pred = model(data)
                rul_loss = criterion_rul(rul_pred, targets)
                total_loss = rul_loss
                domain_loss = torch.tensor(0.0)
            
            # Backward pass
            total_loss.backward()
            
            # Gradient clipping for stability
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            
            # Single optimizer step
            optimizer.step()
            
            train_rul_loss += rul_loss.item()
            if use_da:
                train_domain_loss += domain_loss.item()
            train_batches += 1
        
        # Validation (unchanged)
        model.eval()
        val_loss = 0.0
        val_batches = 0
        with torch.no_grad():
            for data, targets, _, _ in val_loader:
                data, targets = data.to(device), targets.to(device)
                rul_pred = model(data)
                if isinstance(rul_pred, tuple):
                    rul_pred = rul_pred[0]
                val_loss += criterion_rul(rul_pred, targets).item()
                val_batches += 1
        
        avg_train_loss = train_rul_loss / train_batches
        avg_val_loss = val_loss / val_batches
        train_losses.append(avg_train_loss)
        val_losses.append(avg_val_loss)
        
        if use_da:
            avg_domain_loss = train_domain_loss / train_batches
            domain_losses.append(avg_domain_loss)
        
        scheduler.step(avg_val_loss)
        
        # Early stopping (unchanged)
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            best_state = model.state_dict().copy()
            patience_counter = 0
            torch.save({...}, f"/kaggle/working/best_model_epoch_{epoch}.pth")
        else:
            patience_counter += 1
        
        # Logging (unchanged)
        if (epoch + 1) % 5 == 0 or epoch == 0:
            da_info = f", Domain Loss: {avg_domain_loss:.6f}" if use_da else ""
            logger.info(...)
        
        if patience_counter >= early_stopping_patience:
            logger.info(f"Early stopping at epoch {epoch + 1}")
            break
    
    # Load best model (unchanged)
    if 'best_state' in locals():
        model.load_state_dict(best_state)
        logger.info(f"Loaded best model with val loss: {best_val_loss:.6f}")
    
    return model, train_losses, val_losses, domain_losses

# ============================================================================
# METRICS FUNCTION (from main code)
# ============================================================================
def calculate_comprehensive_metrics(preds, targets):
    preds = np.array(preds).flatten()
    targets = np.array(targets).flatten()

    metrics = {
        'MAE': np.mean(np.abs(preds - targets)),
        'RMSE': np.sqrt(np.mean((preds - targets) ** 2)),
        'R2': 1 - np.sum((preds - targets) ** 2) / (np.sum((targets - np.mean(targets)) ** 2) + 1e-10),
    }

    # Within 10% accuracy (only for positive targets)
    mask = targets > 0
    if np.sum(mask) > 0:
        rel_errors = np.abs(preds[mask] - targets[mask]) / targets[mask]
        metrics['Within_10%'] = np.mean(rel_errors <= 0.1) * 100
    else:
        metrics['Within_10%'] = 0.0

    return metrics

# ============================================================================
# MAIN VALIDATION FUNCTION
# ============================================================================
def run_robust_statistical_validation(train_cache_path, test_cache_path,
                                      num_seeds=15, model_type='Simple_LSTM',
                                      master_seed=42):
    """
    Run statistical validation with independent seeds.

    Args:
        train_cache_path: full path to the training cache .pkl file
        test_cache_path:  full path to the test cache .pkl file
        num_seeds: number of independent seeds to run
        model_type: string identifying the model architecture
        master_seed: master seed for generating independent seeds
    """
    print("="*80)
    print("ROBUST STATISTICAL VALIDATION FOR PHD")
    print("="*80)

    # 1. Generate independent seeds
    print(f"\n🌱 Generating {num_seeds} independent random seeds...")
    seed_generator = np.random.RandomState(master_seed)
    independent_seeds = seed_generator.randint(0, 2**31-1, size=num_seeds)
    print(f"First 5 seeds: {independent_seeds[:5]}")

    # 2. Load cache files
    print("\n📊 Loading cache data...")
    try:
        with open(train_cache_path, 'rb') as f:
            train_cache = pickle.load(f)
        with open(test_cache_path, 'rb') as f:
            test_cache = pickle.load(f)
    except Exception as e:
        print(f"❌ Error loading cache files: {e}")
        return None

    # Verify cache structure
    required_keys = ['ews_sequences', 'rul_targets', 'bearing_ids', 'max_ruls']
    for key in required_keys:
        if key not in train_cache or key not in test_cache:
            print(f"❌ Cache missing key: {key}")
            return None

    # Convert to numpy arrays
    train_features = np.array(train_cache['ews_sequences'], dtype=np.float32)
    train_labels = np.array(train_cache['rul_targets'], dtype=np.float32)
    train_bearing_ids = train_cache['bearing_ids']
    train_max_ruls = np.array(train_cache['max_ruls'], dtype=np.float32)

    test_features = np.array(test_cache['ews_sequences'], dtype=np.float32)
    test_labels = np.array(test_cache['rul_targets'], dtype=np.float32)
    test_bearing_ids = test_cache['bearing_ids']
    test_max_ruls = np.array(test_cache['max_ruls'], dtype=np.float32)

    # Create full datasets
    full_train_dataset = SimpleDataset(train_features, train_labels,
                                       train_bearing_ids, train_max_ruls)
    test_dataset = SimpleDataset(test_features, test_labels,
                                 test_bearing_ids, test_max_ruls)

    print(f"✓ Train samples: {len(full_train_dataset)}")
    print(f"✓ Test samples: {len(test_dataset)}")
    print(f"✓ Feature dimension: {train_features.shape[2]}")

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"✓ Using device: {device}")

    all_results = []

    for seed_idx, cur_seed in enumerate(independent_seeds):
        print(f"\n--- Seed {seed_idx+1}/{num_seeds} (Seed={cur_seed}) ---")

        # Set all random seeds
        torch.manual_seed(cur_seed)
        np.random.seed(cur_seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed(cur_seed)
            torch.cuda.manual_seed_all(cur_seed)
            torch.backends.cudnn.deterministic = True
            torch.backends.cudnn.benchmark = False

        # Create train/val split (80/20) – chronological per bearing
        train_indices, val_indices = create_balanced_train_val_split(
            full_train_dataset, val_split=0.2, random_seed=cur_seed
        )

        train_subset = Subset(full_train_dataset, train_indices)
        val_subset = Subset(full_train_dataset, val_indices)

        print(f"  Train: {len(train_subset)}, Val: {len(val_subset)}")

        # Create model using factory
        model = get_model(model_type, input_size=train_features.shape[2])

        # Data loaders
        batch_size = 32
        train_loader = DataLoader(train_subset, batch_size=batch_size, shuffle=True)
        val_loader = DataLoader(val_subset, batch_size=batch_size, shuffle=False)
        test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

        # Train
        use_da = (model_type == 'lstm_da')
        model, _, _, _ = train_model_with_validation(
            model, train_loader, val_loader,
            epochs=100, lr=0.001, device=device,
            early_stopping_patience=20,
            use_da=use_da,
            lambda_da=0.1
        )

        # Evaluate on test set
        model.eval()
        predictions, targets = [], []
        with torch.no_grad():
            for data, target, _, max_rul in test_loader:
                data = data.to(device)
                pred = model(data).cpu().numpy().flatten()
                targ = target.numpy().flatten()
                max_rul_np = max_rul.numpy().flatten()
                for i in range(len(pred)):
                    pred_actual = pred[i] * max_rul_np[i]
                    targ_actual = targ[i] * max_rul_np[i]
                    predictions.append(pred_actual)
                    targets.append(targ_actual)

        predictions = np.array(predictions)
        targets = np.array(targets)

        metrics = calculate_comprehensive_metrics(predictions, targets)

        all_results.append({
            'seed': cur_seed,
            'MAE': metrics['MAE'],
            'RMSE': metrics['RMSE'],
            'R2': metrics['R2'],
            'Within_10%': metrics['Within_10%']
        })

        print(f"  MAE: {metrics['MAE']:.2f}, R²: {metrics['R2']:.3f}, Within 10%: {metrics['Within_10%']:.1f}%")

    # 6. Statistical summary
    print(f"\n{'='*80}")
    print("📊 STATISTICAL ANALYSIS")
    print(f"{'='*80}")

    results_df = pd.DataFrame(all_results)
    for metric in ['MAE', 'R2', 'Within_10%']:
        values = results_df[metric].values
        mean_val = np.mean(values)
        std_val = np.std(values)
        sem_val = std_val / np.sqrt(len(values))
        if len(values) > 1:
            ci_low, ci_high = stats.t.interval(0.95, len(values)-1,
                                               loc=mean_val, scale=sem_val)
        else:
            ci_low, ci_high = mean_val, mean_val
        print(f"\n{metric}:")
        print(f"  Mean ± SD: {mean_val:.4f} ± {std_val:.4f}")
        print(f"  95% CI: [{ci_low:.4f}, {ci_high:.4f}]")
        print(f"  Range: [{np.min(values):.4f}, {np.max(values):.4f}]")

    # 7. Save results
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    results_dir = "/kaggle/working/statistical_results"
    os.makedirs(results_dir, exist_ok=True)
    results_file = os.path.join(results_dir,
                                f"results_{model_type}_{num_seeds}seeds_{timestamp}.csv")
    results_df.to_csv(results_file, index=False)
    print(f"\n✅ Results saved to: {results_file}")

    return results_df

# ============================================================================
# MAIN EXECUTION
# ============================================================================
if __name__ == "__main__":
    print("ROBUST STATISTICAL VALIDATION FOR PHD")
    print("Please provide the paths to your train and test cache files.\n")

    # Example – replace with actual paths
    train_cache_path = "/kaggle/working/cache/dataset_40f573235fffe95ca598b22294a9de96.pkl"
    test_cache_path  = "/kaggle/working/cache/dataset_58419fc36b975a6c18740516d1c6da0d.pkl"

    # Uncomment for interactive input:
    # train_cache_path = input("Train cache path: ").strip()
    # test_cache_path  = input("Test cache path: ").strip()

    if not os.path.exists(train_cache_path) or not os.path.exists(test_cache_path):
        print("❌ Cache files not found. Please update the paths.")
        print("   After running your main experiment, look in /kaggle/working/cache/")
        print("   and identify the two largest .pkl files (train and test).")
        exit(1)

    num_seeds_input = input("Number of seeds (default=15): ").strip()
    num_seeds = int(num_seeds_input) if num_seeds_input.isdigit() else 15

    model_type_input = input("Model type (simple_lstm / lstm_attn / bilstm_attn / lstm_da / ...) [default=simple_lstm]: ").strip()
    if not model_type_input:
        model_type_input = 'simple_lstm'

    results = run_robust_statistical_validation(
        train_cache_path=train_cache_path,
        test_cache_path=test_cache_path,
        num_seeds=num_seeds,
        model_type=model_type_input,
        master_seed=42
    )

    if results is not None:
        print("\n" + "="*80)
        print("✅ VALIDATION COMPLETE – SCIENTIFICALLY VALID")
        print("="*80)

 #  Bayesian validation

In [ ]:
# ============================================================================
# BAYESIAN VALIDATION OF MODEL PERFORMANCE (FIXED)
# ============================================================================
import numpy as np
import pandas as pd
import pymc as pm
import arviz as az
import matplotlib.pyplot as plt
import os

# ----------------------------------------------------------------------------
# CONFIGURATION - adjust these paths and names
# ----------------------------------------------------------------------------
# Path to the results CSV from your validation script
results_file = "/kaggle/working/statistical_results/results_bilstm_attn_15seeds_20260227_174615.csv"  # CHANGE to your actual file

# If you also have results for the no-attention model, you can compare
results_noattn_file = ""  # Optional, e.g. "/kaggle/working/statistical_results/results_noattn_15seeds.csv"

# Metrics to analyze
metrics = ['MAE', 'R2', 'Within_10%']

# ----------------------------------------------------------------------------
# Load data
# ----------------------------------------------------------------------------
df = pd.read_csv(results_file)
print(f"Loaded {len(df)} runs from {results_file}")
print(df[metrics].describe())

# ----------------------------------------------------------------------------
# Bayesian model for a single metric
# ----------------------------------------------------------------------------
def bayesian_estimate(data, metric_name):
    """Estimate posterior of mean and std for a metric using a Normal model."""
    with pm.Model() as model:
        # Prior for mean: normal with wide scale
        mu = pm.Normal(f'mu_{metric_name}', mu=data.mean(), sigma=data.std()*10)
        # Prior for standard deviation: half-normal
        sigma = pm.HalfNormal(f'sigma_{metric_name}', sigma=data.std()*5)
        # Likelihood
        obs = pm.Normal(f'obs_{metric_name}', mu=mu, sigma=sigma, observed=data)
        # Sample
        trace = pm.sample(2000, tune=1000, cores=2, random_seed=42, progressbar=True)
    return trace

# ----------------------------------------------------------------------------
# Run Bayesian inference for each metric
# ----------------------------------------------------------------------------
print("\n" + "="*60)
print("BAYESIAN POSTERIOR ESTIMATES")
print("="*60)

traces = {}
for metric in metrics:
    print(f"\n--- {metric} ---")
    data = df[metric].values
    trace = bayesian_estimate(data, metric)
    traces[metric] = trace
    
    # Summarise
    summary = az.summary(trace, hdi_prob=0.95, var_names=[f'mu_{metric}', f'sigma_{metric}'])
    print(summary)
    
    # Plot posterior (FIXED: set title after plotting)
    ax = az.plot_posterior(trace, var_names=[f'mu_{metric}'], figsize=(8,3), textsize=10)
    ax.set_title(f'Posterior of mean {metric}')
    plt.tight_layout()
    plt.show()

# ----------------------------------------------------------------------------
# If you have a second model, compute probability that attention is better
# ----------------------------------------------------------------------------
if results_noattn_file and os.path.exists(results_noattn_file):
    df_noattn = pd.read_csv(results_noattn_file)
    print("\n" + "="*60)
    print("COMPARISON: Attention vs No-Attention")
    print("="*60)
    
    for metric in metrics:
        data_attn = df[metric].values
        data_noattn = df_noattn[metric].values
        
        # Bayesian estimate of the difference
        with pm.Model() as model_compare:
            mu_attn = pm.Normal('mu_attn', mu=data_attn.mean(), sigma=data_attn.std()*10)
            mu_noattn = pm.Normal('mu_noattn', mu=data_noattn.mean(), sigma=data_noattn.std()*10)
            sigma_attn = pm.HalfNormal('sigma_attn', sigma=data_attn.std()*5)
            sigma_noattn = pm.HalfNormal('sigma_noattn', sigma=data_noattn.std()*5)
            
            obs_attn = pm.Normal('obs_attn', mu=mu_attn, sigma=sigma_attn, observed=data_attn)
            obs_noattn = pm.Normal('obs_noattn', mu=mu_noattn, sigma=sigma_noattn, observed=data_noattn)
            
            diff = pm.Deterministic('diff', mu_attn - mu_noattn)
            
            trace_comp = pm.sample(2000, tune=1000, cores=2, random_seed=42, progressbar=True)
        
        # Probability that attention is better (for MAE lower is better, for R2/Within higher is better)
        diff_samples = trace_comp.posterior['diff'].values.flatten()
        if metric == 'MAE':
            prob_better = np.mean(diff_samples < 0)   # attention has lower MAE
        else:
            prob_better = np.mean(diff_samples > 0)   # attention has higher R2 or Within_10%
        
        print(f"\n{metric}:")
        print(f"  Mean difference (attention - noattn) = {np.mean(diff_samples):.3f}")
        print(f"  95% HDI: {az.hdi(diff_samples, hdi_prob=0.95)}")
        print(f"  Probability attention is better = {prob_better:.3f}")
        
        # Plot the difference (FIXED)
        ax = az.plot_posterior(diff_samples, figsize=(8,3), textsize=10)
        ax.set_title(f'Difference (attention - noattn) for {metric}')
        ax.axvline(0, color='red', linestyle='--')
        plt.tight_layout()
        plt.show()
else:
    print("\nNo no-attention results file found – skipping comparison.")

print("\n✅ Bayesian validation complete.")

# detailed diagnostic report: best/worst seeds

In [ ]:
#========================================================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Load the results
results_df = pd.read_csv('/kaggle/working/statistical_results/results_bilstm_attn_15seeds_20260227_174615.csv')

# Check column names
print("📊 Available columns:")
for col in results_df.columns:
    print(f"  - {col}")

# Find the actual seed column name
seed_col = None
for col in results_df.columns:
    if 'seed' in col.lower():
        seed_col = col
        print(f"\n✅ Found seed column: '{seed_col}'")
        break

if seed_col is None:
    # If no seed column found, create one
    results_df['seed_num'] = range(1, len(results_df) + 1)
    seed_col = 'seed_num'
    print(f"\n⚠️  No seed column found. Created '{seed_col}' column.")

# Find the "Within 10%" column name (it might have underscore or space)
within_10_col = None
for col in results_df.columns:
    if '10' in col or 'within' in col.lower():
        within_10_col = col
        print(f"✅ Found 'Within 10%' column: '{within_10_col}'")
        break

# Find other key columns
mae_col = None
r2_col = None

for col in results_df.columns:
    if col.upper() == 'MAE':
        mae_col = col
    elif col.upper() == 'R2' or col == 'R²':
        r2_col = col

if mae_col is None:
    for col in results_df.columns:
        if 'mae' in col.lower():
            mae_col = col
            break

if r2_col is None:
    for col in results_df.columns:
        if 'r2' in col.lower() or 'r²' in col.lower():
            r2_col = col
            break

print(f"\n📈 Key metric columns:")
print(f"  MAE column: '{mae_col}'")
print(f"  R² column: '{r2_col}'")
print(f"  Within 10% column: '{within_10_col}'")

# Analyze best/worst performing seeds
print(f"\n{'='*60}")
print("TOP 5 BEST PERFORMING SEEDS (Lowest MAE):")
print('='*60)

# Sort by MAE and get top 5
results_df_sorted = results_df.sort_values(by=mae_col)
top_seeds = results_df_sorted.head(5)

for idx, row in top_seeds.iterrows():
    print(f"Seed {row.get(seed_col, idx+1)}:")
    print(f"  MAE: {row[mae_col]:.2f} minutes")
    print(f"  R²: {row[r2_col]:.3f}")
    if within_10_col:
        print(f"  Within 10%: {row[within_10_col]:.1f}%")
    print()

print(f"\n{'='*60}")
print("BOTTOM 5 WORST PERFORMING SEEDS (Highest MAE):")
print('='*60)

bottom_seeds = results_df_sorted.tail(5)

for idx, row in bottom_seeds.iterrows():
    print(f"Seed {row.get(seed_col, idx+1)}:")
    print(f"  MAE: {row[mae_col]:.2f} minutes")
    print(f"  R²: {row[r2_col]:.3f}")
    if within_10_col:
        print(f"  Within 10%: {row[within_10_col]:.1f}%")
    print()

# Statistical analysis
print(f"\n{'='*60}")
print("STATISTICAL ANALYSIS:")
print('='*60)

print(f"Total seeds: {len(results_df)}")
print(f"MAE Mean ± Std: {results_df[mae_col].mean():.2f} ± {results_df[mae_col].std():.2f}")
print(f"R² Mean ± Std: {results_df[r2_col].mean():.3f} ± {results_df[r2_col].std():.3f}")

if within_10_col:
    print(f"Within 10% Mean ± Std: {results_df[within_10_col].mean():.1f}% ± {results_df[within_10_col].std():.1f}%")

# Calculate stability metrics
mae_cv = (results_df[mae_col].std() / results_df[mae_col].mean()) * 100
r2_cv = (results_df[r2_col].std() / abs(results_df[r2_col].mean())) * 100 if results_df[r2_col].mean() != 0 else 0

print(f"\nCoefficient of Variation (lower = more stable):")
print(f"  MAE CV: {mae_cv:.1f}%")
print(f"  R² CV: {r2_cv:.1f}%")

# Correlation analysis
print(f"\n{'='*60}")
print("CORRELATION ANALYSIS:")
print('='*60)

corr_cols = [mae_col, r2_col]
if within_10_col:
    corr_cols.append(within_10_col)

correlation_matrix = results_df[corr_cols].corr()
print("Correlation Matrix:")
print(correlation_matrix)

# Visualizations
print(f"\n{'='*60}")
print("CREATING VISUALIZATIONS...")
print('='*60)

# Create a figure with multiple subplots
fig, axes = plt.subplots(2, 3, figsize=(18, 12))

# 1. MAE distribution
ax1 = axes[0, 0]
ax1.hist(results_df[mae_col], bins=10, alpha=0.7, edgecolor='black', color='skyblue')
ax1.axvline(results_df[mae_col].mean(), color='red', linestyle='--', 
            linewidth=2, label=f'Mean: {results_df[mae_col].mean():.2f}')
ax1.set_xlabel('MAE (minutes)')
ax1.set_ylabel('Frequency')
ax1.set_title(f'MAE Distribution (n={len(results_df)})')
ax1.legend()
ax1.grid(True, alpha=0.3)

# 2. R² distribution
ax2 = axes[0, 1]
ax2.hist(results_df[r2_col], bins=10, alpha=0.7, edgecolor='black', color='lightgreen')
ax2.axvline(results_df[r2_col].mean(), color='red', linestyle='--',
            linewidth=2, label=f'Mean: {results_df[r2_col].mean():.3f}')
ax2.set_xlabel('R²')
ax2.set_ylabel('Frequency')
ax2.set_title(f'R² Distribution (n={len(results_df)})')
ax2.legend()
ax2.grid(True, alpha=0.3)

# 3. MAE vs R² scatter plot
ax3 = axes[0, 2]
scatter = ax3.scatter(results_df[mae_col], results_df[r2_col], alpha=0.6, s=100)
ax3.set_xlabel('MAE (minutes)')
ax3.set_ylabel('R²')
ax3.set_title('MAE vs R² Trade-off')
ax3.grid(True, alpha=0.3)

# Add trend line
z = np.polyfit(results_df[mae_col], results_df[r2_col], 1)
p = np.poly1d(z)
ax3.plot(results_df[mae_col].sort_values(), p(results_df[mae_col].sort_values()), 
         "r--", alpha=0.8, label=f'Trend: R² = {z[0]:.3f}*MAE + {z[1]:.3f}')
ax3.legend()

# 4. Seed performance over time
ax4 = axes[1, 0]
seed_numbers = range(1, len(results_df) + 1)
ax4.plot(seed_numbers, results_df[mae_col], 'o-', linewidth=1.5, markersize=6,
         label='MAE per seed')
ax4.axhline(results_df[mae_col].mean(), color='red', linestyle='--',
            alpha=0.7, label='Mean MAE')
ax4.set_xlabel('Seed Number')
ax4.set_ylabel('MAE (minutes)')
ax4.set_title('MAE Across Different Random Seeds')
ax4.legend()
ax4.grid(True, alpha=0.3)

# 5. Box plot of key metrics
ax5 = axes[1, 1]
data_to_plot = [results_df[mae_col], results_df[r2_col]]
labels = ['MAE', 'R²']
if within_10_col:
    data_to_plot.append(results_df[within_10_col])
    labels.append('Within 10%')

box = ax5.boxplot(data_to_plot, patch_artist=True)
colors = ['lightblue', 'lightgreen', 'lightcoral'][:len(data_to_plot)]
for patch, color in zip(box['boxes'], colors):
    patch.set_facecolor(color)

ax5.set_xticklabels(labels)
ax5.set_ylabel('Value')
ax5.set_title('Distribution of Key Metrics')
ax5.grid(True, alpha=0.3, axis='y')

# 6. Summary statistics table
ax6 = axes[1, 2]
ax6.axis('off')

summary_text = (
    f"Statistical Validation Summary\n"
    f"Model: Simple LSTM\n"
    f"Seeds: {len(results_df)}\n"
    f"Split: Cross-condition 2\n\n"
    f"MAE: {results_df[mae_col].mean():.2f} ± {results_df[mae_col].std():.2f}\n"
    f"     Range: [{results_df[mae_col].min():.2f}, {results_df[mae_col].max():.2f}]\n\n"
    f"R²: {results_df[r2_col].mean():.3f} ± {results_df[r2_col].std():.3f}\n"
    f"     Range: [{results_df[r2_col].min():.3f}, {results_df[r2_col].max():.3f}]\n"
)

if within_10_col:
    summary_text += (
        f"\nWithin 10%: {results_df[within_10_col].mean():.1f}% ± "
        f"{results_df[within_10_col].std():.1f}%\n"
        f"     Range: [{results_df[within_10_col].min():.1f}%, "
        f"{results_df[within_10_col].max():.1f}%]"
    )

ax6.text(0.05, 0.95, summary_text, fontsize=10, verticalalignment='top',
         fontfamily='monospace', bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

plt.suptitle('Statistical Validation Analysis - Simplebest LSTM Model', 
             fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('/kaggle/working/statistical_analysis_detailed.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\n✅ Analysis complete!")
print(f"📊 Summary statistics displayed above")
print(f"📈 Visualization saved to: /kaggle/working/statistical_analysis_detailed.png")

# Additional insights
print(f"\n{'='*60}")
print("KEY INSIGHTS:")
print('='*60)

# Identify stable vs unstable seeds
mae_threshold = results_df[mae_col].mean() + results_df[mae_col].std()
stable_seeds = results_df[results_df[mae_col] <= mae_threshold]
unstable_seeds = results_df[results_df[mae_col] > mae_threshold]

print(f"Stable seeds (MAE ≤ mean + std): {len(stable_seeds)} seeds")
print(f"Unstable seeds (MAE > mean + std): {len(unstable_seeds)} seeds")

if len(unstable_seeds) > 0:
    print("\nUnstable seeds (investigate these):")
    for idx, row in unstable_seeds.iterrows():
        print(f"  Seed {row.get(seed_col, idx+1)}: MAE={row[mae_col]:.2f}, R²={row[r2_col]:.3f}")

# Check for any patterns
print(f"\nPerformance Patterns:")
if (results_df[mae_col].corr(pd.Series(range(len(results_df)))) > 0.5 or 
    results_df[mae_col].corr(pd.Series(range(len(results_df)))) < -0.5):
    print("  ⚠️  Potential trend: MAE shows pattern with seed number")
else:
    print("  ✓ No clear trend: MAE is independent of seed number")

# Calculate improvement potential
best_mae = results_df[mae_col].min()
worst_mae = results_df[mae_col].max()
improvement_potential = ((worst_mae - best_mae) / worst_mae) * 100

print(f"\nImprovement Potential:")
print(f"  Best MAE: {best_mae:.2f} minutes")
print(f"  Worst MAE: {worst_mae:.2f} minutes")
print(f"  Potential improvement: {improvement_potential:.1f}%")

# Healthindicators witha ttention comparision

In [ ]:
# ============================================================================
# CORRECTED ATTENTION ANALYSIS – PHYSICS ALIGNMENT VALIDATION
# ============================================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import pickle
import os
import json
from pathlib import Path
import logging
import warnings
warnings.filterwarnings('ignore')

# ----------------------------------------------------------------------------
# Logging setup
# ----------------------------------------------------------------------------
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

# ----------------------------------------------------------------------------
# Model definitions – EXACT copies from your training code
# ----------------------------------------------------------------------------
class GradientReversal(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x, lambda_param):
        ctx.lambda_param = lambda_param
        return x.view_as(x)
    @staticmethod
    def backward(ctx, grad_output):
        return -ctx.lambda_param * grad_output, None

class DomainAdaptationLayer(nn.Module):
    def __init__(self, input_size, num_domains=3):
        super().__init__()
        self.domain_classifier = nn.Sequential(
            nn.Linear(input_size, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, num_domains),
            nn.LogSoftmax(dim=1)
        )
    def forward(self, features, lambda_param=0.1, reverse=False):
        if reverse:
            features = GradientReversal.apply(features, lambda_param)
        return self.domain_classifier(features)
        
class EWS_Bidirectional_LSTM_Attention(nn.Module):
    """Bidirectional LSTM with Attention - from main code"""
    def __init__(self, input_size, hidden_size=128, num_layers=2, dropout=0.3):
        super().__init__()
        self.input_size = input_size
        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0,
            bidirectional=True
        )
        self.attention = nn.Sequential(
            nn.Linear(hidden_size * 2, 64),
            nn.Tanh(),
            nn.Linear(64, 1)
        )
        self.fc = nn.Sequential(
            nn.Linear(hidden_size * 2, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(128, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, 32),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.Linear(32, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        lstm_out, _ = self.lstm(x)
        attention_weights = self.attention(lstm_out)
        attention_weights = attention_weights / torch.sqrt(torch.tensor(64.0, device=x.device)) 
        attention_weights = torch.softmax(attention_weights, dim=1)
        self.last_attention = attention_weights.detach().cpu()
        context = torch.sum(attention_weights * lstm_out, dim=1)
        return self.fc(context)

# (If you need other model classes like EWS_LSTM_Attention_DA, add them here.
#  For this analysis we assume the model is the bidirectional attention variant.)
class EWS_LSTM_Attention_DA(nn.Module):
    """Bidirectional LSTM with Attention + Domain Adaptation"""
    def __init__(self, input_size, hidden_size=128, num_layers=2, dropout=0.3, use_da=True):
        super().__init__()
        self.use_da = use_da
        self.input_size = input_size
        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0,
            bidirectional=True
        )
        self.attention = nn.Sequential(
            nn.Linear(hidden_size * 2, 64),
            nn.Dropout(dropout),
            nn.Tanh(),
            nn.Linear(64, 1)
        )
        self.fc = nn.Sequential(
            nn.Linear(hidden_size * 2, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(128, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, 32),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.Linear(32, 1),
            nn.Sigmoid()
        )
        if self.use_da:
            # Include both domain_layer and domain_classifier to match the saved checkpoint
            self.domain_layer = DomainAdaptationLayer(hidden_size * 2)
            self.domain_classifier = nn.Sequential(
                nn.Linear(hidden_size * 2, 64),
                nn.ReLU(),
                nn.Dropout(dropout),
                nn.Linear(64, 3)
            )
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight, mode='fan_in', nonlinearity='relu')
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0.01)
            elif isinstance(m, nn.LSTM):
                for name, param in m.named_parameters():
                    if 'weight' in name:
                        nn.init.orthogonal_(param)
                    elif 'bias' in name:
                        nn.init.constant_(param, 0.0)

    def forward(self, x, return_domain=False, lambda_param=0.1):
        lstm_out, _ = self.lstm(x)
        attention_weights = self.attention(lstm_out)
        attention_weights = attention_weights / torch.sqrt(torch.tensor(64.0, device=x.device))
        attention_weights = torch.softmax(attention_weights, dim=1)
        self.last_attention = attention_weights.detach().cpu()
        context = torch.sum(attention_weights * lstm_out, dim=1)

        domain_pred = None
        if self.use_da and return_domain:
            context_rev = GradientReversal.apply(context, lambda_param)
            domain_pred = self.domain_classifier(context_rev)

        rul_pred = self.fc(context)
        if return_domain:
            return rul_pred, domain_pred
        return rul_pred
class Simple_LSTM(nn.Module):
    """Simple LSTM (Vanilla)"""
    def __init__(self, input_size, hidden_size=128, num_layers=2, dropout=0.3):
        super().__init__()
        self.input_size = input_size
        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0,
            bidirectional=False
        )
        self.fc = nn.Sequential(
            nn.Linear(hidden_size, 64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 1),
            nn.Sigmoid()
        )
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight, mode='fan_in', nonlinearity='relu')
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0.01)
            elif isinstance(m, nn.LSTM):
                for name, param in m.named_parameters():
                    if 'weight' in name:
                        nn.init.orthogonal_(param)
                    elif 'bias' in name:
                        nn.init.constant_(param, 0.0)

    def forward(self, x):
        lstm_out, (hidden, _) = self.lstm(x)
        last_hidden = hidden[-1]
        return self.fc(last_hidden)
# ----------------------------------------------------------------------------
# Dataset class to load cached data (same as in training)
# ----------------------------------------------------------------------------
class CachedDataset(Dataset):
    def __init__(self, cache_path, bearing_list=None):
        """
        Load data from a cache .pkl file.
        If bearing_list is provided, filter to those bearings.
        """
        with open(cache_path, 'rb') as f:
            cache = pickle.load(f)

        required_keys = ['ews_sequences', 'rul_targets', 'bearing_ids', 'max_ruls']
        for key in required_keys:
            if key not in cache:
                raise KeyError(f"Cache missing required key: {key}")

        self.sequences = [np.array(seq, dtype=np.float32) for seq in cache['ews_sequences']]
        self.targets = cache['rul_targets']
        self.bearing_ids = cache['bearing_ids']
        self.max_ruls = cache['max_ruls']

        # Filter by bearing if requested
        if bearing_list is not None:
            keep = [i for i, bid in enumerate(self.bearing_ids) if bid in bearing_list]
            self.sequences = [self.sequences[i] for i in keep]
            self.targets = [self.targets[i] for i in keep]
            self.bearing_ids = [self.bearing_ids[i] for i in keep]
            self.max_ruls = [self.max_ruls[i] for i in keep]
            logger.info(f"Filtered to {len(self.sequences)} sequences for bearings {bearing_list}")

        logger.info(f"Loaded dataset with {len(self.sequences)} sequences, "
                    f"feature dim = {self.sequences[0].shape[1]}")

    def __len__(self):
        return len(self.sequences)

    def __getitem__(self, idx):
        return (torch.FloatTensor(self.sequences[idx]),
                torch.FloatTensor([self.targets[idx]]),
                self.bearing_ids[idx],
                self.max_ruls[idx])

# ----------------------------------------------------------------------------
# Attention extractor
# ----------------------------------------------------------------------------
class AttentionExtractor:
    def __init__(self, model, device='cpu'):
        self.model = model.to(device)
        self.device = device
        self.model.eval()

    def extract(self, dataset, batch_size=32):
        loader = DataLoader(dataset, batch_size=batch_size, shuffle=False)
        all_attentions = []
        all_bearings = []
        with torch.no_grad():
            for data, _, bearing_ids, _ in loader:
                data = data.to(self.device)
                _ = self.model(data)
                # Squeeze the last dimension to get (batch, seq_len)
                att = self.model.last_attention.squeeze(-1).cpu().numpy()
                all_attentions.append(att)
                all_bearings.extend(bearing_ids)
        all_attentions = np.concatenate(all_attentions, axis=0)
        return all_attentions, all_bearings

# ----------------------------------------------------------------------------
# Frequency band mapping (exactly as in SHAP analysis)
# ----------------------------------------------------------------------------
FREQUENCY_BANDS = {
    'H_cA4 (0-800Hz)': list(range(0, 7)),      # fault frequencies
    'H_cD4 (800-1600Hz)': list(range(7, 14)),
    'H_cD3 (1600-3200Hz)': list(range(14, 20)),
    'H_EWS (stats)': list(range(20, 44)),
    'V_cA4 (0-800Hz)': list(range(44, 51)),    # fault frequencies
    'V_cD4 (800-1600Hz)': list(range(51, 58)),
    'V_cD3 (1600-3200Hz)': list(range(58, 64)),
    'V_EWS (stats)': list(range(64, 88))
}

def compute_band_importance(attention_weights, n_bootstrap=1000):
    """
    attention_weights : numpy array of shape (n_samples, timesteps)
    returns: dict band -> mean importance, ratio, CI
    """
    n_samples, T = attention_weights.shape

    # Average over samples -> (T,)
    avg_time = np.mean(attention_weights, axis=0)

    # Expand to 88 features (repeat per timestep)
    expanded = np.repeat(avg_time[:, np.newaxis], 88, axis=1)  # (T, 88)
    per_feature = np.mean(expanded, axis=0)                    # (88,)

    # Band means
    band_means = {}
    for band, idx in FREQUENCY_BANDS.items():
        band_means[band] = np.mean(per_feature[idx])

    # cA4/other ratio
    cA4_bands = ['H_cA4 (0-800Hz)', 'V_cA4 (0-800Hz)']
    cA4_mean = np.mean([band_means[b] for b in cA4_bands])
    other_bands = [b for b in FREQUENCY_BANDS if b not in cA4_bands]
    other_mean = np.mean([band_means[b] for b in other_bands])
    ratio = cA4_mean / other_mean

    # Bootstrap CI
    ratios = []
    for _ in range(n_bootstrap):
        idx = np.random.choice(n_samples, n_samples, replace=True)
        att_boot = attention_weights[idx]                 # (n_samples, T)
        avg_time_boot = np.mean(att_boot, axis=0)        # (T,)
        expanded_boot = np.repeat(avg_time_boot[:, np.newaxis], 88, axis=1)
        per_feature_boot = np.mean(expanded_boot, axis=0)
        cA4_boot = np.mean([np.mean(per_feature_boot[FREQUENCY_BANDS[b]]) for b in cA4_bands])
        other_boot = np.mean([np.mean(per_feature_boot[FREQUENCY_BANDS[b]]) for b in other_bands])
        ratios.append(cA4_boot / other_boot)

    ci_low, ci_high = np.percentile(ratios, [2.5, 97.5])
    # Convert to Python native types for JSON serialization
    band_means = {k: float(v) for k, v in band_means.items()}
    ratio = float(ratio)
    ci_low = float(ci_low)
    ci_high = float(ci_high)
    per_feature = per_feature.tolist()   # convert numpy array to list of floats

    return band_means, ratio, ci_low, ci_high, per_feature

    

# ----------------------------------------------------------------------------
# Plotting function
# ----------------------------------------------------------------------------
def plot_band_importance(band_means, ratio, ci_low, ci_high, per_feature,
                         bearing_name, operating_rpm, output_dir):
    # Convert to numpy array if it's a list (for indexing with lists)
    if isinstance(per_feature, list):
        per_feature = np.array(per_feature)
    
    plt.figure(figsize=(12, 6))
    bands = list(band_means.keys())
    means = list(band_means.values())
    colors = ['red' if 'cA4' in b else 'steelblue' for b in bands]

    # Bar plot
    bars = plt.bar(range(len(bands)), means, color=colors, alpha=0.8, edgecolor='black')

    # Add error bars (standard deviation across features within each band)
    band_stds = {}
    for band, idx in FREQUENCY_BANDS.items():
        band_stds[band] = np.std(per_feature[idx])
    stds = [band_stds[b] for b in bands]
    plt.errorbar(range(len(bands)), means, yerr=stds, fmt='none', ecolor='gray', capsize=3)

    plt.xlabel('Frequency Band')
    plt.ylabel('Mean Attention (after expansion)')
    plt.title(f'{bearing_name}: cA4/other = {ratio:.3f}  (95% CI [{ci_low:.3f}, {ci_high:.3f}])')
    plt.xticks(range(len(bands)), bands, rotation=45, ha='right')
    plt.grid(True, axis='y', alpha=0.3)

    # Add fault frequency annotation (as before)
    f_r = operating_rpm / 60
    fault_freqs = {
        'BPFO': 3.052 * f_r,
        'BPFI': 4.948 * f_r,
        'BSF': 1.991 * f_r,
        'FTF': 0.3815 * f_r
    }
    fault_text = f'LDK UER204 @ {operating_rpm} RPM\n'
    for name, freq in fault_freqs.items():
        fault_text += f'{name}: {freq:.1f} Hz\n'
    fault_text += 'All in cA4 bands (0‑800 Hz)'
    plt.text(0.02, 0.98, fault_text, transform=plt.gca().transAxes, fontsize=10,
             verticalalignment='top', bbox=dict(boxstyle='round', facecolor='yellow', alpha=0.5))

    plt.tight_layout()
    plot_path = output_dir / f'attention_bands_{bearing_name}.png'
    plt.savefig(plot_path, dpi=300, bbox_inches='tight')
    plt.close()
    logger.info(f"Saved plot to {plot_path}")

# ----------------------------------------------------------------------------
# Main analysis function
# ----------------------------------------------------------------------------
def analyze_model(model_path, cache_path, bearing_list, operating_rpm,
                  output_dir="./attention_analysis", n_bootstrap=1000):
    """
    model_path : path to the saved model .pth file
    cache_path : path to the cache .pkl file containing test data
    bearing_list : list of bearing names to analyze (e.g., ['Bearing3_2', ...])
    operating_rpm : rotational speed for fault frequency annotation
    output_dir : where to save results
    """
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    # Device
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    logger.info(f"Using device: {device}")

    # Load model
    logger.info(f"Loading model from {model_path}")
    checkpoint = torch.load(model_path, map_location=device, weights_only=False)
    if isinstance(checkpoint, dict) and 'model_state_dict' in checkpoint:
        state_dict = checkpoint['model_state_dict']
    elif isinstance(checkpoint, dict) and 'state_dict' in checkpoint:
        state_dict = checkpoint['state_dict']
    else:
        state_dict = checkpoint

    # Infer input size from state_dict
    input_size = None
    for key in state_dict.keys():
        if 'lstm.weight_ih_l0' in key:
            input_size = state_dict[key].shape[1]
            break
    if input_size is None:
        input_size = 88  # fallback
        logger.warning("Could not infer input size, using default 88")

    model = EWS_Bidirectional_LSTM_Attention(input_size=input_size)
    model.load_state_dict(state_dict, strict=False)
    model.to(device)
    model.eval()
    logger.info(f"Model loaded, total parameters: {sum(p.numel() for p in model.parameters())}")

    # Load dataset
    logger.info(f"Loading dataset from {cache_path}")
    dataset = CachedDataset(cache_path, bearing_list=bearing_list)
    if len(dataset) == 0:
        raise ValueError("No data found for the specified bearings.")

    # Extract attention
    extractor = AttentionExtractor(model, device)
    att_weights, bearings = extractor.extract(dataset)
    logger.info(f"Extracted attention weights: {att_weights.shape}")

    # Compute band importance per bearing (if multiple bearings present, you may want per‑bearing or aggregated)
    # Here we aggregate all samples (they are already filtered to the desired bearings)
    band_means, ratio, ci_low, ci_high, per_feature = compute_band_importance(att_weights, n_bootstrap)

    # Plot
    plot_band_importance(band_means, ratio, ci_low, ci_high, per_feature,
                         bearing_name="+".join(bearing_list), operating_rpm=operating_rpm,
                         output_dir=output_dir)

    # Save results
    results = {
        'model_path': str(model_path),
        'cache_path': str(cache_path),
        'bearings': bearing_list,
        'operating_rpm': operating_rpm,
        'n_samples': att_weights.shape[0],
        'band_means': band_means,
        'ratio': ratio,
        'ci_95': [ci_low, ci_high],
        'per_feature_importance': per_feature  # <-- removed .tolist()
    }
    with open(output_dir / 'attention_results.json', 'w') as f:
        json.dump(results, f, indent=2)
    logger.info(f"Results saved to {output_dir / 'attention_results.json'}")

    # Print summary
    print("\n" + "="*60)
    print(f"ATTENTION ANALYSIS SUMMARY for {bearing_list}")
    print("="*60)
    print(f"cA4/other ratio = {ratio:.4f}  95% CI [{ci_low:.4f}, {ci_high:.4f}]")
    if ratio > 1:
        print("→ Attention focuses more on fault frequency bands (cA4).")
    else:
        print("→ Attention does NOT preferentially focus on fault frequencies.")
    print("="*60)

    return results

# ----------------------------------------------------------------------------
# Example usage – MODIFY THESE PATHS ACCORDING TO YOUR SETUP
# ----------------------------------------------------------------------------
if __name__ == "__main__":
    # ===== CONFIGURE THESE =====
    MODEL_PATH = "/kaggle/working/models/model_Cross-condition_1_bilstm_attn_waveletTrue_DAFalse_seq10_win1024_val0.2_seed42_20260227_172613.pth"
    CACHE_PATH = "/kaggle/working/cache/dataset_58419fc36b975a6c18740516d1c6da0d.pkl"   # cache file containing test bearings for the split
    BEARINGS = ['Bearing3_2', 'Bearing3_3', 'Bearing3_4']   # test bearings for Split 1 (40Hz)
    OPERATING_RPM = 2400                                     # 40Hz * 60 = 2400 RPM # split 2 2250
    OUTPUT_DIR = "./attention_analysis"
    N_BOOTSTRAP = 1000
    # ===========================

    analyze_model(
        model_path=MODEL_PATH,
        cache_path=CACHE_PATH,
        bearing_list=BEARINGS,
        operating_rpm=OPERATING_RPM,
        output_dir=OUTPUT_DIR,
        n_bootstrap=N_BOOTSTRAP
    )

# Feature Importance

In [ ]:
# ==============================================================================
# VERIFICATION: Check feature‑band mapping (fixed)
# ==============================================================================
import pickle
import numpy as np
import os

# Load a cache file (adjust path if needed)
cache_dir = "/kaggle/working/cache"
cache_files = [f for f in os.listdir(cache_dir) if f.endswith('.pkl')]
if not cache_files:
    raise FileNotFoundError("No cache files found.")

# Use the first cache file (or pick one that contains test bearings)
cache_path = os.path.join(cache_dir, cache_files[0])
print(f"Loading cache: {cache_path}")
with open(cache_path, 'rb') as f:
    cache_data = pickle.load(f)

# Get the first sample (sequence) and convert to numpy array
sample = np.array(cache_data['ews_sequences'][0])   # shape: (seq_len, n_features)
print(f"Sample shape: {sample.shape}")
print(f"First time step, first 10 feature values: {sample[0, :10]}")

# Generate feature names using the same function as in your SHAP script
def create_feature_names(seq_len=10, n_features=88):
    base_names = [
        'CSD_AR1', 'CSD_Variance', 'CSD_Spectral',
        'STAT_Skewness', 'STAT_Kurtosis', 'STAT_ZeroCross', 'STAT_Entropy',
        'ACF_Lag1', 'ACF_Lag2', 'ACF_Lag5', 'ACF_Lag10', 'ACF_Lag20',
        'SPEC_Centroid', 'SPEC_Spread', 'SPEC_Skewness', 'SPEC_Kurtosis',
        'NLD_DFA', 'NLD_Hurst',
        'STAT_SNR', 'STAT_Crest', 'STAT_Impulse', 'STAT_Clearance',
        'STAT_Shape', 'STAT_PeakToPeak',
    ] + [f'WAV_{i+1}' for i in range(20)]
    assert len(base_names) == 44
    feature_names = []
    for t in range(seq_len):
        for ch in ['H', 'V']:
            for base in base_names:
                feature_names.append(f'T{t:02d}_{ch}_{base}')
    return feature_names

feature_names = create_feature_names(seq_len=sample.shape[0], n_features=sample.shape[1])
print(f"First 10 feature names: {feature_names[:10]}")

# Band definitions (must match your feature extraction order)
bands = {
    'H_cA4 (0-800Hz)': list(range(0, 7)),
    'H_cD4 (800-1600Hz)': list(range(7, 14)),
    'H_cD3 (1600-3200Hz)': list(range(14, 20)),
    'H_EWS (stats)': list(range(20, 44)),
    'V_cA4 (0-800Hz)': list(range(44, 51)),
    'V_cD4 (800-1600Hz)': list(range(51, 58)),
    'V_cD3 (1600-3200Hz)': list(range(58, 64)),
    'V_EWS (stats)': list(range(64, 88)),
}

# Compute mean feature value per band (averaged over all time steps of the first sample)
band_means = {}
for band_name, idx_list in bands.items():
    # average over time steps and the selected features
    band_values = sample[:, idx_list].mean()
    band_means[band_name] = band_values

print("\nMean feature values per band (first sample):")
for band, val in band_means.items():
    print(f"  {band}: {val:.6f}")

# Check cA4 vs other bands
cA4_bands = ['H_cA4 (0-800Hz)', 'V_cA4 (0-800Hz)']
cA4_mean = np.mean([band_means[b] for b in cA4_bands])
other_bands = [b for b in bands if b not in cA4_bands]
other_mean = np.mean([band_means[b] for b in other_bands])
print(f"\ncA4 mean: {cA4_mean:.6f}, other mean: {other_mean:.6f}, ratio: {cA4_mean/other_mean:.3f}")